<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 06.01 — INITIALIZATION AND RESEARCH SCOPE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06")
print("CANDIDATE STRATEGY EVALUATION")
print("=" * 100)

print("\nResearch objective:")
print(
    "Empirically evaluate candidate missing-value imputation "
    "strategies under controlled missingness scenarios and "
    "generate evidence for AIR-LLM adaptive strategy selection."
)

print("\nResearch scope:")
print("  • Multiple benchmark datasets")
print("  • Feature-level evaluation")
print("  • MCAR and MAR missingness")
print("  • Missingness rates configured in Notebook 00")
print("  • Repeated deterministic experiments")
print("  • Ground-truth recovery from observed values")
print("  • Numerical reconstruction metrics")
print("  • Categorical reconstruction metrics")
print("  • Distributional fidelity")
print("  • Dependency preservation")
print("  • Downstream predictive utility")
print("  • Runtime measurement")
print("  • LLM recommendation evaluation")
print("  • Statistical significance analysis")
print("  • Adaptive utility-based selection")

print("\nNotebook sequence:")
for cell, purpose in [
    ("06.01", "Notebook 06 initialization and research scope"),
    ("06.02", "Load Notebook 00 master configuration"),
    ("06.03", "Validate datasets, features, targets, and experiment configuration"),
    ("06.04", "Load processed experimental datasets"),
    ("06.05", "Build evaluation dataset registry"),
    ("06.06", "Detect/validate feature types"),
    ("06.07", "Build predictor-selection function"),
    ("06.08", "Build missingness-mask generation"),
    ("06.09", "Build predictor encoding pipeline"),
    ("06.10", "Define statistical candidate imputers"),
    ("06.11", "Define KNN candidate"),
    ("06.12", "Define iterative/MICE candidates"),
    ("06.13", "Define Random Forest candidate"),
    ("06.14", "Define Gradient Boosting candidate"),
    ("06.15", "Define MissForest candidate"),
    ("06.16", "Define SoftImpute / matrix-factorization candidates"),
    ("06.17", "Candidate strategy compatibility validation"),
    ("06.18", "Build LLM-recommended candidate evaluation plan"),
    ("06.19", "Validate evaluation-plan integrity and expected experiment count"),
    ("06.20", "Small-run preparation / candidate execution preflight"),
    ("06.21", "Corrected Candidate Evaluation Engine + small validation + full execution"),
    ("06.22", "Evaluation Status Summary"),
    ("06.23", "Candidate Performance Analysis"),
    ("06.24", "LLM Recommendation Evaluation"),
    ("06.25", "Statistical Significance"),
    ("06.26", "Adaptive Utility-Based Selection"),
]:
    print(f"  {cell}  {purpose}")

print("\nResearch principle:")
print(
    "Observed values are masked to create known ground truth; "
    "imputation is performed without access to the masked truth."
)

print("\n" + "=" * 100)
print("NOTEBOOK 06 INITIALIZATION : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 06
CANDIDATE STRATEGY EVALUATION

Research objective:
Empirically evaluate candidate missing-value imputation strategies under controlled missingness scenarios and generate evidence for AIR-LLM adaptive strategy selection.

Research scope:
  • Multiple benchmark datasets
  • Feature-level evaluation
  • MCAR and MAR missingness
  • Missingness rates configured in Notebook 00
  • Repeated deterministic experiments
  • Ground-truth recovery from observed values
  • Numerical reconstruction metrics
  • Categorical reconstruction metrics
  • Distributional fidelity
  • Dependency preservation
  • Downstream predictive utility
  • Runtime measurement
  • LLM recommendation evaluation
  • Statistical significance analysis
  • Adaptive utility-based selection

Notebook sequence:
  06.01  Notebook 06 initialization and research scope
  06.02  Load Notebook 00 master configuration
  06.03  Validate datasets, features, targets, and experiment configuration
  06.04  Load proces

In [2]:
# ============================================================
# NOTEBOOK 06.02 — MOUNT GOOGLE DRIVE + LOAD MASTER CONFIG
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.02")
print("MOUNT GOOGLE DRIVE + LOAD NOTEBOOK 00 MASTER CONFIGURATION")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import yaml
import random
import warnings
import time
import inspect
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


print("\n[06.02.01] Imports completed.")


# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

try:

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    print(
        "\n[06.02.02] Google Drive mounted successfully."
    )

except ImportError:

    raise RuntimeError(
        "This notebook is intended to run in Google Colab, "
        "but google.colab could not be imported."
    )


# ============================================================
# 3. VALIDATE DRIVE
# ============================================================

DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE_ROOT = (
    DRIVE_ROOT /
    "MyDrive"
)


if not DRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive mount point was not found:\n"
        f"{DRIVE_ROOT}"
    )


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google MyDrive directory was not found:\n"
        f"{MYDRIVE_ROOT}"
    )


print(
    "\n[06.02.03] Google Drive filesystem validation: PASS"
)

print(
    f"Drive root : {DRIVE_ROOT}"
)

print(
    f"MyDrive    : {MYDRIVE_ROOT}"
)


# ============================================================
# 4. CANONICAL AIR-LLM PROJECT ROOT
# ============================================================

PROJECT_ROOT = (
    MYDRIVE_ROOT /
    "AIR_LLM_Research"
)


CONFIG_PATH = (
    PROJECT_ROOT /
    "config" /
    "config.yaml"
)


print(
    "\n[06.02.04] AIR-LLM project paths initialized."
)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)

print(
    f"CONFIG_PATH  : {CONFIG_PATH}"
)


# ============================================================
# 5. VALIDATE PROJECT ROOT
# ============================================================

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "\nAIR-LLM project root not found:\n"
        f"{PROJECT_ROOT}\n\n"
        "Expected canonical project root:\n"
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


if not PROJECT_ROOT.is_dir():

    raise NotADirectoryError(
        f"AIR-LLM project root is not a directory:\n"
        f"{PROJECT_ROOT}"
    )


print(
    "\n[06.02.05] AIR-LLM project root validation: PASS"
)


# ============================================================
# 6. VALIDATE CONFIGURATION DIRECTORY
# ============================================================

CONFIG_DIR = (
    PROJECT_ROOT /
    "config"
)


if not CONFIG_DIR.exists():

    raise FileNotFoundError(
        f"Configuration directory not found:\n"
        f"{CONFIG_DIR}"
    )


if not CONFIG_DIR.is_dir():

    raise NotADirectoryError(
        f"Configuration path is not a directory:\n"
        f"{CONFIG_DIR}"
    )


print(
    "[06.02.06] Configuration directory validation: PASS"
)


# ============================================================
# 7. VALIDATE MASTER CONFIGURATION FILE
# ============================================================

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        "\nNotebook 00 configuration not found:\n"
        f"{CONFIG_PATH}\n\n"
        "Run Notebook 00 first and ensure "
        "config/config.yaml was exported."
    )


if not CONFIG_PATH.is_file():

    raise FileNotFoundError(
        f"Configuration path is not a file:\n"
        f"{CONFIG_PATH}"
    )


print(
    "[06.02.07] config.yaml validation: PASS"
)


# ============================================================
# 8. LOAD NOTEBOOK 00 MASTER CONFIGURATION
# ============================================================

try:

    with open(
        CONFIG_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        CONFIG = yaml.safe_load(f)

except yaml.YAMLError as exc:

    raise RuntimeError(
        "Failed to parse config.yaml.\n"
        f"Error: {exc}"
    )


if not isinstance(
    CONFIG,
    dict
):

    raise TypeError(
        "config.yaml must contain a dictionary."
    )


if len(CONFIG) == 0:

    raise RuntimeError(
        "config.yaml was loaded but contains no "
        "configuration sections."
    )


print(
    "\n[06.02.08] Master configuration loaded."
)


# ============================================================
# 9. EXTRACT COMMON CONFIGURATION VALUES
# ============================================================
# These variables are created only when their corresponding
# sections exist. Existing Notebook 00 configuration remains
# the authoritative source.

if isinstance(
    CONFIG.get("datasets"),
    dict
):

    DATASET_CONFIG = CONFIG[
        "datasets"
    ]

elif isinstance(
    CONFIG.get("dataset_registry"),
    dict
):

    DATASET_CONFIG = CONFIG[
        "dataset_registry"
    ]

else:

    DATASET_CONFIG = {}


# ------------------------------------------------------------
# Dataset IDs
# ------------------------------------------------------------

if DATASET_CONFIG:

    DATASET_IDS = list(
        DATASET_CONFIG.keys()
    )

else:

    DATASET_IDS = []


# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

if isinstance(
    CONFIG.get("experiment"),
    dict
):

    EXPERIMENT_CONFIG = CONFIG[
        "experiment"
    ]

else:

    EXPERIMENT_CONFIG = {}


# ============================================================
# 10. CONFIGURATION SUMMARY
# ============================================================

print(
    "\n" + "-" * 100
)

print(
    "MASTER CONFIGURATION SUMMARY"
)

print(
    "-" * 100
)

print(
    f"Project root                  : {PROJECT_ROOT}"
)

print(
    f"Configuration file            : {CONFIG_PATH}"
)

print(
    f"Top-level configuration keys : {len(CONFIG)}"
)

print(
    f"Dataset IDs detected          : {len(DATASET_IDS)}"
)

if DATASET_IDS:

    print(
        "Datasets                      : "
        +
        ", ".join(
            str(x)
            for x in DATASET_IDS
        )
    )


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

required_state = {

    "PROJECT_ROOT":
        PROJECT_ROOT,

    "CONFIG_PATH":
        CONFIG_PATH,

    "CONFIG":
        CONFIG,

}


missing_state = [

    name

    for name, value
    in required_state.items()

    if value is None

]


if missing_state:

    raise RuntimeError(
        "Required Notebook 06.02 state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_state
        )
    )


print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 06.02 — MASTER CONFIGURATION LOAD: PASSED"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 06.02
MOUNT GOOGLE DRIVE + LOAD NOTEBOOK 00 MASTER CONFIGURATION

[06.02.01] Imports completed.
Mounted at /content/drive

[06.02.02] Google Drive mounted successfully.

[06.02.03] Google Drive filesystem validation: PASS
Drive root : /content/drive
MyDrive    : /content/drive/MyDrive

[06.02.04] AIR-LLM project paths initialized.
PROJECT_ROOT : /content/drive/MyDrive/AIR_LLM_Research
CONFIG_PATH  : /content/drive/MyDrive/AIR_LLM_Research/config/config.yaml

[06.02.05] AIR-LLM project root validation: PASS
[06.02.06] Configuration directory validation: PASS
[06.02.07] config.yaml validation: PASS

[06.02.08] Master configuration loaded.

----------------------------------------------------------------------------------------------------
MASTER CONFIGURATION SUMMARY
----------------------------------------------------------------------------------------------------
Project root                  : /content/drive/MyDrive/AIR_LLM_Research
Configuration file            : 

In [3]:
# ============================================================
# NOTEBOOK 06.03 — DATASET / FEATURE / TARGET / EXPERIMENT
#                 VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.03")
print("DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION")
print("=" * 100)


# ============================================================
# 1. IMPORTS AND REQUIRED STATE
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


if "CONFIG" not in globals():
    raise RuntimeError(
        "CONFIG is not available. Run Notebook 06.02 first."
    )

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


# ============================================================
# 2. CANONICAL DATASET REGISTRY
# ============================================================

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


# ============================================================
# 3. TARGET REGISTRY
# ============================================================

KNOWN_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


TARGET_REGISTRY = {}


# Try Notebook 00 configuration first
config_target_sources = [
    CONFIG.get("TARGET_REGISTRY"),
    CONFIG.get("target_registry"),
    CONFIG.get("targets"),
]


features_config = CONFIG.get("features", {})

if isinstance(features_config, dict):

    config_target_sources.extend([
        features_config.get("TARGET_REGISTRY"),
        features_config.get("target_registry"),
        features_config.get("targets"),
    ])


for source in config_target_sources:

    if not isinstance(source, dict):
        continue

    for dataset_id, value in source.items():

        if isinstance(value, str):

            TARGET_REGISTRY[
                str(dataset_id)
            ] = value

        elif isinstance(value, dict):

            for key in [
                "target",
                "target_column",
                "target_feature",
                "column",
            ]:

                if value.get(key) is not None:

                    TARGET_REGISTRY[
                        str(dataset_id)
                    ] = str(value[key])

                    break


# Fill any missing canonical targets
for dataset_id in EXPECTED_DATASETS:

    if dataset_id not in TARGET_REGISTRY:

        TARGET_REGISTRY[
            dataset_id
        ] = KNOWN_TARGETS[dataset_id]


# Validate target registry
missing_targets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in TARGET_REGISTRY
]


if missing_targets:

    raise RuntimeError(
        "Missing target definitions:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_targets
        )
    )


# ============================================================
# 4. PROCESSED DATASET FILE DISCOVERY
# ============================================================

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)


if not PROCESSED_DIR.exists():

    raise FileNotFoundError(
        f"Processed-data directory not found:\n"
        f"{PROCESSED_DIR}"
    )


def find_processed_dataset(dataset_id):

    candidates = [
        PROCESSED_DIR / f"{dataset_id}_processed.csv",
        PROCESSED_DIR / f"{dataset_id}.csv",
        PROCESSED_DIR / dataset_id / "processed.csv",
    ]

    for path in candidates:

        if path.exists():
            return path

    matches = list(
        PROCESSED_DIR.rglob(
            f"{dataset_id}*.csv"
        )
    )

    if matches:
        return matches[0]

    return None


DATASET_PATH_REGISTRY = {}

missing_files = []


for dataset_id in EXPECTED_DATASETS:

    path = find_processed_dataset(
        dataset_id
    )

    if path is None:

        missing_files.append(
            dataset_id
        )

    else:

        DATASET_PATH_REGISTRY[
            dataset_id
        ] = path


if missing_files:

    raise FileNotFoundError(
        "Processed dataset files not found:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_files
        )
    )


# ============================================================
# 5. DATASET SCHEMA / TARGET VALIDATION
# ============================================================

validation_rows = []


for dataset_id in EXPECTED_DATASETS:

    path = DATASET_PATH_REGISTRY[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # Read header only — full data loading occurs in 06.04
    header = pd.read_csv(
        path,
        nrows=0
    )

    columns = list(
        header.columns
    )

    target_present = (
        target in columns
    )

    if not target_present:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"'{dataset_id}'.\n"
            f"File: {path}"
        )

    validation_rows.append({

        "dataset_id":
            dataset_id,

        "columns":
            len(columns),

        "target":
            target,

        "target_present":
            True,

        "path":
            str(path),

    })


DATASET_VALIDATION_DF = pd.DataFrame(
    validation_rows
)


# ============================================================
# 6. MISSINGNESS EXPERIMENT CONFIGURATION
# ============================================================

missingness_config = CONFIG.get(
    "missingness",
    {}
)

if not isinstance(
    missingness_config,
    dict
):

    missingness_config = {}


# ------------------------------------------------------------
# Missingness mechanisms
# ------------------------------------------------------------

missingness_mechanisms = (
    missingness_config.get("mechanisms")
    or
    missingness_config.get("missingness_mechanisms")
    or
    CONFIG.get("missingness_mechanisms")
)


if missingness_mechanisms is None:

    # Use the mechanisms defined for the AIR-LLM experiment
    missingness_mechanisms = [
        "MCAR",
        "MAR",
    ]


missingness_mechanisms = [
    str(x).upper()
    for x in missingness_mechanisms
]


valid_mechanisms = {
    "MCAR",
    "MAR",
    "MNAR",
}


invalid_mechanisms = (
    set(missingness_mechanisms)
    -
    valid_mechanisms
)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(
                invalid_mechanisms
            )
        )
    )


# ------------------------------------------------------------
# Missingness rates
# ------------------------------------------------------------

missingness_rates = (
    missingness_config.get("rates")
    or
    missingness_config.get("missingness_rates")
    or
    CONFIG.get("missingness_rates")
)


if missingness_rates is None:

    missingness_rates = [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ]


missingness_rates = [
    float(x)
    for x in missingness_rates
]


if not all(
    0 < x < 1
    for x in missingness_rates
):

    raise RuntimeError(
        "Missingness rates must be between 0 and 1:\n"
        f"{missingness_rates}"
    )


# ------------------------------------------------------------
# Repetitions
# ------------------------------------------------------------

repetitions = (
    missingness_config.get("repetitions")
    or
    CONFIG.get("repetitions")
    or
    5
)

repetitions = int(
    repetitions
)


if repetitions < 1:

    raise RuntimeError(
        "Repetitions must be >= 1."
    )


# ------------------------------------------------------------
# Master seed
# ------------------------------------------------------------

MASTER_SEED = int(
    CONFIG.get("master_seed")
    or
    CONFIG.get("seed")
    or
    42
)


# ============================================================
# 7. EXPORT VALIDATED EXPERIMENT STATE
# ============================================================

EXPERIMENT_CONFIGURATION = {

    "datasets":
        EXPECTED_DATASETS,

    "targets":
        TARGET_REGISTRY,

    "missingness_mechanisms":
        missingness_mechanisms,

    "missingness_rates":
        missingness_rates,

    "repetitions":
        repetitions,

    "master_seed":
        MASTER_SEED,

}


# ============================================================
# 8. DISPLAY VALIDATION
# ============================================================

print(
    "\n[06.03.01] Dataset / target configuration: PASS"
)

print(
    "[06.03.02] Processed dataset discovery: PASS"
)

print(
    "[06.03.03] Dataset schema / target validation: PASS"
)


display(
    DATASET_VALIDATION_DF
)


print(
    "\n" + "-" * 100
)

print("EXPERIMENT CONFIGURATION")

print("-" * 100)

print(
    f"Datasets                : "
    f"{EXPECTED_DATASETS}"
)

print(
    f"Targets                 : "
    f"{TARGET_REGISTRY}"
)

print(
    f"Missingness mechanisms  : "
    f"{missingness_mechanisms}"
)

print(
    f"Missingness rates       : "
    f"{missingness_rates}"
)

print(
    f"Repetitions             : "
    f"{repetitions}"
)

print(
    f"Master seed             : "
    f"{MASTER_SEED}"
)


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

assert len(
    DATASET_PATH_REGISTRY
) == len(
    EXPECTED_DATASETS
)

assert len(
    TARGET_REGISTRY
) == len(
    EXPECTED_DATASETS
)

assert all(
    0 < rate < 1
    for rate in missingness_rates
)

assert repetitions >= 1


print(
    "\n" + "=" * 100
)

print(
    "DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.03
DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION

[06.03.01] Dataset / target configuration: PASS
[06.03.02] Processed dataset discovery: PASS
[06.03.03] Dataset schema / target validation: PASS


,dataset_id,columns,target,target_present,path
0,adult_income,15,income,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...
1,bank_marketing,17,y,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...
2,diabetes_130us,48,readmitted,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...



----------------------------------------------------------------------------------------------------
EXPERIMENT CONFIGURATION
----------------------------------------------------------------------------------------------------
Datasets                : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets                 : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Missingness mechanisms  : ['MCAR', 'MAR', 'MNAR']
Missingness rates       : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions             : 5
Master seed             : 42

DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED


In [4]:
# ============================================================
# NOTEBOOK 06.04 — LOAD PROCESSED EXPERIMENTAL DATASETS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.04")
print("LOAD PROCESSED EXPERIMENTAL DATASETS")
print("=" * 100)


PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)


if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed-data directory not found:\n{PROCESSED_DIR}"
    )


def find_processed_dataset(dataset_id):

    candidates = [
        PROCESSED_DIR / f"{dataset_id}_processed.csv",
        PROCESSED_DIR / f"{dataset_id}.csv",
        PROCESSED_DIR / dataset_id / "processed.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    recursive_matches = list(
        PROCESSED_DIR.rglob(
            f"{dataset_id}*.csv"
        )
    )

    if recursive_matches:
        return recursive_matches[0]

    return None


EVALUATION_DATA = {}
DATASET_PATH_REGISTRY = {}

for dataset_id in DATASET_IDS:

    path = find_processed_dataset(
        dataset_id
    )

    if path is None:
        raise FileNotFoundError(
            f"No processed dataset found for '{dataset_id}'."
        )

    df = pd.read_csv(path)

    if df.empty:
        raise RuntimeError(
            f"Processed dataset '{dataset_id}' is empty."
        )

    EVALUATION_DATA[dataset_id] = df
    DATASET_PATH_REGISTRY[dataset_id] = path

    print(
        f"{dataset_id:<20} "
        f"rows={len(df):>8,} "
        f"columns={df.shape[1]:>4} "
        f"path={path}"
    )


print(
    f"\nDatasets loaded: {len(EVALUATION_DATA)}"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.04
LOAD PROCESSED EXPERIMENTAL DATASETS
adult_income         rows=  32,561 columns=  15 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/adult_income_processed.csv
bank_marketing       rows=  45,211 columns=  17 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/bank_marketing_processed.csv
diabetes_130us       rows= 101,766 columns=  48 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/diabetes_130us_processed.csv

Datasets loaded: 3


In [5]:
# ============================================================
# NOTEBOOK 06.05 — DATASET INTEGRITY / DATA-TYPE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.05")
print("DATASET INTEGRITY / DATA-TYPE VALIDATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "CONFIG",
    "PROJECT_ROOT",
    "EVALUATION_DATA",
    "DATASET_PATH_REGISTRY",
    "EXPECTED_DATASETS",
    "TARGET_REGISTRY",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required Notebook 06 state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.03 → 06.04 first."
    )


if not isinstance(
    EVALUATION_DATA,
    dict
):

    raise RuntimeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not EVALUATION_DATA:

    raise RuntimeError(
        "EVALUATION_DATA is empty."
    )


# ============================================================
# 2. DATASET PRESENCE VALIDATION
# ============================================================

missing_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in EVALUATION_DATA
]


if missing_datasets:

    raise RuntimeError(
        "Required datasets are missing from EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_datasets
        )
    )


print(
    "\n[06.05.01] Dataset presence validation: PASS"
)


# ============================================================
# 3. DATASET INTEGRITY VALIDATION
# ============================================================

DATASET_INTEGRITY_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    # --------------------------------------------------------
    # Basic DataFrame validation
    # --------------------------------------------------------

    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id} is not a pandas DataFrame."
        )


    if df.empty:

        raise RuntimeError(
            f"{dataset_id} is empty."
        )


    # --------------------------------------------------------
    # Column validation
    # --------------------------------------------------------

    if df.shape[1] == 0:

        raise RuntimeError(
            f"{dataset_id} contains no columns."
        )


    if df.columns.duplicated().any():

        duplicated_columns = (
            df.columns[
                df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id} contains duplicate columns:\n"
            +
            "\n".join(
                f"  - {x}"
                for x
                in duplicated_columns
            )
        )


    # --------------------------------------------------------
    # Target validation
    # --------------------------------------------------------

    target = TARGET_REGISTRY[
        dataset_id
    ]


    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"{dataset_id}."
        )


    # --------------------------------------------------------
    # Missing-value summary
    # --------------------------------------------------------

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )


    missing_columns = int(
        df.isna()
        .any()
        .sum()
    )


    # --------------------------------------------------------
    # Duplicate-row count
    # --------------------------------------------------------

    duplicate_rows = int(
        df.duplicated()
        .sum()
    )


    DATASET_INTEGRITY_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(
                len(df)
            ),

        "columns":
            int(
                df.shape[1]
            ),

        "target":
            target,

        "target_present":
            True,

        "missing_cells":
            missing_cells,

        "missing_columns":
            missing_columns,

        "duplicate_rows":
            duplicate_rows,

    })


DATASET_INTEGRITY_DF = pd.DataFrame(
    DATASET_INTEGRITY_ROWS
)


# ============================================================
# 4. DATA-TYPE VALIDATION
# ============================================================

DATA_TYPE_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    numerical_columns = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )


    categorical_columns = (
        df.select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_dtype = str(
        df[target].dtype
    )


    target_is_numeric = (
        pd.api.types.is_numeric_dtype(
            df[target]
        )
    )


    DATA_TYPE_ROWS.append({

        "dataset_id":
            dataset_id,

        "numerical_features":
            len(
                numerical_columns
            ),

        "categorical_features":
            len(
                categorical_columns
            ),

        "target":
            target,

        "target_dtype":
            target_dtype,

        "target_is_numeric":
            target_is_numeric,

        "total_features":
            len(
                df.columns
            ),

    })


DATA_TYPE_VALIDATION_DF = pd.DataFrame(
    DATA_TYPE_ROWS
)


print(
    "[06.05.02] Dataset integrity validation: PASS"
)

print(
    "[06.05.03] Data-type validation: PASS"
)


# ============================================================
# 5. FEATURE-TYPE REGISTRY
# ============================================================

FEATURE_TYPE_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature in df.columns:

        # Target is excluded from imputation-feature analysis
        if feature == target:
            continue


        if pd.api.types.is_numeric_dtype(
            df[feature]
        ):

            feature_type = "numerical"

        else:

            feature_type = "categorical"


        FEATURE_TYPE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "feature_type":
                feature_type,

            "dtype":
                str(
                    df[feature].dtype
                ),

            "missing_count":
                int(
                    df[feature]
                    .isna()
                    .sum()
                ),

            "missing_rate":
                float(
                    df[feature]
                    .isna()
                    .mean()
                ),

            "unique_values":
                int(
                    df[feature]
                    .nunique(
                        dropna=True
                    )
                ),

        })


FEATURE_TYPE_DF = pd.DataFrame(
    FEATURE_TYPE_ROWS
)


# Compatibility aliases for later notebooks
FEATURE_TYPE_VALIDATION_DF = (
    FEATURE_TYPE_DF.copy()
)


FEATURE_PROFILE_DF = (
    FEATURE_TYPE_DF.copy()
)


# ============================================================
# 6. DATA QUALITY VALIDATION
# ============================================================

invalid_feature_types = (
    FEATURE_TYPE_DF[
        ~FEATURE_TYPE_DF[
            "feature_type"
        ].isin(
            [
                "numerical",
                "categorical",
            ]
        )
    ]
)


if not invalid_feature_types.empty:

    raise RuntimeError(
        "Unsupported feature types detected:\n"
        +
        invalid_feature_types[
            [
                "dataset_id",
                "feature",
                "feature_type",
            ]
        ]
        .to_string(
            index=False
        )
    )


# Check for columns containing only missing values
all_missing_features = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature in df.columns:

        if feature == target:
            continue


        if df[feature].isna().all():

            all_missing_features.append(
                f"{dataset_id} -> {feature}"
            )


if all_missing_features:

    raise RuntimeError(
        "Features containing 100% missing values detected:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in all_missing_features
        )
    )


print(
    "[06.05.04] Feature data-quality validation: PASS"
)


# ============================================================
# 7. FINAL DATASET VALIDATION TABLE
# ============================================================

display(
    DATASET_INTEGRITY_DF
)


print(
    "\nDATA-TYPE SUMMARY"
)

display(
    DATA_TYPE_VALIDATION_DF
)


print(
    "\nFEATURE-TYPE SUMMARY"
)

display(
    FEATURE_TYPE_DF.head(20)
)


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

assert (
    len(EVALUATION_DATA)
    ==
    len(EXPECTED_DATASETS)
)


assert not DATASET_INTEGRITY_DF.empty


assert not DATA_TYPE_VALIDATION_DF.empty


assert not FEATURE_TYPE_DF.empty


assert set(
    FEATURE_TYPE_DF[
        "feature_type"
    ].unique()
).issubset(
    {
        "numerical",
        "categorical",
    }
)


# ============================================================
# 9. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "DATASET INTEGRITY SUMMARY"
)

print(
    "-" * 100
)

for dataset_id in EXPECTED_DATASETS:

    row = DATASET_INTEGRITY_DF[
        DATASET_INTEGRITY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ].iloc[0]


    print(
        f"{dataset_id:<20} "
        f"rows={row['rows']:>8,}  "
        f"columns={row['columns']:>3}  "
        f"missing_cells={row['missing_cells']:>8,}  "
        f"duplicates={row['duplicate_rows']:>6,}"
    )


print(
    "\nFeature-type records : "
    f"{len(FEATURE_TYPE_DF):,}"
)


print(
    "Numerical features    : "
    f"{sum(FEATURE_TYPE_DF['feature_type'] == 'numerical'):,}"
)


print(
    "Categorical features  : "
    f"{sum(FEATURE_TYPE_DF['feature_type'] == 'categorical'):,}"
)


print(
    "\n" + "=" * 100
)

print(
    "DATASET INTEGRITY / DATA-TYPE VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.05
DATASET INTEGRITY / DATA-TYPE VALIDATION

[06.05.01] Dataset presence validation: PASS
[06.05.02] Dataset integrity validation: PASS
[06.05.03] Data-type validation: PASS
[06.05.04] Feature data-quality validation: PASS


,dataset_id,rows,columns,target,target_present,missing_cells,missing_columns,duplicate_rows
0,adult_income,32561,15,income,True,4262,3,24
1,bank_marketing,45211,17,y,True,0,0,0
2,diabetes_130us,101766,48,readmitted,True,374017,9,0



DATA-TYPE SUMMARY


,dataset_id,numerical_features,categorical_features,target,target_dtype,target_is_numeric,total_features
0,adult_income,6,9,income,object,False,15
1,bank_marketing,7,10,y,object,False,17
2,diabetes_130us,11,37,readmitted,object,False,48



FEATURE-TYPE SUMMARY


,dataset_id,feature,feature_type,dtype,missing_count,missing_rate,unique_values
0,adult_income,age,numerical,int64,0,0.000000,73
1,adult_income,workclass,categorical,object,1836,0.056386,8
2,adult_income,fnlwgt,numerical,int64,0,0.000000,21648
3,adult_income,education,categorical,object,0,0.000000,16
4,adult_income,education_num,numerical,int64,0,0.000000,16
5,adult_income,marital_status,categorical,object,0,0.000000,7
6,adult_income,occupation,categorical,object,1843,0.056601,14
7,adult_income,relationship,categorical,object,0,0.000000,6
8,adult_income,race,categorical,object,0,0.000000,5
9,adult_income,sex,categorical,object,0,0.000000,2



DATASET INTEGRITY SUMMARY
----------------------------------------------------------------------------------------------------
adult_income         rows=  32,561  columns= 15  missing_cells=   4,262  duplicates=    24
bank_marketing       rows=  45,211  columns= 17  missing_cells=       0  duplicates=     0
diabetes_130us       rows= 101,766  columns= 48  missing_cells= 374,017  duplicates=     0

Feature-type records : 77
Numerical features    : 24
Categorical features  : 53

DATASET INTEGRITY / DATA-TYPE VALIDATION : PASSED


In [6]:
# ============================================================
# NOTEBOOK 06.06 — FEATURE CHARACTERIZATION / VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.06")
print("FEATURE CHARACTERIZATION / VALIDATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "EXPECTED_DATASETS",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.03 → 06.04 → 06.05 first."
    )


# ============================================================
# 2. FEATURE TYPE DETECTION
# ============================================================

def detect_feature_type(series):

    # Boolean variables are categorical
    if pd.api.types.is_bool_dtype(series):
        return "categorical"

    # Numeric variables
    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    # Object, string, category, datetime-like and
    # other non-numeric variables are treated as categorical
    return "categorical"


# ============================================================
# 3. BUILD FEATURE CHARACTERIZATION REGISTRY
# ============================================================

FEATURE_TYPE_REGISTRY = {}

feature_rows = []


for dataset_id in EXPECTED_DATASETS:

    if dataset_id not in EVALUATION_DATA:
        raise RuntimeError(
            f"Dataset '{dataset_id}' is missing from EVALUATION_DATA."
        )


    df = EVALUATION_DATA[
        dataset_id
    ]


    target = TARGET_REGISTRY.get(
        dataset_id
    )


    if target is None:
        raise RuntimeError(
            f"No target defined for dataset '{dataset_id}'."
        )


    if target not in df.columns:
        raise RuntimeError(
            f"Target '{target}' not found in dataset '{dataset_id}'."
        )


    FEATURE_TYPE_REGISTRY[
        dataset_id
    ] = {}


    for feature in df.columns:

        series = df[feature]

        feature_type = detect_feature_type(
            series
        )

        FEATURE_TYPE_REGISTRY[
            dataset_id
        ][feature] = feature_type


        # ----------------------------------------------------
        # Basic statistics
        # ----------------------------------------------------

        n_rows = len(df)

        missing_count = int(
            series.isna().sum()
        )

        missing_rate = (
            missing_count / n_rows
            if n_rows > 0
            else 0.0
        )

        observed_count = int(
            series.notna().sum()
        )

        unique_values = int(
            series.nunique(
                dropna=True
            )
        )


        # ----------------------------------------------------
        # Cardinality
        # ----------------------------------------------------

        cardinality_ratio = (
            unique_values / observed_count
            if observed_count > 0
            else 0.0
        )


        # ----------------------------------------------------
        # Constant / all-missing detection
        # ----------------------------------------------------

        all_missing = (
            observed_count == 0
        )

        constant_feature = (
            observed_count > 0
            and unique_values <= 1
        )


        # ----------------------------------------------------
        # Target status
        # ----------------------------------------------------

        is_target = (
            feature == target
        )


        # ----------------------------------------------------
        # Dtype
        # ----------------------------------------------------

        dtype = str(
            series.dtype
        )


        # ----------------------------------------------------
        # Numerical statistics
        # ----------------------------------------------------

        if feature_type == "numerical":

            numeric_series = pd.to_numeric(
                series,
                errors="coerce"
            )

            finite_values = numeric_series[
                np.isfinite(
                    numeric_series
                )
            ]

            if len(finite_values) > 0:

                minimum = float(
                    finite_values.min()
                )

                maximum = float(
                    finite_values.max()
                )

                mean_value = float(
                    finite_values.mean()
                )

                std_value = float(
                    finite_values.std(
                        ddof=0
                    )
                )

            else:

                minimum = np.nan
                maximum = np.nan
                mean_value = np.nan
                std_value = np.nan

        else:

            minimum = np.nan
            maximum = np.nan
            mean_value = np.nan
            std_value = np.nan


        # ----------------------------------------------------
        # Characterization record
        # ----------------------------------------------------

        feature_rows.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "feature_type":
                feature_type,

            "dtype":
                dtype,

            "is_target":
                is_target,

            "rows":
                int(n_rows),

            "observed_count":
                observed_count,

            "missing_count":
                missing_count,

            "missing_rate":
                float(missing_rate),

            "unique_values":
                unique_values,

            "cardinality_ratio":
                float(cardinality_ratio),

            "constant_feature":
                bool(constant_feature),

            "all_missing":
                bool(all_missing),

            "minimum":
                minimum,

            "maximum":
                maximum,

            "mean":
                mean_value,

            "std":
                std_value,

        })


# ============================================================
# 4. CREATE AUTHORITATIVE FEATURE DATAFRAME
# ============================================================

FEATURE_CHARACTERIZATION_DF = pd.DataFrame(
    feature_rows
)


if FEATURE_CHARACTERIZATION_DF.empty:

    raise RuntimeError(
        "Feature characterization dataframe is empty."
    )


# ============================================================
# 5. AUTHORITATIVE FEATURE-TYPE DATAFRAME
# ============================================================

FEATURE_TYPE_DF = (
    FEATURE_CHARACTERIZATION_DF[
        [
            "dataset_id",
            "feature",
            "feature_type",
            "dtype",
            "is_target",
            "unique_values",
            "missing_count",
            "missing_rate",
        ]
    ]
    .copy()
)


# Compatibility aliases for downstream notebooks
FEATURE_TYPE_VALIDATION_DF = (
    FEATURE_TYPE_DF.copy()
)

FEATURE_PROFILE_DF = (
    FEATURE_CHARACTERIZATION_DF.copy()
)


# ============================================================
# 6. VALIDATE FEATURE TYPES
# ============================================================

valid_feature_types = {
    "numerical",
    "categorical",
}


invalid_feature_types = (
    set(
        FEATURE_TYPE_DF[
            "feature_type"
        ].unique()
    )
    -
    valid_feature_types
)


if invalid_feature_types:

    raise RuntimeError(
        "Unsupported feature types detected:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                invalid_feature_types
            )
        )
    )


# ============================================================
# 7. VALIDATE ALL-MISSING FEATURES
# ============================================================

all_missing_features = (
    FEATURE_CHARACTERIZATION_DF[
        FEATURE_CHARACTERIZATION_DF[
            "all_missing"
        ]
    ]
)


if not all_missing_features.empty:

    print(
        "\nWARNING: All-missing features detected:"
    )

    display(
        all_missing_features[
            [
                "dataset_id",
                "feature",
                "feature_type",
            ]
        ]
    )


# ============================================================
# 8. CONSTANT FEATURE REPORT
# ============================================================

constant_features = (
    FEATURE_CHARACTERIZATION_DF[
        FEATURE_CHARACTERIZATION_DF[
            "constant_feature"
        ]
    ]
)


if not constant_features.empty:

    print(
        "\nConstant features detected:"
    )

    display(
        constant_features[
            [
                "dataset_id",
                "feature",
                "feature_type",
                "unique_values",
            ]
        ]
    )


# ============================================================
# 9. TARGET VALIDATION
# ============================================================

target_validation_rows = []


for dataset_id in EXPECTED_DATASETS:

    target = TARGET_REGISTRY[
        dataset_id
    ]

    target_rows = (
        FEATURE_CHARACTERIZATION_DF[
            (
                FEATURE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                ==
                dataset_id
            )
            &
            (
                FEATURE_CHARACTERIZATION_DF[
                    "feature"
                ]
                ==
                target
            )
        ]
    )


    if len(target_rows) != 1:

        raise RuntimeError(
            f"Target validation failed for "
            f"{dataset_id}: {target}"
        )


    target_row = target_rows.iloc[0]


    target_validation_rows.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "feature_type":
            target_row[
                "feature_type"
            ],

        "dtype":
            target_row[
                "dtype"
            ],

        "unique_values":
            target_row[
                "unique_values"
            ],

        "missing_count":
            target_row[
                "missing_count"
            ],

        "valid":
            True,

    })


TARGET_VALIDATION_DF = pd.DataFrame(
    target_validation_rows
)


# ============================================================
# 10. DATASET-LEVEL FEATURE SUMMARY
# ============================================================

DATASET_FEATURE_SUMMARY_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    dataset_features = (
        FEATURE_CHARACTERIZATION_DF[
            (
                FEATURE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                ==
                dataset_id
            )
        ]
    )


    non_target_features = (
        dataset_features[
            ~dataset_features[
                "is_target"
            ]
        ]
    )


    numerical_count = int(
        (
            non_target_features[
                "feature_type"
            ]
            ==
            "numerical"
        ).sum()
    )


    categorical_count = int(
        (
            non_target_features[
                "feature_type"
            ]
            ==
            "categorical"
        ).sum()
    )


    DATASET_FEATURE_SUMMARY_ROWS.append({

        "dataset_id":
            dataset_id,

        "total_columns":
            int(
                len(dataset_features)
            ),

        "target":
            TARGET_REGISTRY[
                dataset_id
            ],

        "features_for_imputation":
            int(
                len(non_target_features)
            ),

        "numerical_features":
            numerical_count,

        "categorical_features":
            categorical_count,

        "features_with_missing":
            int(
                (
                    non_target_features[
                        "missing_count"
                    ]
                    >
                    0
                ).sum()
            ),

        "constant_features":
            int(
                non_target_features[
                    "constant_feature"
                ].sum()
            ),

    })


DATASET_FEATURE_SUMMARY_DF = pd.DataFrame(
    DATASET_FEATURE_SUMMARY_ROWS
)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print(
    "\n[06.06.01] Feature characterization completed."
)

print(
    f"Total feature records : "
    f"{len(FEATURE_CHARACTERIZATION_DF):,}"
)


print(
    "\nDATASET FEATURE SUMMARY"
)

display(
    DATASET_FEATURE_SUMMARY_DF
)


print(
    "\nTARGET VALIDATION"
)

display(
    TARGET_VALIDATION_DF
)


print(
    "\nFEATURE CHARACTERIZATION"
)

display(
    FEATURE_CHARACTERIZATION_DF
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

if len(
    FEATURE_CHARACTERIZATION_DF
) != sum(
    len(
        EVALUATION_DATA[
            dataset_id
        ].columns
    )
    for dataset_id
    in EXPECTED_DATASETS
):

    raise RuntimeError(
        "Feature characterization record count does not "
        "match the total number of dataset columns."
    )


if not set(
    FEATURE_TYPE_DF[
        "feature_type"
    ].unique()
).issubset(
    valid_feature_types
):

    raise RuntimeError(
        "Final feature-type validation failed."
    )


if not all(
    TARGET_VALIDATION_DF[
        "valid"
    ]
):

    raise RuntimeError(
        "One or more target validations failed."
    )


# ============================================================
# 13. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "FEATURE CHARACTERIZATION SUMMARY"
)

print(
    "-" * 100
)

for dataset_id in EXPECTED_DATASETS:

    row = DATASET_FEATURE_SUMMARY_DF[
        DATASET_FEATURE_SUMMARY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ].iloc[0]


    print(
        f"{dataset_id:<20} "
        f"imputation_features="
        f"{row['features_for_imputation']:>3}  "
        f"numerical="
        f"{row['numerical_features']:>3}  "
        f"categorical="
        f"{row['categorical_features']:>3}  "
        f"missing_features="
        f"{row['features_with_missing']:>3}"
    )


print(
    "\nFeature-type registry datasets : "
    f"{len(FEATURE_TYPE_REGISTRY)}"
)

print(
    "Feature characterization records: "
    f"{len(FEATURE_CHARACTERIZATION_DF):,}"
)

print(
    "Numerical feature records       : "
    f"{(
        FEATURE_CHARACTERIZATION_DF['feature_type']
        == 'numerical'
    ).sum():,}"
)

print(
    "Categorical feature records     : "
    f"{(
        FEATURE_CHARACTERIZATION_DF['feature_type']
        == 'categorical'
    ).sum():,}"
)


print(
    "\n" + "=" * 100
)

print(
    "FEATURE CHARACTERIZATION / VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.06
FEATURE CHARACTERIZATION / VALIDATION

Constant features detected:


,dataset_id,feature,feature_type,unique_values
69,diabetes_130us,examide,categorical,1
70,diabetes_130us,citoglipton,categorical,1



[06.06.01] Feature characterization completed.
Total feature records : 80

DATASET FEATURE SUMMARY


,dataset_id,total_columns,target,features_for_imputation,numerical_features,categorical_features,features_with_missing,constant_features
0,adult_income,15,income,14,6,8,3,0
1,bank_marketing,17,y,16,7,9,0,0
2,diabetes_130us,48,readmitted,47,11,36,9,2



TARGET VALIDATION


,dataset_id,target,feature_type,dtype,unique_values,missing_count,valid
0,adult_income,income,categorical,object,2,0,True
1,bank_marketing,y,categorical,object,2,0,True
2,diabetes_130us,readmitted,categorical,object,3,0,True



FEATURE CHARACTERIZATION


,dataset_id,feature,feature_type,dtype,is_target,rows,observed_count,missing_count,missing_rate,unique_values,cardinality_ratio,constant_feature,all_missing,minimum,maximum,mean,std
0,adult_income,age,numerical,int64,False,32561,32561,0,0.000000,73,0.002242,False,False,17.0,90.0,38.581647,13.640223
1,adult_income,workclass,categorical,object,False,32561,30725,1836,0.056386,8,0.000260,False,False,NaN,NaN,NaN,NaN
2,adult_income,fnlwgt,numerical,int64,False,32561,32561,0,0.000000,21648,0.664844,False,False,12285.0,1484705.0,189778.366512,105548.356881
3,adult_income,education,categorical,object,False,32561,32561,0,0.000000,16,0.000491,False,False,NaN,NaN,NaN,NaN
4,adult_income,education_num,numerical,int64,False,32561,32561,0,0.000000,16,0.000491,False,False,1.0,16.0,10.080679,2.572681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin-rosiglitazone,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
76,diabetes_130us,metformin-pioglitazone,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
77,diabetes_130us,change,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
78,diabetes_130us,diabetesMed,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN



FEATURE CHARACTERIZATION SUMMARY
----------------------------------------------------------------------------------------------------
adult_income         imputation_features= 14  numerical=  6  categorical=  8  missing_features=  3
bank_marketing       imputation_features= 16  numerical=  7  categorical=  9  missing_features=  0
diabetes_130us       imputation_features= 47  numerical= 11  categorical= 36  missing_features=  9

Feature-type registry datasets : 3
Feature characterization records: 80
Numerical feature records       : 24
Categorical feature records     : 56

FEATURE CHARACTERIZATION / VALIDATION : PASSED


In [7]:
# ============================================================
# NOTEBOOK 06.07 — PREDICTOR SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.07")
print("LEAKAGE-SAFE / DATA-AWARE PREDICTOR SELECTION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_REGISTRY",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required predictor-selection state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.06 first."
    )


# ============================================================
# 2. SAFE FEATURE ASSOCIATION SCORE
# ============================================================

def _association_score(
    series,
    target_series,
    feature_type,
    target_type
):
    """
    Estimate univariate association between a candidate
    predictor and the target.

    This score is used ONLY for predictor prioritization.
    It is not an evaluation metric for imputation quality.
    """

    valid_mask = (
        series.notna()
        &
        target_series.notna()
    )


    if valid_mask.sum() < 10:
        return 0.0


    x = series.loc[
        valid_mask
    ]

    y = target_series.loc[
        valid_mask
    ]


    # --------------------------------------------------------
    # Numerical predictor
    # --------------------------------------------------------

    if feature_type == "numerical":

        x_numeric = pd.to_numeric(
            x,
            errors="coerce"
        )

        y_numeric = pd.to_numeric(
            y,
            errors="coerce"
        )


        valid_numeric = (
            x_numeric.notna()
            &
            y_numeric.notna()
        )


        if valid_numeric.sum() < 10:
            return 0.0


        x_numeric = x_numeric.loc[
            valid_numeric
        ]

        y_numeric = y_numeric.loc[
            valid_numeric
        ]


        # If target is numerical, use absolute
        # Spearman correlation.
        if target_type == "numerical":

            correlation = (
                x_numeric
                .corr(
                    y_numeric,
                    method="spearman"
                )
            )

            if pd.isna(correlation):
                return 0.0

            return float(
                abs(correlation)
            )


        # Numerical predictor against categorical target.
        # Rank correlation with encoded target provides a
        # deterministic screening signal.
        y_codes = (
            pd.Series(
                pd.factorize(
                    y_numeric
                )[0],
                index=y_numeric.index
            )
        )


        correlation = (
            x_numeric
            .corr(
                y_codes,
                method="spearman"
            )
        )


        if pd.isna(correlation):
            return 0.0


        return float(
            abs(correlation)
        )


    # --------------------------------------------------------
    # Categorical predictor
    # --------------------------------------------------------

    x_codes = pd.Series(
        pd.factorize(x)[0],
        index=x.index
    )


    if target_type == "numerical":

        y_numeric = pd.to_numeric(
            y,
            errors="coerce"
        )


        valid_numeric = (
            y_numeric.notna()
        )


        if valid_numeric.sum() < 10:
            return 0.0


        # Eta-like association using between-category
        # variance relative to total variance.
        y_valid = y_numeric.loc[
            valid_numeric
        ]

        x_valid = x_codes.loc[
            valid_numeric
        ]


        overall_mean = float(
            y_valid.mean()
        )


        total_variance = float(
            ((y_valid - overall_mean) ** 2).sum()
        )


        if total_variance <= 0:
            return 0.0


        between_variance = 0.0


        for category in x_valid.unique():

            group = y_valid[
                x_valid == category
            ]


            if len(group) == 0:
                continue


            between_variance += (
                len(group)
                *
                (
                    float(group.mean())
                    -
                    overall_mean
                ) ** 2
            )


        score = (
            between_variance
            /
            total_variance
        )


        return float(
            np.clip(
                score,
                0.0,
                1.0
            )
        )


    # --------------------------------------------------------
    # Categorical predictor vs categorical target
    # --------------------------------------------------------

    # Cramér's V
    contingency = pd.crosstab(
        x,
        y
    )


    if (
        contingency.shape[0] < 2
        or
        contingency.shape[1] < 2
    ):
        return 0.0


    observed = (
        contingency
        .to_numpy(
            dtype=float
        )
    )


    total = observed.sum()


    if total <= 0:
        return 0.0


    row_totals = observed.sum(
        axis=1,
        keepdims=True
    )

    column_totals = observed.sum(
        axis=0,
        keepdims=True
    )


    expected = (
        row_totals
        *
        column_totals
        /
        total
    )


    valid_expected = (
        expected > 0
    )


    chi_square = np.sum(
        (
            (
                observed
                -
                expected
            ) ** 2
            /
            np.where(
                valid_expected,
                expected,
                1.0
            )
        )
    )


    n = observed.sum()

    phi2 = (
        chi_square / n
    )


    rows, columns = observed.shape

    denominator = min(
        columns - 1,
        rows - 1
    )


    if denominator <= 0:
        return 0.0


    return float(
        np.sqrt(
            phi2 / denominator
        )
    )


# ============================================================
# 3. MAIN PREDICTOR SELECTION FUNCTION
# ============================================================

def select_predictors(
    df,
    target_feature,
    candidate_feature=None,
    max_predictors=20,
    min_observed_fraction=0.50
):
    """
    Select context predictors for imputation.

    Design principles:
      1. Exclude the target.
      2. Exclude the feature currently being imputed.
      3. Remove all-missing and near-empty predictors.
      4. Prefer predictors with stronger association
         with the target.
      5. Prefer predictors with lower missingness when
         association is comparable.
      6. Preserve deterministic ordering.
    """

    if not isinstance(
        df,
        pd.DataFrame
    ):
        raise TypeError(
            "df must be a pandas DataFrame."
        )


    if target_feature not in df.columns:

        raise KeyError(
            f"Target '{target_feature}' not found."
        )


    if max_predictors < 1:

        raise ValueError(
            "max_predictors must be >= 1."
        )


    # --------------------------------------------------------
    # Candidate exclusion
    # --------------------------------------------------------

    excluded = {
        target_feature
    }


    if candidate_feature is not None:

        if candidate_feature not in df.columns:

            raise KeyError(
                f"Candidate feature "
                f"'{candidate_feature}' not found."
            )

        excluded.add(
            candidate_feature
        )


    candidates = [
        column
        for column in df.columns
        if column not in excluded
    ]


    if not candidates:
        return []


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_type = (
        FEATURE_TYPE_REGISTRY
        .get(
            target_feature
        )
    )


    # Dataset-aware registry
    if target_feature not in FEATURE_TYPE_REGISTRY:

        dataset_match = None

        for dataset_id, registry in (
            FEATURE_TYPE_REGISTRY.items()
        ):

            if target_feature in registry:

                dataset_match = registry[
                    target_feature
                ]

                break


        target_type = dataset_match


    if target_type is None:

        target_type = (
            "numerical"
            if pd.api.types.is_numeric_dtype(
                df[target_feature]
            )
            else "categorical"
        )


    # --------------------------------------------------------
    # Score candidate predictors
    # --------------------------------------------------------

    predictor_rows = []


    for feature in candidates:

        series = df[
            feature
        ]


        observed_fraction = float(
            series.notna().mean()
        )


        missing_rate = float(
            series.isna().mean()
        )


        unique_values = int(
            series.nunique(
                dropna=True
            )
        )


        # Exclude predictors that provide too little
        # observed information.
        if (
            observed_fraction
            <
            min_observed_fraction
        ):
            continue


        # Exclude constant predictors.
        if unique_values <= 1:
            continue


        feature_type = None


        # Try to retrieve type from the registry.
        for dataset_id, registry in (
            FEATURE_TYPE_REGISTRY.items()
        ):

            if feature in registry:

                feature_type = registry[
                    feature
                ]

                break


        if feature_type is None:

            feature_type = (
                "numerical"
                if pd.api.types.is_numeric_dtype(
                    series
                )
                else "categorical"
            )


        association = _association_score(
            series=series,
            target_series=df[
                target_feature
            ],
            feature_type=feature_type,
            target_type=target_type,
        )


        predictor_rows.append({

            "feature":
                feature,

            "feature_type":
                feature_type,

            "association_score":
                association,

            "missing_rate":
                missing_rate,

            "observed_fraction":
                observed_fraction,

            "unique_values":
                unique_values,

        })


    if not predictor_rows:
        return []


    predictor_df = pd.DataFrame(
        predictor_rows
    )


    # --------------------------------------------------------
    # Deterministic ranking
    # --------------------------------------------------------

    predictor_df = (
        predictor_df
        .sort_values(
            by=[
                "association_score",
                "observed_fraction",
                "unique_values",
                "feature",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
    )


    predictor_df = (
        predictor_df
        .head(
            max_predictors
        )
    )


    return predictor_df[
        "feature"
    ].tolist()


# ============================================================
# 4. PREDICTOR-SELECTION DIAGNOSTIC
# ============================================================

print(
    "\n[06.07.01] Predictor-selection function: READY"
)

print(
    "[06.07.02] Leakage exclusion: ENABLED"
)

print(
    "[06.07.03] Missingness filtering: ENABLED"
)

print(
    "[06.07.04] Association-based ranking: ENABLED"
)

print(
    "[06.07.05] Deterministic ranking: ENABLED"
)


# ============================================================
# 5. FUNCTION SANITY CHECK
# ============================================================

for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    predictors = select_predictors(
        df=df,
        target_feature=target,
        max_predictors=10,
    )


    if not isinstance(
        predictors,
        list
    ):

        raise RuntimeError(
            f"Predictor selection failed for "
            f"{dataset_id}."
        )


    if target in predictors:

        raise RuntimeError(
            f"Target leakage detected for "
            f"{dataset_id}."
        )


    print(
        f"\n{dataset_id}"
    )

    print(
        f"  Target    : {target}"
    )

    print(
        f"  Predictors: {len(predictors)}"
    )

    print(
        f"  Top       : {predictors[:5]}"
    )


# ============================================================
# 6. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "PREDICTOR SELECTION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.07
LEAKAGE-SAFE / DATA-AWARE PREDICTOR SELECTION

[06.07.01] Predictor-selection function: READY
[06.07.02] Leakage exclusion: ENABLED
[06.07.03] Missingness filtering: ENABLED
[06.07.04] Association-based ranking: ENABLED
[06.07.05] Deterministic ranking: ENABLED

adult_income
  Target    : income
  Predictors: 10
  Top       : ['relationship', 'marital_status', 'education', 'occupation', 'sex']

bank_marketing
  Target    : y
  Predictors: 10
  Top       : ['poutcome', 'month', 'contact', 'housing', 'job']

diabetes_130us
  Target    : readmitted
  Predictors: 10
  Top       : ['diag_1', 'diag_2', 'diag_3', 'medical_specialty', 'payer_code']

PREDICTOR SELECTION : PASSED


In [8]:
# ============================================================
# NOTEBOOK 06.08 — MISSINGNESS MASK GENERATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.08")
print("ROBUST MISSINGNESS-MASK GENERATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "np",
    "pd",
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_REGISTRY",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required state missing for Notebook 06.08:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\nRun Notebook 06.02 through 06.07 first."
    )


# ============================================================
# 2. PREPARE PREDICTORS FOR MAR
# ============================================================

def _prepare_mar_predictors(
    predictors
):
    """
    Convert observed predictors into numeric variables
    suitable for deterministic MAR propensity construction.

    Numerical:
        median imputation + standardization

    Categorical:
        frequency encoding
    """

    if not isinstance(
        predictors,
        pd.DataFrame
    ):
        raise TypeError(
            "predictors must be a pandas DataFrame."
        )

    if predictors.empty:
        return pd.DataFrame(
            index=predictors.index
        )

    encoded = pd.DataFrame(
        index=predictors.index
    )

    for column in predictors.columns:

        series = predictors[column]

        # ----------------------------------------------------
        # Numerical predictor
        # ----------------------------------------------------

        if pd.api.types.is_numeric_dtype(
            series
        ):

            values = pd.to_numeric(
                series,
                errors="coerce"
            )

            median_value = values.median()

            if pd.isna(median_value):
                median_value = 0.0

            values = values.fillna(
                median_value
            )

            std_value = values.std()

            if (
                pd.isna(std_value)
                or
                std_value == 0
            ):

                encoded[column] = 0.0

            else:

                encoded[column] = (
                    values - values.mean()
                ) / std_value

        # ----------------------------------------------------
        # Categorical predictor
        # ----------------------------------------------------

        else:

            values = (
                series
                .astype("string")
                .fillna("__MISSING__")
            )

            frequencies = (
                values
                .value_counts(
                    normalize=True
                )
            )

            encoded[column] = (
                values
                .map(
                    frequencies
                )
                .fillna(0.0)
                .astype(float)
            )

    return encoded


# ============================================================
# 3. BUILD MAR PROPENSITY
# ============================================================

def _build_mar_propensity(
    predictors,
    random_state=42
):
    """
    Construct a deterministic propensity score from
    observed predictor information.
    """

    encoded = _prepare_mar_predictors(
        predictors
    )

    if encoded.empty:
        return None

    values = encoded.to_numpy(
        dtype=float
    )

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    if values.shape[1] == 0:
        return None

    raw_score = values.mean(
        axis=1
    )

    score = (
        pd.Series(
            raw_score,
            index=predictors.index
        )
        .rank(
            method="average",
            pct=True
        )
        .to_numpy()
    )

    rng = np.random.default_rng(
        int(random_state)
    )

    score = (
        score
        +
        rng.uniform(
            0.0,
            1e-10,
            size=len(score)
        )
    )

    return score


# ============================================================
# 4. MAIN MISSINGNESS MASK GENERATOR
# ============================================================

def generate_missingness_mask(
    series,
    rate,
    mechanism="MCAR",
    seed=42,
    predictors=None
):
    """
    Generate a deterministic missingness mask.

    True  = value selected for artificial masking
    False = value remains observed

    Only originally observed values are eligible for masking.
    """

    # --------------------------------------------------------
    # Validate target series
    # --------------------------------------------------------

    if not isinstance(
        series,
        pd.Series
    ):
        raise TypeError(
            "series must be a pandas Series."
        )

    # --------------------------------------------------------
    # Validate mechanism
    # --------------------------------------------------------

    mechanism = str(
        mechanism
    ).strip().upper()

    if mechanism not in {
        "MCAR",
        "MAR",
    }:
        raise ValueError(
            "Supported mechanisms are MCAR and MAR."
        )

    # --------------------------------------------------------
    # Validate rate
    # --------------------------------------------------------

    rate = float(rate)

    if not 0 < rate < 1:
        raise ValueError(
            "Missingness rate must satisfy 0 < rate < 1."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    # --------------------------------------------------------
    # Preserve originally missing values
    # --------------------------------------------------------

    observed = (
        series.notna()
        .to_numpy()
    )

    candidate_indices = np.flatnonzero(
        observed
    )

    mask = pd.Series(
        False,
        index=series.index,
        dtype=bool
    )

    if len(candidate_indices) == 0:
        return mask

    # --------------------------------------------------------
    # Number of values to mask
    # --------------------------------------------------------

    n_remove = int(
        round(
            len(candidate_indices)
            *
            rate
        )
    )

    n_remove = max(
        1,
        min(
            n_remove,
            len(candidate_indices)
        )
    )

    # ========================================================
    # MCAR
    # ========================================================

    if mechanism == "MCAR":

        selected = rng.choice(
            candidate_indices,
            size=n_remove,
            replace=False
        )

    # ========================================================
    # MAR
    # ========================================================

    else:

        if predictors is None:
            raise ValueError(
                "MAR requires observed predictor data."
            )

        if not isinstance(
            predictors,
            pd.DataFrame
        ):
            raise TypeError(
                "predictors must be a pandas DataFrame."
            )

        if not predictors.index.equals(
            series.index
        ):
            raise ValueError(
                "Predictor index must exactly match "
                "the target series index."
            )

        predictor_df = predictors.drop(
            columns=[
                series.name
            ],
            errors="ignore"
        )

        if predictor_df.empty:
            raise ValueError(
                "MAR requires at least one predictor."
            )

        propensity = _build_mar_propensity(
            predictor_df,
            random_state=seed
        )

        if propensity is None:
            raise ValueError(
                "Unable to construct MAR propensity."
            )

        candidate_scores = propensity[
            candidate_indices
        ]

        tie_break = rng.uniform(
            0.0,
            1e-10,
            size=len(candidate_scores)
        )

        ranking_score = (
            candidate_scores
            +
            tie_break
        )

        order = np.argsort(
            ranking_score
        )

        selected = candidate_indices[
            order[-n_remove:]
        ]

    # --------------------------------------------------------
    # Construct mask
    # --------------------------------------------------------

    mask.iloc[
        selected
    ] = True

    return mask


# ============================================================
# 5. MASK VALIDATION FUNCTION
# ============================================================

def validate_missingness_mask(
    series,
    mask,
    requested_rate,
    mechanism
):
    """
    Validate structural correctness and requested masking rate.
    """

    if not isinstance(
        mask,
        pd.Series
    ):
        raise RuntimeError(
            "Missingness mask must be a pandas Series."
        )

    if not mask.index.equals(
        series.index
    ):
        raise RuntimeError(
            "Mask index does not match target series."
        )

    # Never mask values that were already missing.
    illegal_masking = (
        mask
        &
        series.isna()
    )

    if illegal_masking.any():
        raise RuntimeError(
            "Mask attempts to overwrite originally "
            "missing values."
        )

    observed_count = int(
        series.notna().sum()
    )

    masked_count = int(
        mask.sum()
    )

    if observed_count == 0:

        return {
            "valid": True,
            "mechanism": str(mechanism),
            "requested_rate": float(
                requested_rate
            ),
            "actual_rate": 0.0,
            "masked_count": 0,
            "observed_count": 0,
        }

    actual_rate = (
        masked_count
        /
        observed_count
    )

    # Because masking operates on integer cells,
    # the realized rate can differ slightly from the
    # requested rate.
    tolerance = max(
        1.0 / observed_count,
        1e-12
    )

    valid = (
        masked_count > 0
        and
        abs(
            actual_rate
            -
            float(requested_rate)
        )
        <= tolerance
    )

    return {
        "valid": bool(valid),
        "mechanism": str(mechanism),
        "requested_rate": float(
            requested_rate
        ),
        "actual_rate": float(
            actual_rate
        ),
        "masked_count": masked_count,
        "observed_count": observed_count,
    }


# ============================================================
# 6. INITIALIZATION REPORT
# ============================================================

print(
    "\n[06.08.01] Missingness generator initialized."
)

print(
    "[06.08.02] MCAR generation: ENABLED"
)

print(
    "[06.08.03] MAR generation: ENABLED"
)

print(
    "[06.08.04] Numerical predictors: ENABLED"
)

print(
    "[06.08.05] Categorical predictors: ENABLED"
)

print(
    "[06.08.06] Original missing-value protection: ENABLED"
)

print(
    "[06.08.07] Deterministic seed control: ENABLED"
)


# ============================================================
# 7. SANITY-CHECK DATASET
# ============================================================

_test_dataset = EXPECTED_DATASETS[0]

_test_df = EVALUATION_DATA[
    _test_dataset
]

_test_target = TARGET_REGISTRY[
    _test_dataset
]

# Use age when available; otherwise use first
# non-target feature.
if "age" in _test_df.columns:

    _test_feature = "age"

else:

    _test_feature = next(
        column
        for column in _test_df.columns
        if column != _test_target
    )


# ============================================================
# 8. BUILD TEST PREDICTOR SET
# ============================================================

_test_predictor_names = select_predictors(
    _test_df,
    target_feature=_test_target,
    candidate_feature=_test_feature,
    max_predictors=10,
)

_test_predictors = _test_df[
    _test_predictor_names
].copy()


# ============================================================
# 9. MCAR TEST
# ============================================================

_test_mcar_mask = generate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    rate=0.10,
    mechanism="MCAR",
    seed=42,
    predictors=_test_predictors,
)

_test_mcar_validation = validate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    mask=_test_mcar_mask,
    requested_rate=0.10,
    mechanism="MCAR",
)


# ============================================================
# 10. MAR TEST
# ============================================================

_test_mar_mask = generate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    rate=0.10,
    mechanism="MAR",
    seed=42,
    predictors=_test_predictors,
)

_test_mar_validation = validate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    mask=_test_mar_mask,
    requested_rate=0.10,
    mechanism="MAR",
)


# ============================================================
# 11. VALIDATION REPORT
# ============================================================

MASK_VALIDATION_DF = pd.DataFrame([
    _test_mcar_validation,
    _test_mar_validation,
])


display(
    MASK_VALIDATION_DF
)


if not bool(
    MASK_VALIDATION_DF[
        "valid"
    ].all()
):
    raise RuntimeError(
        "Missingness-mask validation failed."
    )


# ============================================================
# 12. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "MISSINGNESS-MASK GENERATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.08
ROBUST MISSINGNESS-MASK GENERATION

[06.08.01] Missingness generator initialized.
[06.08.02] MCAR generation: ENABLED
[06.08.03] MAR generation: ENABLED
[06.08.04] Numerical predictors: ENABLED
[06.08.05] Categorical predictors: ENABLED
[06.08.06] Original missing-value protection: ENABLED
[06.08.07] Deterministic seed control: ENABLED


,valid,mechanism,requested_rate,actual_rate,masked_count,observed_count
0,True,MCAR,0.1,0.099997,3256,32561
1,True,MAR,0.1,0.099997,3256,32561



MISSINGNESS-MASK GENERATION : PASSED


In [9]:
# ============================================================
# NOTEBOOK 06.09 — PREDICTOR ENCODING PIPELINE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.09")
print("BUILD LEAKAGE-SAFE PREDICTOR ENCODING PIPELINE")
print("=" * 100)


# ============================================================
# 1. REQUIRED IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# 2. PREDICTOR ENCODING FUNCTION
# ============================================================

def encode_predictors(
    X_train,
    X_test,
    sparse_output=True
):
    """
    Build a leakage-safe predictor preprocessing pipeline.

    Numerical predictors:
        median imputation
        standardization

    Categorical predictors:
        most-frequent imputation
        one-hot encoding
        unseen categories ignored

    The encoder is fitted ONLY on X_train and subsequently
    applied to X_test.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Training predictors.

    X_test : pandas.DataFrame
        Validation/test predictors.

    sparse_output : bool
        Whether one-hot encoded output should remain sparse.

    Returns
    -------
    X_train_encoded
    X_test_encoded
    predictor_encoder
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        X_test,
        pd.DataFrame
    ):
        raise TypeError(
            "X_test must be a pandas DataFrame."
        )

    if X_train.empty:
        raise ValueError(
            "X_train contains no predictor columns."
        )

    if X_test.empty:
        raise ValueError(
            "X_test contains no predictor columns."
        )

    # --------------------------------------------------------
    # Ensure identical predictor schema
    # --------------------------------------------------------

    if list(
        X_train.columns
    ) != list(
        X_test.columns
    ):

        raise ValueError(
            "X_train and X_test must contain the same "
            "predictor columns in the same order."
        )

    # --------------------------------------------------------
    # Copy inputs
    # --------------------------------------------------------

    X_train = X_train.copy()

    X_test = X_test.copy()

    # --------------------------------------------------------
    # Detect numerical predictors
    # --------------------------------------------------------

    numerical_columns = (
        X_train
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    # --------------------------------------------------------
    # Detect categorical predictors
    # --------------------------------------------------------

    categorical_columns = [
        column
        for column in X_train.columns
        if column not in numerical_columns
    ]

    transformers = []

    # ========================================================
    # 3. NUMERICAL PIPELINE
    # ========================================================

    if numerical_columns:

        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
        ])

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_columns,
            )
        )

    # ========================================================
    # 4. CATEGORICAL PIPELINE
    # ========================================================

    if categorical_columns:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=sparse_output,
                    dtype=np.float32,
                )
            ),
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    # --------------------------------------------------------
    # Validate transformer construction
    # --------------------------------------------------------

    if not transformers:

        raise RuntimeError(
            "Unable to construct predictor preprocessing "
            "pipeline."
        )

    # ========================================================
    # 5. COLUMN TRANSFORMER
    # ========================================================

    predictor_encoder = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # ========================================================
    # 6. FIT ONLY ON TRAINING DATA
    # ========================================================

    X_train_encoded = (
        predictor_encoder
        .fit_transform(
            X_train
        )
    )

    # ========================================================
    # 7. TRANSFORM TEST DATA
    # ========================================================

    X_test_encoded = (
        predictor_encoder
        .transform(
            X_test
        )
    )

    # ========================================================
    # 8. OUTPUT VALIDATION
    # ========================================================

    if X_train_encoded.shape[1] == 0:

        raise RuntimeError(
            "Predictor encoding produced zero features."
        )

    if (
        X_train_encoded.shape[1]
        !=
        X_test_encoded.shape[1]
    ):

        raise RuntimeError(
            "Encoded train/test feature dimensions differ."
        )

    return (
        X_train_encoded,
        X_test_encoded,
        predictor_encoder,
    )


# ============================================================
# 9. DIAGNOSTIC REPORT
# ============================================================

print(
    "\n[06.09.01] Numerical preprocessing : "
    "median imputation + standardization"
)

print(
    "[06.09.02] Categorical preprocessing : "
    "mode imputation + one-hot encoding"
)

print(
    "[06.09.03] Unknown categories : "
    "ignored safely"
)

print(
    "[06.09.04] Encoder fitting : "
    "TRAINING DATA ONLY"
)

print(
    "[06.09.05] Sparse encoding : "
    "AVAILABLE"
)

print(
    "[06.09.06] Leakage protection : "
    "ENABLED"
)


# ============================================================
# 10. FUNCTION SANITY CHECK
# ============================================================

_test_dataset = EXPECTED_DATASETS[0]

_test_df = EVALUATION_DATA[
    _test_dataset
]

_test_target = TARGET_REGISTRY[
    _test_dataset
]

_test_feature = (
    "age"
    if "age" in _test_df.columns
    else next(
        column
        for column in _test_df.columns
        if column != _test_target
    )
)

_test_predictor_names = select_predictors(
    _test_df,
    target_feature=_test_target,
    candidate_feature=_test_feature,
    max_predictors=5,
)

_test_X = _test_df[
    _test_predictor_names
].copy()

_test_split = max(
    1,
    int(
        len(_test_X) * 0.8
    )
)

_test_X_train = _test_X.iloc[
    :_test_split
].copy()

_test_X_test = _test_X.iloc[
    _test_split:
].copy()


(
    _test_train_encoded,
    _test_test_encoded,
    _test_encoder,
) = encode_predictors(
    _test_X_train,
    _test_X_test,
    sparse_output=True,
)


if (
    _test_train_encoded.shape[1]
    !=
    _test_test_encoded.shape[1]
):

    raise RuntimeError(
        "06.09 sanity check failed: "
        "train/test encoded dimensions differ."
    )


print(
    "\n[06.09.07] Sanity check dataset : "
    f"{_test_dataset}"
)

print(
    "[06.09.08] Predictor count : "
    f"{len(_test_predictor_names)}"
)

print(
    "[06.09.09] Encoded dimensions : "
    f"{_test_train_encoded.shape[1]}"
)

print(
    "\n" + "=" * 100
)

print(
    "PREDICTOR ENCODING PIPELINE : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.09
BUILD LEAKAGE-SAFE PREDICTOR ENCODING PIPELINE

[06.09.01] Numerical preprocessing : median imputation + standardization
[06.09.02] Categorical preprocessing : mode imputation + one-hot encoding
[06.09.03] Unknown categories : ignored safely
[06.09.04] Encoder fitting : TRAINING DATA ONLY
[06.09.05] Sparse encoding : AVAILABLE
[06.09.06] Leakage protection : ENABLED

[06.09.07] Sanity check dataset : adult_income
[06.09.08] Predictor count : 5
[06.09.09] Encoded dimensions : 45

PREDICTOR ENCODING PIPELINE : PASSED


In [10]:
# ============================================================
# NOTEBOOK 06.10 — STATISTICAL CANDIDATE IMPUTERS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.10")
print("STATISTICAL CANDIDATE IMPUTERS")
print("=" * 100)


# ============================================================
# 1. REQUIRED IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. UTILITY: VALIDATE INPUT
# ============================================================

def _validate_imputation_inputs(
    train_series,
    test_series
):

    if not isinstance(
        train_series,
        pd.Series
    ):
        raise TypeError(
            "train_series must be a pandas Series."
        )

    if not isinstance(
        test_series,
        pd.Series
    ):
        raise TypeError(
            "test_series must be a pandas Series."
        )

    if train_series.empty:
        raise ValueError(
            "train_series is empty."
        )

    if test_series.empty:
        raise ValueError(
            "test_series is empty."
        )

    if train_series.isna().all():
        raise ValueError(
            "train_series contains no observed values."
        )


# ============================================================
# 3. MEAN IMPUTATION
# ============================================================

def impute_mean(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if not pd.api.types.is_numeric_dtype(
        train_series
    ):
        raise TypeError(
            "Mean imputation requires a numerical feature."
        )

    value = train_series.mean()

    if pd.isna(value):
        raise ValueError(
            "Unable to estimate mean from training data."
        )

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 4. MEDIAN IMPUTATION
# ============================================================

def impute_median(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if not pd.api.types.is_numeric_dtype(
        train_series
    ):
        raise TypeError(
            "Median imputation requires a numerical feature."
        )

    value = train_series.median()

    if pd.isna(value):
        raise ValueError(
            "Unable to estimate median from training data."
        )

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 5. MODE IMPUTATION
# ============================================================

def impute_mode(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    mode_values = (
        train_series
        .dropna()
        .mode()
    )

    if mode_values.empty:
        raise ValueError(
            "Unable to estimate mode from training data."
        )

    value = mode_values.iloc[0]

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 6. CONSTANT IMPUTATION
# ============================================================

def impute_constant(
    train_series,
    test_series,
    constant_value=None
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if constant_value is None:

        if pd.api.types.is_numeric_dtype(
            train_series
        ):
            constant_value = 0.0

        else:
            constant_value = "MISSING"

    result = test_series.copy()

    result = result.fillna(
        constant_value
    )

    return result


# ============================================================
# 7. RANDOM-SAMPLE IMPUTATION
# ============================================================

def impute_random_sample(
    train_series,
    test_series,
    random_state=42
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    observed = (
        train_series
        .dropna()
        .to_numpy()
    )

    if len(observed) == 0:
        raise ValueError(
            "Random-sample imputation requires "
            "observed training values."
        )

    result = test_series.copy()

    missing_mask = result.isna()

    missing_count = int(
        missing_mask.sum()
    )

    if missing_count == 0:
        return result

    rng = np.random.default_rng(
        int(random_state)
    )

    sampled_values = rng.choice(
        observed,
        size=missing_count,
        replace=True
    )

    result.loc[
        missing_mask
    ] = sampled_values

    return result


# ============================================================
# 8. STRATEGY REGISTRY
# ============================================================

STATISTICAL_IMPUTERS = {

    "mean":
        impute_mean,

    "median":
        impute_median,

    "mode":
        impute_mode,

    "constant":
        impute_constant,

    "random_sample":
        impute_random_sample,
}


# ============================================================
# 9. BASIC IMPLEMENTATION VALIDATION
# ============================================================

if not STATISTICAL_IMPUTERS:

    raise RuntimeError(
        "Statistical imputer registry is empty."
    )


required_statistical_strategies = {
    "mean",
    "median",
    "mode",
    "constant",
    "random_sample",
}


missing_statistical_strategies = (
    required_statistical_strategies
    -
    set(STATISTICAL_IMPUTERS.keys())
)


if missing_statistical_strategies:

    raise RuntimeError(
        "Missing statistical strategies:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_statistical_strategies
            )
        )
    )


# ============================================================
# 10. FUNCTION SANITY TEST
# ============================================================

_test_train_numeric = pd.Series(
    [10.0, 20.0, 30.0, 40.0],
    name="numeric_test"
)

_test_masked_numeric = pd.Series(
    [10.0, np.nan, 30.0, np.nan],
    name="numeric_test"
)

_test_mean = impute_mean(
    _test_train_numeric,
    _test_masked_numeric
)

_test_median = impute_median(
    _test_train_numeric,
    _test_masked_numeric
)

_test_random = impute_random_sample(
    _test_train_numeric,
    _test_masked_numeric,
    random_state=42
)


if _test_mean.isna().any():
    raise RuntimeError(
        "Mean imputation sanity check failed."
    )

if _test_median.isna().any():
    raise RuntimeError(
        "Median imputation sanity check failed."
    )

if _test_random.isna().any():
    raise RuntimeError(
        "Random-sample imputation sanity check failed."
    )


_test_train_categorical = pd.Series(
    ["A", "B", "A", "C"],
    name="categorical_test"
)

_test_masked_categorical = pd.Series(
    ["A", np.nan, "A", np.nan],
    name="categorical_test"
)

_test_mode = impute_mode(
    _test_train_categorical,
    _test_masked_categorical
)

_test_constant = impute_constant(
    _test_train_categorical,
    _test_masked_categorical
)

if _test_mode.isna().any():
    raise RuntimeError(
        "Mode imputation sanity check failed."
    )

if _test_constant.isna().any():
    raise RuntimeError(
        "Constant imputation sanity check failed."
    )


# ============================================================
# 11. FINAL REPORT
# ============================================================

print(
    "\n[06.10.01] Mean imputation       : READY"
)

print(
    "[06.10.02] Median imputation     : READY"
)

print(
    "[06.10.03] Mode imputation       : READY"
)

print(
    "[06.10.04] Constant imputation   : READY"
)

print(
    "[06.10.05] Random-sample         : READY"
)

print(
    "[06.10.06] Training-only fitting : ENABLED"
)

print(
    "[06.10.07] Leakage protection    : ENABLED"
)

print(
    "\nRegistered statistical strategies:"
)

for strategy_id in sorted(
    STATISTICAL_IMPUTERS
):

    print(
        f"  ✓ {strategy_id}"
    )


print(
    "\nSanity tests: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "STATISTICAL CANDIDATE IMPUTERS : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.10
STATISTICAL CANDIDATE IMPUTERS

[06.10.01] Mean imputation       : READY
[06.10.02] Median imputation     : READY
[06.10.03] Mode imputation       : READY
[06.10.04] Constant imputation   : READY
[06.10.05] Random-sample         : READY
[06.10.06] Training-only fitting : ENABLED
[06.10.07] Leakage protection    : ENABLED

Registered statistical strategies:
  ✓ constant
  ✓ mean
  ✓ median
  ✓ mode
  ✓ random_sample

Sanity tests: PASSED

STATISTICAL CANDIDATE IMPUTERS : PASSED


In [11]:
# ============================================================
# NOTEBOOK 06.11 — KNN IMPUTATION CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.11")
print("LEAKAGE-SAFE KNN IMPUTATION CANDIDATE")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 2. KNN IMPUTATION FUNCTION
# ============================================================

def fit_knn(
    X_train,
    X_test=None,
    n_neighbors=5,
    weights="uniform"
):
    """
    Leakage-safe KNN imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame or numpy.ndarray
        Training feature matrix containing missing values.

    X_test : pandas.DataFrame or numpy.ndarray, optional
        Validation/test feature matrix containing missing values.

    n_neighbors : int
        Number of nearest neighbours.

    weights : {"uniform", "distance"}
        Weighting scheme used by KNNImputer.

    Returns
    -------
    X_train_imputed
        Imputed training matrix.

    X_test_imputed
        Imputed test matrix, or None.

    imputer
        Fitted KNNImputer.

    scaler
        Fitted StandardScaler.
    """

    # --------------------------------------------------------
    # Validate number of neighbours
    # --------------------------------------------------------

    if not isinstance(
        n_neighbors,
        int
    ):
        raise TypeError(
            "n_neighbors must be an integer."
        )

    if n_neighbors < 1:
        raise ValueError(
            "n_neighbors must be >= 1."
        )

    if weights not in {
        "uniform",
        "distance"
    }:

        raise ValueError(
            "weights must be 'uniform' or 'distance'."
        )

    # --------------------------------------------------------
    # Convert to DataFrame for consistent handling
    # --------------------------------------------------------

    if isinstance(
        X_train,
        pd.DataFrame
    ):

        train_columns = (
            X_train.columns.tolist()
        )

        X_train_work = X_train.copy()

    else:

        X_train_work = pd.DataFrame(
            X_train
        )

        train_columns = (
            X_train_work.columns.tolist()
        )

    if X_train_work.empty:

        raise ValueError(
            "X_train contains no features."
        )

    # --------------------------------------------------------
    # Validate numeric input
    # --------------------------------------------------------

    non_numeric_columns = (
        X_train_work
        .select_dtypes(
            exclude=[np.number]
        )
        .columns
        .tolist()
    )

    if non_numeric_columns:

        raise TypeError(
            "KNN candidate requires numerical input. "
            "Encode categorical predictors before calling "
            "fit_knn(). Non-numerical columns: "
            +
            ", ".join(
                map(
                    str,
                    non_numeric_columns
                )
            )
        )

    # --------------------------------------------------------
    # Validate finite values
    # --------------------------------------------------------

    X_train_work = X_train_work.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X_test is not None:

        if isinstance(
            X_test,
            pd.DataFrame
        ):

            if list(
                X_test.columns
            ) != train_columns:

                raise ValueError(
                    "X_train and X_test must contain "
                    "identical columns in identical order."
                )

            X_test_work = X_test.copy()

        else:

            X_test_work = pd.DataFrame(
                X_test,
                columns=train_columns
            )

        X_test_work = X_test_work.replace(
            [np.inf, -np.inf],
            np.nan
        )

    else:

        X_test_work = None

    # ========================================================
    # 3. STANDARDIZATION
    # ========================================================

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train_work
        )
    )

    if X_test_work is not None:

        X_test_scaled = (
            scaler.transform(
                X_test_work
            )
        )

    else:

        X_test_scaled = None

    # ========================================================
    # 4. KNN IMPUTATION
    # ========================================================

    effective_neighbors = min(
        n_neighbors,
        max(
            1,
            len(X_train_work) - 1
        )
    )

    imputer = KNNImputer(
        n_neighbors=effective_neighbors,
        weights=weights
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train_scaled
        )
    )

    if X_test_scaled is not None:

        X_test_imputed = (
            imputer.transform(
                X_test_scaled
            )
        )

    else:

        X_test_imputed = None

    # ========================================================
    # 5. OUTPUT VALIDATION
    # ========================================================

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "KNN imputation produced non-finite "
            "training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "KNN imputation produced non-finite "
            "test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer,
        scaler
    )


# ============================================================
# 6. KNN STRATEGY WRAPPER
# ============================================================

def impute_knn(
    train_matrix,
    test_matrix,
    n_neighbors=5,
    weights="uniform"
):
    """
    Convenience wrapper for AIR-LLM candidate evaluation.
    """

    (
        train_imputed,
        test_imputed,
        imputer,
        scaler
    ) = fit_knn(
        train_matrix,
        test_matrix,
        n_neighbors=n_neighbors,
        weights=weights
    )

    return {
        "train_imputed": train_imputed,
        "test_imputed": test_imputed,
        "imputer": imputer,
        "scaler": scaler,
        "n_neighbors": n_neighbors,
        "weights": weights,
    }


# ============================================================
# 7. KNN CONFIGURATION
# ============================================================

KNN_CONFIG = {

    "strategy_id":
        "knn",

    "method":
        "KNNImputer",

    "n_neighbors":
        5,

    "weights":
        "uniform",

    "scaling":
        "standard",

    "leakage_safe":
        True,
}


# ============================================================
# 8. SANITY TEST
# ============================================================

_test_train = pd.DataFrame({

    "x1": [
        1.0,
        2.0,
        3.0,
        4.0,
        5.0
    ],

    "x2": [
        10.0,
        20.0,
        np.nan,
        40.0,
        50.0
    ],

    "x3": [
        100.0,
        np.nan,
        300.0,
        400.0,
        500.0
    ],
})


_test_test = pd.DataFrame({

    "x1": [
        6.0,
        7.0
    ],

    "x2": [
        np.nan,
        70.0
    ],

    "x3": [
        600.0,
        np.nan
    ],
})


(
    _test_train_imputed,
    _test_test_imputed,
    _test_imputer,
    _test_scaler
) = fit_knn(
    _test_train,
    _test_test,
    n_neighbors=3
)


if not np.isfinite(
    _test_train_imputed
).all():

    raise RuntimeError(
        "KNN training sanity test failed."
    )


if not np.isfinite(
    _test_test_imputed
).all():

    raise RuntimeError(
        "KNN test sanity test failed."
    )


if (
    _test_train_imputed.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "KNN training output shape changed."
    )


if (
    _test_test_imputed.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "KNN test output shape changed."
    )


# ============================================================
# 9. FINAL VALIDATION REPORT
# ============================================================

print(
    "\n[06.11.01] KNN imputer              : READY"
)

print(
    "[06.11.02] Distance scaling         : ENABLED"
)

print(
    "[06.11.03] Training-only fitting    : ENABLED"
)

print(
    "[06.11.04] Test transformation      : ENABLED"
)

print(
    "[06.11.05] Non-finite protection    : ENABLED"
)

print(
    "[06.11.06] Parameter validation     : ENABLED"
)

print(
    "[06.11.07] Reproducible configuration: READY"
)

print(
    "\nKNN configuration:"
)

for key, value in KNN_CONFIG.items():

    print(
        f"  {key:<20}: {value}"
    )

print(
    "\nSanity test: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "KNN IMPUTATION CANDIDATE : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.11
LEAKAGE-SAFE KNN IMPUTATION CANDIDATE

[06.11.01] KNN imputer              : READY
[06.11.02] Distance scaling         : ENABLED
[06.11.03] Training-only fitting    : ENABLED
[06.11.04] Test transformation      : ENABLED
[06.11.05] Non-finite protection    : ENABLED
[06.11.06] Parameter validation     : ENABLED
[06.11.07] Reproducible configuration: READY

KNN configuration:
  strategy_id         : knn
  method              : KNNImputer
  n_neighbors         : 5
  weights             : uniform
  scaling             : standard
  leakage_safe        : True

Sanity test: PASSED

KNN IMPUTATION CANDIDATE : PASSED


In [12]:
# ============================================================
# NOTEBOOK 06.12 — ITERATIVE / MICE CANDIDATES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.12")
print("LEAKAGE-SAFE ITERATIVE / MICE IMPUTATION CANDIDATES")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.experimental import (
    enable_iterative_imputer  # noqa: F401
)

from sklearn.impute import IterativeImputer

from sklearn.linear_model import BayesianRidge


# ============================================================
# 2. COMMON VALIDATION
# ============================================================

def _validate_iterative_inputs(
    X_train,
    X_test=None
):

    if isinstance(
        X_train,
        pd.DataFrame
    ):

        X_train_work = X_train.copy()

    else:

        X_train_work = pd.DataFrame(
            X_train
        )

    if X_train_work.empty:

        raise ValueError(
            "X_train contains no features."
        )

    non_numeric = (
        X_train_work
        .select_dtypes(
            exclude=[np.number]
        )
        .columns
        .tolist()
    )

    if non_numeric:

        raise TypeError(
            "Iterative/MICE imputation requires "
            "numerical input. Encode categorical "
            "predictors before calling this function. "
            "Non-numerical columns: "
            +
            ", ".join(
                map(
                    str,
                    non_numeric
                )
            )
        )

    X_train_work = (
        X_train_work
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    if X_test is not None:

        if isinstance(
            X_test,
            pd.DataFrame
        ):

            X_test_work = X_test.copy()

            if list(
                X_train_work.columns
            ) != list(
                X_test_work.columns
            ):

                raise ValueError(
                    "X_train and X_test must contain "
                    "identical columns in identical order."
                )

        else:

            X_test_work = pd.DataFrame(
                X_test,
                columns=X_train_work.columns
            )

        X_test_work = (
            X_test_work
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

    else:

        X_test_work = None

    return (
        X_train_work,
        X_test_work
    )


# ============================================================
# 3. ITERATIVE IMPUTER
# ============================================================

def fit_iterative(
    X_train,
    X_test=None,
    estimator=None,
    random_state=42,
    max_iter=10,
    tol=1e-3,
    initial_strategy="median"
):
    """
    Leakage-safe iterative regression imputation.

    The imputer is fitted only on X_train and then
    applied to X_test.

    Suitable for numerical feature representations.
    """

    (
        X_train_work,
        X_test_work
    ) = _validate_iterative_inputs(
        X_train,
        X_test
    )

    # --------------------------------------------------------
    # Default estimator
    # --------------------------------------------------------

    if estimator is None:

        estimator = BayesianRidge()

    # --------------------------------------------------------
    # Construct imputer
    # --------------------------------------------------------

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=int(max_iter),
        tol=float(tol),
        random_state=int(random_state),
        initial_strategy=initial_strategy,
        skip_complete=True,
        sample_posterior=False,
        min_value=None,
        max_value=None,
    )

    # --------------------------------------------------------
    # Fit ONLY on training data
    # --------------------------------------------------------

    X_train_imputed = (
        imputer
        .fit_transform(
            X_train_work
        )
    )

    # --------------------------------------------------------
    # Transform evaluation/test data
    # --------------------------------------------------------

    if X_test_work is not None:

        X_test_imputed = (
            imputer
            .transform(
                X_test_work
            )
        )

    else:

        X_test_imputed = None

    # --------------------------------------------------------
    # Validate output
    # --------------------------------------------------------

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "Iterative imputation produced "
            "non-finite training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "Iterative imputation produced "
            "non-finite test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


# ============================================================
# 4. MICE IMPUTER
# ============================================================

def fit_mice(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=10,
    tol=1e-3
):
    """
    MICE-style stochastic chained-equation imputation.

    sample_posterior=True introduces stochastic posterior
    sampling and therefore makes the candidate distinct from
    deterministic IterativeImputer.

    The model is fitted only on X_train.
    """

    (
        X_train_work,
        X_test_work
    ) = _validate_iterative_inputs(
        X_train,
        X_test
    )

    # --------------------------------------------------------
    # Bayesian Ridge is used as the conditional estimator.
    # --------------------------------------------------------

    estimator = BayesianRidge()

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=int(max_iter),
        tol=float(tol),
        random_state=int(random_state),
        initial_strategy="median",
        skip_complete=True,
        sample_posterior=True,
    )

    # --------------------------------------------------------
    # Fit only on training data
    # --------------------------------------------------------

    X_train_imputed = (
        imputer
        .fit_transform(
            X_train_work
        )
    )

    # --------------------------------------------------------
    # Transform evaluation/test data
    # --------------------------------------------------------

    if X_test_work is not None:

        X_test_imputed = (
            imputer
            .transform(
                X_test_work
            )
        )

    else:

        X_test_imputed = None

    # --------------------------------------------------------
    # Output validation
    # --------------------------------------------------------

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "MICE produced non-finite training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "MICE produced non-finite test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


# ============================================================
# 5. STRATEGY REGISTRY
# ============================================================

ITERATIVE_IMPUTERS = {

    "iterativeimputer":
        fit_iterative,

    "mice":
        fit_mice,
}


# ============================================================
# 6. CONFIGURATION
# ============================================================

ITERATIVE_CONFIG = {

    "iterativeimputer": {

        "estimator":
            "BayesianRidge",

        "max_iter":
            10,

        "tol":
            1e-3,

        "sample_posterior":
            False,

        "initial_strategy":
            "median",

        "leakage_safe":
            True,
    },

    "mice": {

        "estimator":
            "BayesianRidge",

        "max_iter":
            10,

        "tol":
            1e-3,

        "sample_posterior":
            True,

        "initial_strategy":
            "median",

        "leakage_safe":
            True,
    },
}


# ============================================================
# 7. SANITY TEST DATA
# ============================================================

_test_train = pd.DataFrame({

    "x1": [
        1.0,
        2.0,
        np.nan,
        4.0,
        5.0,
        6.0
    ],

    "x2": [
        10.0,
        np.nan,
        30.0,
        40.0,
        np.nan,
        60.0
    ],

    "x3": [
        100.0,
        200.0,
        300.0,
        np.nan,
        500.0,
        600.0
    ],
})


_test_test = pd.DataFrame({

    "x1": [
        np.nan,
        8.0
    ],

    "x2": [
        70.0,
        np.nan
    ],

    "x3": [
        np.nan,
        800.0
    ],
})


# ============================================================
# 8. ITERATIVE SANITY TEST
# ============================================================

(
    _iter_train,
    _iter_test,
    _iter_model
) = fit_iterative(
    _test_train,
    _test_test,
    random_state=42,
    max_iter=10
)


if not np.isfinite(
    _iter_train
).all():

    raise RuntimeError(
        "Iterative imputer sanity test failed."
    )

if not np.isfinite(
    _iter_test
).all():

    raise RuntimeError(
        "Iterative test transformation failed."
    )


# ============================================================
# 9. MICE SANITY TEST
# ============================================================

(
    _mice_train,
    _mice_test,
    _mice_model
) = fit_mice(
    _test_train,
    _test_test,
    random_state=42,
    max_iter=10
)


if not np.isfinite(
    _mice_train
).all():

    raise RuntimeError(
        "MICE sanity test failed."
    )

if not np.isfinite(
    _mice_test
).all():

    raise RuntimeError(
        "MICE test transformation failed."
    )


# ============================================================
# 10. DIMENSION VALIDATION
# ============================================================

if (
    _iter_train.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "Iterative imputer changed training dimensions."
    )

if (
    _iter_test.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "Iterative imputer changed test dimensions."
    )

if (
    _mice_train.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "MICE changed training dimensions."
    )

if (
    _mice_test.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "MICE changed test dimensions."
    )


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

print(
    "\n[06.12.01] Iterative Imputer : READY"
)

print(
    "[06.12.02] MICE candidate    : READY"
)

print(
    "[06.12.03] Bayesian Ridge    : ENABLED"
)

print(
    "[06.12.04] Posterior sampling: MICE ONLY"
)

print(
    "[06.12.05] Training-only fit : ENABLED"
)

print(
    "[06.12.06] Leakage protection: ENABLED"
)

print(
    "[06.12.07] Non-finite checks : ENABLED"
)

print(
    "\nRegistered iterative candidates:"
)

for strategy_id in sorted(
    ITERATIVE_IMPUTERS
):

    print(
        f"  ✓ {strategy_id}"
    )

print(
    "\nSanity tests: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "ITERATIVE / MICE CANDIDATES : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.12
LEAKAGE-SAFE ITERATIVE / MICE IMPUTATION CANDIDATES

[06.12.01] Iterative Imputer : READY
[06.12.02] MICE candidate    : READY
[06.12.03] Bayesian Ridge    : ENABLED
[06.12.04] Posterior sampling: MICE ONLY
[06.12.05] Training-only fit : ENABLED
[06.12.06] Leakage protection: ENABLED
[06.12.07] Non-finite checks : ENABLED

Registered iterative candidates:
  ✓ iterativeimputer
  ✓ mice

Sanity tests: PASSED

ITERATIVE / MICE CANDIDATES : PASSED


In [13]:
# ============================================================
# NOTEBOOK 06.13 — RANDOM FOREST IMPUTATION CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.13")
print("RANDOM FOREST IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Random Forest imputation
# ------------------------------------------------------------

def fit_random_forest(
    X_train,
    target_train,
    X_missing=None,
    feature_type="numerical",
    random_state=42,
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1
):
    """
    Leakage-safe Random Forest imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Predictor matrix corresponding to target_train.

    target_train : pandas.Series
        Target/feature containing missing values.

    X_missing : pandas.DataFrame, optional
        Predictor matrix used for predicting missing target values.
        If None, X_train is used.

    feature_type : str
        'numerical' or 'categorical'.

    random_state : int
        Reproducibility seed.

    n_estimators : int
        Number of trees.

    max_depth : int or None
        Maximum tree depth.

    min_samples_leaf : int
        Minimum observations per leaf.

    n_jobs : int
        Parallel CPU workers.

    Returns
    -------
    result : pandas.Series
        Imputed target.

    model : fitted RandomForest model
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        target_train,
        pd.Series
    ):
        raise TypeError(
            "target_train must be a pandas Series."
        )

    if len(X_train) != len(target_train):
        raise ValueError(
            "X_train and target_train must have "
            "the same number of rows."
        )

    if X_missing is None:
        X_missing = X_train.copy()

    if not isinstance(
        X_missing,
        pd.DataFrame
    ):
        raise TypeError(
            "X_missing must be a pandas DataFrame."
        )

    if len(X_missing) != len(target_train):
        raise ValueError(
            "X_missing and target_train must have "
            "the same number of rows."
        )

    # --------------------------------------------------------
    # Normalize feature type
    # --------------------------------------------------------

    feature_type = str(
        feature_type
    ).strip().lower()

    if feature_type in {
        "numeric",
        "numerical",
        "float",
        "integer",
        "int"
    }:
        feature_type = "numerical"

    elif feature_type in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool"
    }:
        feature_type = "categorical"

    else:
        raise ValueError(
            f"Unsupported feature_type: {feature_type}"
        )

    # --------------------------------------------------------
    # Identify missing values
    # --------------------------------------------------------

    missing_mask = target_train.isna()

    observed_mask = target_train.notna()

    n_observed = int(
        observed_mask.sum()
    )

    n_missing = int(
        missing_mask.sum()
    )

    if n_missing == 0:

        return (
            target_train.copy(),
            None
        )

    if n_observed < 10:

        raise ValueError(
            "Insufficient observed target values "
            f"for Random Forest imputation: {n_observed}"
        )

    # --------------------------------------------------------
    # Align predictor matrices
    # --------------------------------------------------------

    if not X_train.index.equals(
        target_train.index
    ):

        X_train = X_train.reindex(
            target_train.index
        )

    if not X_missing.index.equals(
        target_train.index
    ):

        X_missing = X_missing.reindex(
            target_train.index
        )

    # --------------------------------------------------------
    # Convert predictors to numeric matrix
    #
    # The predictor encoding pipeline should normally already
    # perform this step. This additional safeguard prevents
    # object/string columns from reaching sklearn.
    # --------------------------------------------------------

    if not all(
        pd.api.types.is_numeric_dtype(
            X_train[column]
        )
        for column in X_train.columns
    ):

        encoder_imputer = SimpleImputer(
            strategy="median"
        )

        X_train_numeric = pd.DataFrame(
            encoder_imputer.fit_transform(
                X_train.apply(
                    pd.to_numeric,
                    errors="coerce"
                )
            ),
            index=X_train.index
        )

        X_missing_numeric = pd.DataFrame(
            encoder_imputer.transform(
                X_missing.apply(
                    pd.to_numeric,
                    errors="coerce"
                )
            ),
            index=X_missing.index
        )

    else:

        X_train_numeric = X_train.copy()
        X_missing_numeric = X_missing.copy()

        X_train_numeric = (
            X_train_numeric
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        X_missing_numeric = (
            X_missing_numeric
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        predictor_imputer = SimpleImputer(
            strategy="median"
        )

        X_train_numeric = pd.DataFrame(
            predictor_imputer.fit_transform(
                X_train_numeric
            ),
            index=X_train.index
        )

        X_missing_numeric = pd.DataFrame(
            predictor_imputer.transform(
                X_missing_numeric
            ),
            index=X_missing.index
        )

    # --------------------------------------------------------
    # Prepare target
    # --------------------------------------------------------

    y_observed = (
        target_train.loc[
            observed_mask
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Select model according to feature type
    # --------------------------------------------------------

    if feature_type == "numerical":

        y_observed = pd.to_numeric(
            y_observed,
            errors="coerce"
        )

        valid_target = (
            y_observed.notna()
            &
            np.isfinite(
                y_observed.to_numpy(
                    dtype=float
                )
            )
        )

        y_observed = (
            y_observed.loc[
                valid_target
            ]
        )

        if len(y_observed) < 10:

            raise ValueError(
                "Insufficient valid numerical "
                "target observations."
            )

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state,
            n_jobs=n_jobs
        )

    else:

        y_observed = (
            y_observed
            .astype(str)
        )

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state,
            n_jobs=n_jobs
        )

    # --------------------------------------------------------
    # Align training predictors with valid target rows
    # --------------------------------------------------------

    X_fit = X_train_numeric.loc[
        y_observed.index
    ]

    # --------------------------------------------------------
    # Fit model
    # --------------------------------------------------------

    model.fit(
        X_fit,
        y_observed
    )

    # --------------------------------------------------------
    # Predict missing values
    # --------------------------------------------------------

    X_predict = (
        X_missing_numeric.loc[
            missing_mask
        ]
    )

    predictions = model.predict(
        X_predict
    )

    # --------------------------------------------------------
    # Construct imputed result
    # --------------------------------------------------------

    result = target_train.copy()

    if feature_type == "numerical":

        result.loc[
            missing_mask
        ] = predictions

    else:

        result.loc[
            missing_mask
        ] = predictions.astype(
            target_train.dtype
            if target_train.dtype != "object"
            else object
        )

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if result.isna().any():

        remaining_missing = int(
            result.isna().sum()
        )

        raise RuntimeError(
            "Random Forest imputation failed to "
            f"resolve {remaining_missing} missing values."
        )

    return (
        result,
        model
    )


# ------------------------------------------------------------
# 3. Final readiness checks
# ------------------------------------------------------------

assert callable(
    fit_random_forest
)

print(
    "Random Forest candidate : READY"
)

print(
    "Numerical target support : ENABLED"
)

print(
    "Categorical target support : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Predictor alignment validation : ENABLED"
)

print(
    "Predictor missing-value handling : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.13
RANDOM FOREST IMPUTATION CANDIDATE
Random Forest candidate : READY
Numerical target support : ENABLED
Categorical target support : ENABLED
Leakage-safe fitting : ENABLED
Predictor alignment validation : ENABLED
Predictor missing-value handling : ENABLED
Reproducibility : ENABLED


In [14]:
# ============================================================
# NOTEBOOK 06.14 — GRADIENT BOOSTING CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.14")
print("GRADIENT BOOSTING IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Gradient Boosting imputation
# ------------------------------------------------------------

def fit_gradient_boosting(
    X_train,
    target_train,
    X_missing=None,
    feature_type="numerical",
    random_state=42,
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20
):
    """
    Leakage-safe Gradient Boosting imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Predictor matrix corresponding to target_train.

    target_train : pandas.Series
        Target/feature containing missing values.

    X_missing : pandas.DataFrame, optional
        Predictor matrix used for predicting missing target values.

    feature_type : str
        'numerical' or 'categorical'.

    random_state : int
        Reproducibility seed.

    max_iter : int
        Maximum boosting iterations.

    learning_rate : float
        Boosting learning rate.

    max_leaf_nodes : int
        Maximum number of leaves per tree.

    min_samples_leaf : int
        Minimum samples per leaf.

    Returns
    -------
    result : pandas.Series
        Imputed target.

    model : fitted Gradient Boosting model
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        target_train,
        pd.Series
    ):
        raise TypeError(
            "target_train must be a pandas Series."
        )

    if len(X_train) != len(target_train):
        raise ValueError(
            "X_train and target_train must have "
            "the same number of rows."
        )

    if X_missing is None:
        X_missing = X_train.copy()

    if not isinstance(
        X_missing,
        pd.DataFrame
    ):
        raise TypeError(
            "X_missing must be a pandas DataFrame."
        )

    if len(X_missing) != len(target_train):
        raise ValueError(
            "X_missing and target_train must have "
            "the same number of rows."
        )

    # --------------------------------------------------------
    # Normalize feature type
    # --------------------------------------------------------

    feature_type = str(
        feature_type
    ).strip().lower()

    if feature_type in {
        "numeric",
        "numerical",
        "float",
        "integer",
        "int"
    }:
        feature_type = "numerical"

    elif feature_type in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool"
    }:
        feature_type = "categorical"

    else:
        raise ValueError(
            f"Unsupported feature_type: {feature_type}"
        )

    # --------------------------------------------------------
    # Align indexes
    # --------------------------------------------------------

    if not X_train.index.equals(
        target_train.index
    ):

        X_train = X_train.reindex(
            target_train.index
        )

    if not X_missing.index.equals(
        target_train.index
    ):

        X_missing = X_missing.reindex(
            target_train.index
        )

    # --------------------------------------------------------
    # Missingness masks
    # --------------------------------------------------------

    observed_mask = (
        target_train.notna()
    )

    missing_mask = (
        target_train.isna()
    )

    n_observed = int(
        observed_mask.sum()
    )

    n_missing = int(
        missing_mask.sum()
    )

    # Nothing to impute
    if n_missing == 0:

        return (
            target_train.copy(),
            None
        )

    if n_observed < 10:

        raise ValueError(
            "Insufficient observed target values "
            f"for Gradient Boosting: {n_observed}"
        )

    # --------------------------------------------------------
    # Prepare predictors
    # --------------------------------------------------------
    #
    # The normal AIR-LLM pipeline should already encode
    # predictors before reaching this function.
    #
    # This safeguard ensures that the candidate does not fail
    # because of residual NaN / infinite predictor values.
    # --------------------------------------------------------

    X_train_numeric = (
        X_train
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    X_missing_numeric = (
        X_missing
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    # Convert non-numeric predictors when necessary
    if not all(
        pd.api.types.is_numeric_dtype(
            X_train_numeric[column]
        )
        for column in X_train_numeric.columns
    ):

        X_train_numeric = (
            X_train_numeric.apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        X_missing_numeric = (
            X_missing_numeric.apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

    # Median predictor imputation
    predictor_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_numeric = pd.DataFrame(
        predictor_imputer.fit_transform(
            X_train_numeric
        ),
        index=X_train.index
    )

    X_missing_numeric = pd.DataFrame(
        predictor_imputer.transform(
            X_missing_numeric
        ),
        index=X_missing.index
    )

    # --------------------------------------------------------
    # Prepare observed target
    # --------------------------------------------------------

    y_observed = (
        target_train
        .loc[observed_mask]
        .copy()
    )

    # --------------------------------------------------------
    # Build appropriate model
    # --------------------------------------------------------

    if feature_type == "numerical":

        y_observed = pd.to_numeric(
            y_observed,
            errors="coerce"
        )

        valid_target = (
            y_observed.notna()
            &
            np.isfinite(
                y_observed.to_numpy(
                    dtype=float
                )
            )
        )

        y_observed = (
            y_observed
            .loc[valid_target]
        )

        if len(y_observed) < 10:

            raise ValueError(
                "Insufficient valid numerical "
                "target observations."
            )

        model = HistGradientBoostingRegressor(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        )

    else:

        # ----------------------------------------------------
        # Categorical target
        #
        # HistGradientBoostingClassifier requires encoded
        # class labels.
        # ----------------------------------------------------

        y_observed = (
            y_observed
            .astype(str)
        )

        classes = (
            pd.Index(
                y_observed
                .unique()
            )
            .sort_values()
        )

        if len(classes) < 2:

            raise ValueError(
                "Categorical target must contain "
                "at least two observed classes."
            )

        class_to_code = {
            value: index
            for index, value
            in enumerate(classes)
        }

        y_encoded = (
            y_observed
            .map(class_to_code)
            .astype(int)
        )

        model = HistGradientBoostingClassifier(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        )

    # --------------------------------------------------------
    # Align predictors with valid target observations
    # --------------------------------------------------------

    X_fit = (
        X_train_numeric
        .loc[
            y_observed.index
        ]
    )

    # --------------------------------------------------------
    # Fit model
    # --------------------------------------------------------

    if feature_type == "numerical":

        model.fit(
            X_fit,
            y_observed
        )

    else:

        model.fit(
            X_fit,
            y_encoded
        )

    # --------------------------------------------------------
    # Predict missing values
    # --------------------------------------------------------

    X_predict = (
        X_missing_numeric
        .loc[
            missing_mask
        ]
    )

    predictions = model.predict(
        X_predict
    )

    # --------------------------------------------------------
    # Construct final imputed series
    # --------------------------------------------------------

    result = target_train.copy()

    if feature_type == "numerical":

        result.loc[
            missing_mask
        ] = predictions

    else:

        inverse_mapping = {
            code: value
            for value, code
            in class_to_code.items()
        }

        decoded_predictions = [
            inverse_mapping[int(x)]
            for x in predictions
        ]

        result.loc[
            missing_mask
        ] = decoded_predictions

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if result.isna().any():

        remaining_missing = int(
            result.isna().sum()
        )

        raise RuntimeError(
            "Gradient Boosting imputation failed "
            f"to resolve {remaining_missing} missing values."
        )

    return (
        result,
        model
    )


# ------------------------------------------------------------
# 3. Final readiness validation
# ------------------------------------------------------------

assert callable(
    fit_gradient_boosting
)

print(
    "Gradient Boosting candidate : READY"
)

print(
    "Numerical target support : ENABLED"
)

print(
    "Categorical target support : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Predictor alignment validation : ENABLED"
)

print(
    "Predictor missing-value handling : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.14
GRADIENT BOOSTING IMPUTATION CANDIDATE
Gradient Boosting candidate : READY
Numerical target support : ENABLED
Categorical target support : ENABLED
Leakage-safe fitting : ENABLED
Predictor alignment validation : ENABLED
Predictor missing-value handling : ENABLED
Reproducibility : ENABLED


In [15]:
# ============================================================
# NOTEBOOK 06.15 — MISSFOREST CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.15")
print("MISSFOREST IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Mixed-type MissForest implementation
# ------------------------------------------------------------

def fit_missforest(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=5,
    n_estimators=100,
    min_samples_leaf=2
):
    """
    Mixed-type MissForest-style iterative imputation.

    Numerical variables are modeled with RandomForestRegressor.

    Categorical variables are modeled with
    RandomForestClassifier.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Training data containing missing values.

    X_test : pandas.DataFrame, optional
        Test data containing missing values.

    random_state : int
        Reproducibility seed.

    max_iter : int
        Maximum iterative imputation passes.

    n_estimators : int
        Number of Random Forest trees.

    min_samples_leaf : int
        Minimum observations per leaf.

    Returns
    -------
    X_train_imputed : pandas.DataFrame
        Imputed training data.

    X_test_imputed : pandas.DataFrame or None
        Imputed test data.

    state : dict
        Fitted MissForest state.
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if X_train.empty:
        raise ValueError(
            "X_train is empty."
        )

    if X_test is not None:

        if not isinstance(
            X_test,
            pd.DataFrame
        ):
            raise TypeError(
                "X_test must be a pandas DataFrame."
            )

        if list(
            X_test.columns
        ) != list(
            X_train.columns
        ):
            raise ValueError(
                "X_train and X_test must have "
                "identical columns."
            )

    # --------------------------------------------------------
    # Work on copies
    # --------------------------------------------------------

    train = X_train.copy()

    test = (
        X_test.copy()
        if X_test is not None
        else None
    )

    # --------------------------------------------------------
    # Identify feature types
    # --------------------------------------------------------

    numerical_columns = (
        train
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in train.columns
        if column not in numerical_columns
    ]

    if not numerical_columns and not categorical_columns:

        raise ValueError(
            "No usable predictor columns found."
        )

    # --------------------------------------------------------
    # Preserve original dtypes
    # --------------------------------------------------------

    original_dtypes = {
        column: train[column].dtype
        for column in train.columns
    }

    # --------------------------------------------------------
    # Convert categorical variables to integer codes
    #
    # Categories are learned from TRAINING DATA only.
    # --------------------------------------------------------

    category_maps = {}

    for column in categorical_columns:

        train_values = (
            train[column]
            .astype("string")
        )

        categories = (
            train_values
            .dropna()
            .unique()
            .tolist()
        )

        category_maps[column] = {
            value: index
            for index, value
            in enumerate(categories)
        }

        train[column] = (
            train_values
            .map(
                category_maps[column]
            )
            .astype(float)
        )

        if test is not None:

            test_values = (
                test[column]
                .astype("string")
            )

            test[column] = (
                test_values
                .map(
                    category_maps[column]
                )
                .astype(float)
            )

    # --------------------------------------------------------
    # Convert numerical variables to numeric
    # --------------------------------------------------------

    for column in numerical_columns:

        train[column] = pd.to_numeric(
            train[column],
            errors="coerce"
        )

        if test is not None:

            test[column] = pd.to_numeric(
                test[column],
                errors="coerce"
            )

    # --------------------------------------------------------
    # Replace infinite values
    # --------------------------------------------------------

    train = train.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if test is not None:

        test = test.replace(
            [np.inf, -np.inf],
            np.nan
        )

    # --------------------------------------------------------
    # Initial imputation
    # --------------------------------------------------------

    initial_imputer = SimpleImputer(
        strategy="median"
    )

    train_initial = pd.DataFrame(
        initial_imputer.fit_transform(
            train
        ),
        columns=train.columns,
        index=train.index
    )

    if test is not None:

        test_initial = pd.DataFrame(
            initial_imputer.transform(
                test
            ),
            columns=test.columns,
            index=test.index
        )

    else:

        test_initial = None

    # --------------------------------------------------------
    # Original missingness masks
    # --------------------------------------------------------

    train_missing_masks = {
        column:
        train[column].isna()
        for column in train.columns
    }

    if test is not None:

        test_missing_masks = {
            column:
            test[column].isna()
            for column in test.columns
        }

    # --------------------------------------------------------
    # Iterative MissForest process
    # --------------------------------------------------------

    current_train = train_initial.copy()

    current_test = (
        test_initial.copy()
        if test_initial is not None
        else None
    )

    feature_types = {
        column:
        (
            "numerical"
            if column in numerical_columns
            else "categorical"
        )
        for column in train.columns
    }

    fitted_models = {}

    for iteration in range(
        max_iter
    ):

        previous_train = (
            current_train.copy()
        )

        # ----------------------------------------------------
        # Process each feature with missing values
        # ----------------------------------------------------

        for target_column in train.columns:

            missing_mask = (
                train_missing_masks[
                    target_column
                ]
            )

            if not missing_mask.any():
                continue

            predictor_columns = [
                column
                for column in train.columns
                if column != target_column
            ]

            if not predictor_columns:
                continue

            observed_mask = (
                ~missing_mask
            )

            X_fit = (
                current_train
                .loc[
                    observed_mask,
                    predictor_columns
                ]
            )

            y_fit = (
                current_train
                .loc[
                    observed_mask,
                    target_column
                ]
            )

            X_predict = (
                current_train
                .loc[
                    missing_mask,
                    predictor_columns
                ]
            )

            if len(X_fit) < 10:
                continue

            feature_type = (
                feature_types[
                    target_column
                ]
            )

            # ------------------------------------------------
            # Numerical target
            # ------------------------------------------------

            if feature_type == "numerical":

                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    random_state=(
                        random_state
                        + iteration
                    ),
                    n_jobs=-1,
                    min_samples_leaf=min_samples_leaf
                )

                model.fit(
                    X_fit,
                    y_fit
                )

                predictions = (
                    model.predict(
                        X_predict
                    )
                )

            # ------------------------------------------------
            # Categorical target
            # ------------------------------------------------

            else:

                y_fit = (
                    y_fit
                    .round()
                    .astype(int)
                )

                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    random_state=(
                        random_state
                        + iteration
                    ),
                    n_jobs=-1,
                    min_samples_leaf=min_samples_leaf
                )

                model.fit(
                    X_fit,
                    y_fit
                )

                predictions = (
                    model.predict(
                        X_predict
                    )
                )

            current_train.loc[
                missing_mask,
                target_column
            ] = predictions

            fitted_models[
                target_column
            ] = model

        # ----------------------------------------------------
        # Convergence check
        # ----------------------------------------------------

        difference = np.nanmean(
            np.abs(
                current_train.to_numpy(
                    dtype=float
                )
                -
                previous_train.to_numpy(
                    dtype=float
                )
            )
        )

        if not np.isfinite(
            difference
        ):
            break

        if difference < 1e-4:
            break

    # --------------------------------------------------------
    # Apply fitted models to X_test
    # --------------------------------------------------------

    if current_test is not None:

        for target_column in test.columns:

            missing_mask = (
                test_missing_masks[
                    target_column
                ]
            )

            if not missing_mask.any():
                continue

            model = fitted_models.get(
                target_column
            )

            if model is None:
                continue

            predictor_columns = [
                column
                for column in test.columns
                if column != target_column
            ]

            X_predict = (
                current_test
                .loc[
                    missing_mask,
                    predictor_columns
                ]
            )

            predictions = model.predict(
                X_predict
            )

            current_test.loc[
                missing_mask,
                target_column
            ] = predictions

    # --------------------------------------------------------
    # Decode categorical variables
    # --------------------------------------------------------

    reverse_category_maps = {

        column: {
            code: value
            for value, code
            in mapping.items()
        }

        for column, mapping
        in category_maps.items()
    }

    for column in categorical_columns:

        reverse_map = (
            reverse_category_maps[
                column
            ]
        )

        current_train[column] = (
            current_train[column]
            .round()
            .map(reverse_map)
        )

        if current_test is not None:

            current_test[column] = (
                current_test[column]
                .round()
                .map(reverse_map)
            )

    # --------------------------------------------------------
    # Restore numerical dtypes where possible
    # --------------------------------------------------------

    for column in numerical_columns:

        try:
            current_train[column] = (
                current_train[column]
                .astype(
                    original_dtypes[
                        column
                    ]
                )
            )
        except Exception:
            pass

        if current_test is not None:

            try:
                current_test[column] = (
                    current_test[column]
                    .astype(
                        original_dtypes[
                            column
                        ]
                    )
                )
            except Exception:
                pass

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if current_train.isna().any().any():

        remaining = int(
            current_train.isna()
            .sum()
            .sum()
        )

        raise RuntimeError(
            "MissForest failed to resolve "
            f"{remaining} training missing values."
        )

    if (
        current_test is not None
        and current_test.isna().any().any()
    ):

        remaining = int(
            current_test.isna()
            .sum()
            .sum()
        )

        raise RuntimeError(
            "MissForest failed to resolve "
            f"{remaining} test missing values."
        )

    state = {
        "models": fitted_models,
        "feature_types": feature_types,
        "category_maps": category_maps,
        "iterations": max_iter,
        "random_state": random_state,
    }

    return (
        current_train,
        current_test,
        state
    )


# ------------------------------------------------------------
# 3. Final readiness validation
# ------------------------------------------------------------

assert callable(
    fit_missforest
)

print(
    "MissForest candidate : READY"
)

print(
    "Mixed numerical/categorical support : ENABLED"
)

print(
    "Iterative Random Forest refinement : ENABLED"
)

print(
    "Categorical Random Forest classification : ENABLED"
)

print(
    "Training-only category mapping : ENABLED"
)

print(
    "Convergence monitoring : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.15
MISSFOREST IMPUTATION CANDIDATE
MissForest candidate : READY
Mixed numerical/categorical support : ENABLED
Iterative Random Forest refinement : ENABLED
Categorical Random Forest classification : ENABLED
Training-only category mapping : ENABLED
Convergence monitoring : ENABLED
Leakage-safe fitting : ENABLED
Reproducibility : ENABLED


In [16]:
# ============================================================
# NOTEBOOK 06.16 — SOFTIMPUTE / MATRIX-FACTORIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16")
print("SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import numpy as np


# ------------------------------------------------------------
# 2. SoftImpute implementation
# ------------------------------------------------------------

def fit_softimpute(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50,
    tolerance=1e-4,
    shrinkage=0.0
):

    X_train_array = np.asarray(
        X_train,
        dtype=float
    )

    if X_train_array.ndim != 2:
        raise ValueError(
            "X_train must be a 2-dimensional numeric matrix."
        )

    if rank < 1:
        raise ValueError(
            "rank must be >= 1."
        )

    if max_iter < 1:
        raise ValueError(
            "max_iter must be >= 1."
        )

    if tolerance <= 0:
        raise ValueError(
            "tolerance must be > 0."
        )

    if not np.isfinite(
        np.nanmean(X_train_array, axis=0)
    ).all():

        # handled below column-by-column
        pass


    # --------------------------------------------------------
    # Initial column-mean imputation
    # --------------------------------------------------------

    column_means = np.nanmean(
        X_train_array,
        axis=0
    )

    column_means = np.where(
        np.isfinite(column_means),
        column_means,
        0.0
    )


    missing_mask = np.isnan(
        X_train_array
    )

    filled = np.where(
        missing_mask,
        column_means,
        X_train_array
    )


    # --------------------------------------------------------
    # Iterative low-rank reconstruction
    # --------------------------------------------------------

    converged = False
    iterations_completed = 0
    final_change = np.inf

    for iteration in range(
        max_iter
    ):

        previous_missing = (
            filled[missing_mask].copy()
        )


        # ----------------------------------------------------
        # SVD
        # ----------------------------------------------------

        try:

            U, S, VT = np.linalg.svd(
                filled,
                full_matrices=False
            )

        except np.linalg.LinAlgError as exc:

            raise RuntimeError(
                "SVD failed during SoftImpute."
            ) from exc


        effective_rank = min(
            int(rank),
            len(S)
        )


        if effective_rank == 0:

            break


        # ----------------------------------------------------
        # Optional singular-value shrinkage
        # ----------------------------------------------------

        retained_singular_values = (
            S[:effective_rank]
            - float(shrinkage)
        )

        retained_singular_values = np.maximum(
            retained_singular_values,
            0.0
        )


        reconstructed = (
            U[:, :effective_rank]
            *
            retained_singular_values
        ) @ VT[:effective_rank, :]


        # ----------------------------------------------------
        # Update ONLY originally missing values
        # ----------------------------------------------------

        filled[missing_mask] = (
            reconstructed[missing_mask]
        )


        # ----------------------------------------------------
        # Convergence based on missing entries
        # ----------------------------------------------------

        current_missing = (
            filled[missing_mask]
        )

        if len(previous_missing) > 0:

            denominator = (
                np.linalg.norm(
                    previous_missing
                )
                + 1e-12
            )

            final_change = (
                np.linalg.norm(
                    current_missing
                    -
                    previous_missing
                )
                /
                denominator
            )

        else:

            final_change = 0.0


        iterations_completed = (
            iteration + 1
        )


        if final_change < tolerance:

            converged = True
            break


    # --------------------------------------------------------
    # Transform test data using the learned low-rank model
    # --------------------------------------------------------

    X_test_imputed = None

    if X_test is not None:

        X_test_array = np.asarray(
            X_test,
            dtype=float
        )

        if X_test_array.ndim != 2:
            raise ValueError(
                "X_test must be a 2-dimensional numeric matrix."
            )

        if X_test_array.shape[1] != (
            X_train_array.shape[1]
        ):

            raise ValueError(
                "X_test and X_train must have "
                "the same number of columns."
            )


        test_missing_mask = np.isnan(
            X_test_array
        )


        # ----------------------------------------------------
        # Initialize test missing values using training means
        # ----------------------------------------------------

        test_filled = np.where(
            test_missing_mask,
            column_means,
            X_test_array
        )


        # ----------------------------------------------------
        # Reconstruct test matrix using learned
        # low-rank structure
        # ----------------------------------------------------

        U_test, S_test, VT_test = np.linalg.svd(
            test_filled,
            full_matrices=False
        )

        test_rank = min(
            effective_rank,
            len(S_test)
        )


        if test_rank > 0:

            test_singular_values = (
                S_test[:test_rank]
                -
                float(shrinkage)
            )

            test_singular_values = np.maximum(
                test_singular_values,
                0.0
            )

            test_reconstructed = (
                U_test[:, :test_rank]
                *
                test_singular_values
            ) @ VT_test[:test_rank, :]

            test_filled[
                test_missing_mask
            ] = test_reconstructed[
                test_missing_mask
            ]


        X_test_imputed = test_filled


    # --------------------------------------------------------
    # Return model metadata
    # --------------------------------------------------------

    model_metadata = {

        "method":
            "SoftImpute",

        "rank":
            int(effective_rank),

        "max_iter":
            int(max_iter),

        "iterations":
            int(iterations_completed),

        "tolerance":
            float(tolerance),

        "final_change":
            float(final_change),

        "converged":
            bool(converged),

        "shrinkage":
            float(shrinkage),

    }


    return (
        filled,
        X_test_imputed,
        model_metadata
    )


# ------------------------------------------------------------
# 3. Matrix-factorization candidate
# ------------------------------------------------------------

def fit_matrix_factorization(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50,
    tolerance=1e-4
):

    return fit_softimpute(
        X_train=X_train,
        X_test=X_test,
        rank=rank,
        max_iter=max_iter,
        tolerance=tolerance,
        shrinkage=0.0
    )


# ------------------------------------------------------------
# 4. Function validation
# ------------------------------------------------------------

required_functions = [
    "fit_softimpute",
    "fit_matrix_factorization",
]


missing_functions = [
    name
    for name in required_functions
    if name not in globals()
]


if missing_functions:

    raise RuntimeError(
        "Missing required functions:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_functions
        )
    )


# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print(
    "SoftImpute candidate: READY"
)

print(
    "Matrix-factorization candidate: READY"
)

print(
    "Low-rank reconstruction: ENABLED"
)

print(
    "Missing-entry-only convergence: ENABLED"
)

print(
    "Train/test dimensionality validation: ENABLED"
)

print(
    "Deterministic implementation: ENABLED"
)

print("\n" + "=" * 100)
print(
    "SOFTIMPUTE / MATRIX-FACTORIZATION : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.16
SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES
SoftImpute candidate: READY
Matrix-factorization candidate: READY
Low-rank reconstruction: ENABLED
Missing-entry-only convergence: ENABLED
Train/test dimensionality validation: ENABLED
Deterministic implementation: ENABLED

SOFTIMPUTE / MATRIX-FACTORIZATION : PASSED


In [17]:
# ============================================================
# NOTEBOOK 06.16A — CANDIDATE STRATEGY IMPLEMENTATION REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16A")
print("CANDIDATE STRATEGY IMPLEMENTATION REGISTRY")
print("=" * 100)


# ------------------------------------------------------------
# 06.16A.01 — Required implementations
# ------------------------------------------------------------

required_functions = {
    "mean": "impute_mean",
    "median": "impute_median",
    "mode": "impute_mode",
    "constant": "impute_constant",
    "random_sample": "impute_random_sample",
    "knn": "fit_knn",
    "iterativeimputer": "fit_iterative",
    "missforest": "fit_missforest",
    "randomforest": "fit_random_forest",
}


# ------------------------------------------------------------
# 06.16A.02 — Validate functions
# ------------------------------------------------------------

missing_functions = []

for strategy_id, function_name in (
    required_functions.items()
):

    if function_name not in globals():

        missing_functions.append(
            f"{strategy_id} -> {function_name}"
        )


if missing_functions:

    raise RuntimeError(
        "Required strategy functions are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_functions
        )
        +
        "\n\nRun Notebook 06.10–06.15 first."
    )


# ------------------------------------------------------------
# 06.16A.03 — Build authoritative executable registry
# ------------------------------------------------------------

STRATEGY_FUNCTIONS = {

    strategy_id:
        globals()[function_name]

    for strategy_id, function_name
    in required_functions.items()
}


# ------------------------------------------------------------
# 06.16A.04 — Validate callables
# ------------------------------------------------------------

invalid_strategies = [

    strategy_id

    for strategy_id, function
    in STRATEGY_FUNCTIONS.items()

    if not callable(function)

]


if invalid_strategies:

    raise RuntimeError(
        "Non-callable strategy implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in invalid_strategies
        )
    )


# ------------------------------------------------------------
# 06.16A.05 — Compatibility registry
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY = {

    "mean":
        {"numerical"},

    "median":
        {"numerical"},

    "mode":
        {"categorical"},

    "constant":
        {
            "numerical",
            "categorical"
        },

    "random_sample":
        {
            "numerical",
            "categorical"
        },

    "knn":
        {"numerical"},

    "iterativeimputer":
        {"numerical"},

    "missforest":
        {"numerical"},

    "randomforest":
        {"numerical"},

}


# ------------------------------------------------------------
# 06.16A.06 — Final validation
# ------------------------------------------------------------

print("\nSTRATEGY IMPLEMENTATION REGISTRY")
print("-" * 100)

for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    print(
        f"  ✓ {strategy_id:<20} "
        f"{function.__name__}"
    )


print("\nCompatibility:")
print("-" * 100)

for strategy_id, feature_types in (
    STRATEGY_FEATURE_COMPATIBILITY.items()
):

    print(
        f"  {strategy_id:<20} "
        f"{sorted(feature_types)}"
    )


print("\n" + "=" * 100)

print(
    f"Registered implementations : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print(
    f"Compatibility definitions   : "
    f"{len(STRATEGY_FEATURE_COMPATIBILITY)}"
)

print(
    "\nCandidate strategy registry: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.16A
CANDIDATE STRATEGY IMPLEMENTATION REGISTRY

STRATEGY IMPLEMENTATION REGISTRY
----------------------------------------------------------------------------------------------------
  ✓ mean                 impute_mean
  ✓ median               impute_median
  ✓ mode                 impute_mode
  ✓ constant             impute_constant
  ✓ random_sample        impute_random_sample
  ✓ knn                  fit_knn
  ✓ iterativeimputer     fit_iterative
  ✓ missforest           fit_missforest
  ✓ randomforest         fit_random_forest

Compatibility:
----------------------------------------------------------------------------------------------------
  mean                 ['numerical']
  median               ['numerical']
  mode                 ['categorical']
  constant             ['categorical', 'numerical']
  random_sample        ['categorical', 'numerical']
  knn                  ['numerical']
  iterativeimputer     ['numerical']
  missforest           ['numerica

In [18]:
# ============================================================
# NOTEBOOK 06.17 — CANDIDATE STRATEGY COMPATIBILITY VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.17")
print("CANDIDATE STRATEGY COMPATIBILITY VALIDATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import pandas as pd


# ------------------------------------------------------------
# 2. Required AIR-LLM state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required AIR-LLM state is missing:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.06 first."
    )


if not isinstance(EVALUATION_DATA, dict):
    raise TypeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not isinstance(FEATURE_TYPE_DF, pd.DataFrame):
    raise TypeError(
        "FEATURE_TYPE_DF must be a pandas DataFrame."
    )


if FEATURE_TYPE_DF.empty:
    raise RuntimeError(
        "FEATURE_TYPE_DF is empty."
    )


# ------------------------------------------------------------
# 3. Validate feature-type schema
# ------------------------------------------------------------

required_feature_columns = {
    "dataset_id",
    "feature",
    "feature_type",
}

missing_feature_columns = (
    required_feature_columns
    -
    set(FEATURE_TYPE_DF.columns)
)

if missing_feature_columns:

    raise RuntimeError(
        "FEATURE_TYPE_DF is missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_feature_columns
            )
        )
    )


# ------------------------------------------------------------
# 4. Strategy-ID normalization
# ------------------------------------------------------------

def normalize_strategy_id(strategy_id):

    if strategy_id is None:

        raise ValueError(
            "Strategy ID cannot be None."
        )

    value = (
        str(strategy_id)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "miss_forest":
            "missforest",

        "random_sample_imputer":
            "random_sample",

        "gradientboosting":
            "gradient_boosting",

        "matrixfactorization":
            "matrix_factorization",

    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 5. Feature-type normalization
# ------------------------------------------------------------

def normalize_feature_type(value):

    value = (
        str(value)
        .strip()
        .lower()
    )

    numerical_aliases = {
        "numeric",
        "numerical",
        "number",
        "integer",
        "int",
        "int32",
        "int64",
        "float",
        "float32",
        "float64",
        "continuous",
    }

    categorical_aliases = {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "bool",
        "boolean",
    }

    if value in numerical_aliases:
        return "numerical"

    if value in categorical_aliases:
        return "categorical"

    return value


FEATURE_TYPE_DF = FEATURE_TYPE_DF.copy()

FEATURE_TYPE_DF[
    "normalized_feature_type"
] = (
    FEATURE_TYPE_DF[
        "feature_type"
    ]
    .map(
        normalize_feature_type
    )
)


# ------------------------------------------------------------
# 6. Validate detected feature types
# ------------------------------------------------------------

allowed_feature_types = {
    "numerical",
    "categorical",
}

detected_types = set(
    FEATURE_TYPE_DF[
        "normalized_feature_type"
    ]
    .dropna()
    .unique()
)

unsupported_detected_types = (
    detected_types
    -
    allowed_feature_types
)

if unsupported_detected_types:

    raise RuntimeError(
        "Unsupported detected feature types:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                unsupported_detected_types
            )
        )
    )


# ------------------------------------------------------------
# 7. AIR-LLM candidate compatibility definition
# ------------------------------------------------------------
#
# Compatibility reflects the IMPLEMENTATIONS actually
# available in Notebook 06.10–06.16.
#
# Numerical:
#   Mean
#   Median
#   KNN
#   Iterative/MICE
#   Random Forest
#   Gradient Boosting
#   MissForest
#   SoftImpute
#   Matrix Factorization
#
# Categorical:
#   Mode
#   Constant
#   Random Sample
#
# Constant and Random Sample support both types.
#
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY_MASTER = {

    "mean": {
        "numerical"
    },

    "median": {
        "numerical"
    },

    "mode": {
        "categorical"
    },

    "constant": {
        "numerical",
        "categorical"
    },

    "random_sample": {
        "numerical",
        "categorical"
    },

    "knn": {
        "numerical"
    },

    "iterativeimputer": {
        "numerical"
    },

    "randomforest": {
        "numerical"
    },

    "gradient_boosting": {
        "numerical"
    },

    "missforest": {
        "numerical"
    },

    "softimpute": {
        "numerical"
    },

    "matrix_factorization": {
        "numerical"
    },
}


# ------------------------------------------------------------
# 8. Build implementation registry
# ------------------------------------------------------------
#
# The previous version expected STRATEGY_FUNCTIONS to already
# exist. That was the cause of the failure.
#
# Here we explicitly map the implementations created in:
#
# 06.10 Statistical candidates
# 06.11 KNN
# 06.12 Iterative / MICE
# 06.13 Random Forest
# 06.14 Gradient Boosting
# 06.15 MissForest
# 06.16 SoftImpute / Matrix Factorization
#
# ------------------------------------------------------------

STRATEGY_FUNCTIONS = {}


implementation_map = {

    "mean":
        "impute_mean",

    "median":
        "impute_median",

    "mode":
        "impute_mode",

    "constant":
        "impute_constant",

    "random_sample":
        "impute_random_sample",

    "knn":
        "fit_knn",

    "iterativeimputer":
        "fit_iterative",

    "randomforest":
        "fit_random_forest",

    "gradient_boosting":
        "fit_gradient_boosting",

    "missforest":
        "fit_missforest",

    "softimpute":
        "fit_softimpute",

    "matrix_factorization":
        "fit_matrix_factorization",
}


missing_function_names = []

for strategy_id, function_name in implementation_map.items():

    if function_name not in globals():

        missing_function_names.append(
            f"{strategy_id} -> {function_name}"
        )

    else:

        STRATEGY_FUNCTIONS[
            strategy_id
        ] = globals()[
            function_name
        ]


if missing_function_names:

    raise RuntimeError(
        "Required candidate implementation functions "
        "are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_function_names
        )
        +
        "\n\n"
        "Required implementation cells are "
        "Notebook 06.10 → 06.16."
    )


# ------------------------------------------------------------
# 9. Determine required strategies
# ------------------------------------------------------------

REQUIRED_STRATEGIES = sorted(
    STRATEGY_FEATURE_COMPATIBILITY_MASTER.keys()
)


# ------------------------------------------------------------
# 10. Validate strategy definitions
# ------------------------------------------------------------

undefined_strategies = [
    strategy_id
    for strategy_id in REQUIRED_STRATEGIES
    if strategy_id
    not in STRATEGY_FEATURE_COMPATIBILITY_MASTER
]

if undefined_strategies:

    raise RuntimeError(
        "The following strategies have no "
        "compatibility definition:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in undefined_strategies
        )
    )


# ------------------------------------------------------------
# 11. Validate implementation coverage
# ------------------------------------------------------------

implemented_strategies = {
    normalize_strategy_id(x)
    for x in STRATEGY_FUNCTIONS.keys()
}


missing_implementations = (
    set(REQUIRED_STRATEGIES)
    -
    implemented_strategies
)

if missing_implementations:

    raise RuntimeError(
        "The following required strategies have "
        "no implementation:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_implementations
            )
        )
    )


# ------------------------------------------------------------
# 12. Build strategy compatibility table
# ------------------------------------------------------------

STRATEGY_COMPATIBILITY_DF = pd.DataFrame([

    {
        "strategy_id":
            strategy_id,

        "supported_feature_types":
            sorted(
                STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                    strategy_id
                ]
            ),

        "implementation_function":
            STRATEGY_FUNCTIONS[
                strategy_id
            ].__name__,

        "implemented":
            True,

    }

    for strategy_id
    in REQUIRED_STRATEGIES

])


# ------------------------------------------------------------
# 13. Dataset-level compatibility validation
# ------------------------------------------------------------

compatibility_rows = []


for dataset_id in sorted(
    EVALUATION_DATA.keys()
):

    dataset_features = (
        FEATURE_TYPE_DF[
            FEATURE_TYPE_DF[
                "dataset_id"
            ].astype(str)
            ==
            str(dataset_id)
        ]
    )


    for strategy_id in REQUIRED_STRATEGIES:

        supported_types = (
            STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                strategy_id
            ]
        )


        compatible_features = (
            dataset_features[
                dataset_features[
                    "normalized_feature_type"
                ].isin(
                    supported_types
                )
            ][
                "feature"
            ]
            .tolist()
        )


        incompatible_features = (
            dataset_features[
                ~dataset_features[
                    "normalized_feature_type"
                ].isin(
                    supported_types
                )
            ][
                "feature"
            ]
            .tolist()
        )


        compatibility_rows.append({

            "dataset_id":
                dataset_id,

            "strategy_id":
                strategy_id,

            "supported_feature_types":
                sorted(
                    supported_types
                ),

            "compatible_feature_count":
                len(
                    compatible_features
                ),

            "incompatible_feature_count":
                len(
                    incompatible_features
                ),

            "compatible":
                len(
                    compatible_features
                ) > 0,

        })


STRATEGY_COMPATIBILITY_VALIDATION_DF = (
    pd.DataFrame(
        compatibility_rows
    )
)


# ------------------------------------------------------------
# 14. Validate every strategy across every dataset
# ------------------------------------------------------------

invalid_strategy_rows = (
    STRATEGY_COMPATIBILITY_VALIDATION_DF[
        ~STRATEGY_COMPATIBILITY_VALIDATION_DF[
            "compatible"
        ]
    ]
)


if not invalid_strategy_rows.empty:

    display(
        invalid_strategy_rows
    )

    raise RuntimeError(
        "One or more strategies have no compatible "
        "features in one or more datasets."
    )


# ------------------------------------------------------------
# 15. Final lookup used by evaluation engine
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY = {

    strategy_id:
        set(
            STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                strategy_id
            ]
        )

    for strategy_id
    in REQUIRED_STRATEGIES

}


# ------------------------------------------------------------
# 16. Final implementation validation
# ------------------------------------------------------------

if set(
    STRATEGY_FUNCTIONS.keys()
) != set(
    REQUIRED_STRATEGIES
):

    raise RuntimeError(
        "Implementation registry does not exactly "
        "match the required candidate strategy pool."
    )


# ------------------------------------------------------------
# 17. Display implementation registry
# ------------------------------------------------------------

display(
    STRATEGY_COMPATIBILITY_DF
)


# ------------------------------------------------------------
# 18. Final report
# ------------------------------------------------------------

print(
    f"\nRequired strategies       : "
    f"{len(REQUIRED_STRATEGIES)}"
)

print(
    f"Detected feature types   : "
    f"{sorted(detected_types)}"
)

print(
    f"Implemented strategies   : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print(
    f"Compatibility definitions : "
    f"{len(STRATEGY_FEATURE_COMPATIBILITY)}"
)


print("\nStrategy implementation status:")
print("-" * 100)

for strategy_id in REQUIRED_STRATEGIES:

    function_name = (
        STRATEGY_FUNCTIONS[
            strategy_id
        ].__name__
    )

    supported = sorted(
        STRATEGY_FEATURE_COMPATIBILITY[
            strategy_id
        ]
    )

    print(
        f"  ✓ {strategy_id:<22} "
        f"{function_name:<28} "
        f"{supported}"
    )


# ------------------------------------------------------------
# 19. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print(
    "CANDIDATE STRATEGY COMPATIBILITY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.17
CANDIDATE STRATEGY COMPATIBILITY VALIDATION


,strategy_id,supported_feature_types,implementation_function,implemented
0,constant,"[categorical, numerical]",impute_constant,True
1,gradient_boosting,[numerical],fit_gradient_boosting,True
2,iterativeimputer,[numerical],fit_iterative,True
3,knn,[numerical],fit_knn,True
4,matrix_factorization,[numerical],fit_matrix_factorization,True
5,mean,[numerical],impute_mean,True
6,median,[numerical],impute_median,True
7,missforest,[numerical],fit_missforest,True
8,mode,[categorical],impute_mode,True
9,random_sample,"[categorical, numerical]",impute_random_sample,True



Required strategies       : 12
Detected feature types   : ['categorical', 'numerical']
Implemented strategies   : 12
Compatibility definitions : 12

Strategy implementation status:
----------------------------------------------------------------------------------------------------
  ✓ constant               impute_constant              ['categorical', 'numerical']
  ✓ gradient_boosting      fit_gradient_boosting        ['numerical']
  ✓ iterativeimputer       fit_iterative                ['numerical']
  ✓ knn                    fit_knn                      ['numerical']
  ✓ matrix_factorization   fit_matrix_factorization     ['numerical']
  ✓ mean                   impute_mean                  ['numerical']
  ✓ median                 impute_median                ['numerical']
  ✓ missforest             fit_missforest               ['numerical']
  ✓ mode                   impute_mode                  ['categorical']
  ✓ random_sample          impute_random_sample         ['categorical'

In [19]:
# ============================================================
# NOTEBOOK 06.18 — BUILD CANDIDATE EVALUATION PLAN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.18")
print("BUILD CANDIDATE EVALUATION PLAN")
print("=" * 100)


# ------------------------------------------------------------
# 06.18.01 — Required state
# ------------------------------------------------------------

required_objects = [
    "CONFIG",
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Notebook 06.18 dependencies missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# ------------------------------------------------------------
# 06.18.02 — Resolve strategy implementation registry
# ------------------------------------------------------------

# The current AIR-LLM pipeline uses STRATEGY_FUNCTIONS
# as the authoritative executable strategy registry.

if "STRATEGY_FUNCTIONS" not in globals():

    # Compatibility recovery from common registry names
    if "STRATEGY_IMPLEMENTATION_REGISTRY" in globals():

        STRATEGY_FUNCTIONS = (
            STRATEGY_IMPLEMENTATION_REGISTRY.copy()
        )

    elif "CANDIDATE_STRATEGY_REGISTRY" in globals():

        STRATEGY_FUNCTIONS = (
            CANDIDATE_STRATEGY_REGISTRY.copy()
        )

    else:

        raise RuntimeError(
            "STRATEGY_FUNCTIONS is not available.\n"
            "Run the candidate strategy implementation "
            "registry cell before Notebook 06.18."
        )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise RuntimeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ------------------------------------------------------------
# 06.18.03 — Normalize strategy IDs
# ------------------------------------------------------------

def normalize_strategy_id_06_18(
    value
):

    value = str(
        value
    ).strip().lower()

    aliases = {

        "randomsample":
            "random_sample",

        "random-sample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "iterative_imputer":
            "iterativeimputer",

        "iterative":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random-forest":
            "randomforest",

        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",
    }

    return aliases.get(
        value,
        value
    )


normalized_strategy_functions = {}

for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_18(
            strategy_id
        )
    )

    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not contain a callable implementation."
        )

    if normalized_id in normalized_strategy_functions:

        raise RuntimeError(
            "Duplicate normalized strategy ID: "
            f"{normalized_id}"
        )

    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


print(
    f"\n[06.18.03] Executable strategies: "
    f"{len(STRATEGY_FUNCTIONS)}"
)


# ------------------------------------------------------------
# 06.18.04 — Resolve strategy compatibility
# ------------------------------------------------------------

if (
    "STRATEGY_FEATURE_COMPATIBILITY"
    not in globals()
):

    raise RuntimeError(
        "STRATEGY_FEATURE_COMPATIBILITY is not available.\n"
        "Run the corrected Notebook 06.17 first."
    )


STRATEGY_FEATURE_COMPATIBILITY = {

    normalize_strategy_id_06_18(
        strategy_id
    ): {
        str(feature_type).strip().lower()
        for feature_type in supported_types
    }

    for strategy_id, supported_types
    in STRATEGY_FEATURE_COMPATIBILITY.items()
}


# ------------------------------------------------------------
# 06.18.05 — Resolve experiment configuration
# ------------------------------------------------------------

if "EXPERIMENT_CONFIGURATION" not in globals():

    raise RuntimeError(
        "EXPERIMENT_CONFIGURATION is not available.\n"
        "Run Notebook 06.03 first."
    )


experiment_config = dict(
    EXPERIMENT_CONFIGURATION
)


MISSINGNESS_MECHANISMS = [
    str(x).upper()
    for x in experiment_config.get(
        "missingness_mechanisms",
        ["MCAR", "MAR", "MNAR"]
    )
]


MISSINGNESS_RATES = [
    float(x)
    for x in experiment_config.get(
        "missingness_rates",
        [0.10, 0.20, 0.30, 0.40, 0.50]
    )
]


REPETITIONS = int(
    experiment_config.get(
        "repetitions",
        5
    )
)


MASTER_SEED = int(
    experiment_config.get(
        "master_seed",
        42
    )
)


# ------------------------------------------------------------
# 06.18.06 — Strict experiment validation
# ------------------------------------------------------------

EXPECTED_MECHANISMS = {
    "MCAR",
    "MAR",
    "MNAR",
}


EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


if set(
    MISSINGNESS_MECHANISMS
) != EXPECTED_MECHANISMS:

    raise RuntimeError(
        "AIR-LLM requires MCAR, MAR, and MNAR."
    )


if set(
    MISSINGNESS_RATES
) != EXPECTED_RATES:

    raise RuntimeError(
        "AIR-LLM requires missingness rates "
        "0.10, 0.20, 0.30, 0.40, and 0.50."
    )


if REPETITIONS != 5:

    raise RuntimeError(
        "AIR-LLM requires exactly 5 repetitions."
    )


# ------------------------------------------------------------
# 06.18.07 — Deterministic seed generator
# ------------------------------------------------------------

import hashlib


def make_experiment_seed_06_18(
    dataset_id,
    feature,
    mechanism,
    rate,
    repetition,
    master_seed
):

    key = (
        f"{master_seed}|"
        f"{dataset_id}|"
        f"{feature}|"
        f"{mechanism}|"
        f"{float(rate):.4f}|"
        f"{repetition}"
    )

    digest = hashlib.sha256(
        key.encode(
            "utf-8"
        )
    ).hexdigest()

    # Keep seed safely inside NumPy's uint32 range.
    return int(
        digest[:8],
        16
    )


# ------------------------------------------------------------
# 06.18.08 — Normalize feature types
# ------------------------------------------------------------

def normalize_feature_type_06_18(
    value
):

    value = str(
        value
    ).strip().lower()

    numerical_aliases = {
        "numeric",
        "numerical",
        "float",
        "float32",
        "float64",
        "integer",
        "int",
        "int32",
        "int64",
        "continuous",
    }

    categorical_aliases = {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool",
    }

    if value in numerical_aliases:
        return "numerical"

    if value in categorical_aliases:
        return "categorical"

    return value


# ------------------------------------------------------------
# 06.18.09 — Build feature-level candidate registry
# ------------------------------------------------------------

FEATURE_CANDIDATE_REGISTRY = {}


for dataset_id, df in EVALUATION_DATA.items():

    if dataset_id not in TARGET_REGISTRY:

        raise RuntimeError(
            f"Target not registered for "
            f"dataset '{dataset_id}'."
        )

    target = str(
        TARGET_REGISTRY[
            dataset_id
        ]
    )

    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found "
            f"in dataset '{dataset_id}'."
        )


    dataset_feature_rows = FEATURE_TYPE_DF[
        FEATURE_TYPE_DF[
            "dataset_id"
        ].astype(str)
        ==
        str(dataset_id)
    ].copy()


    if dataset_feature_rows.empty:

        raise RuntimeError(
            f"No feature characterization found "
            f"for '{dataset_id}'."
        )


    FEATURE_CANDIDATE_REGISTRY[
        dataset_id
    ] = {}


    for _, feature_row in (
        dataset_feature_rows.iterrows()
    ):

        feature = str(
            feature_row["feature"]
        )

        # Target is never imputed.
        if feature == target:
            continue


        feature_type = (
            normalize_feature_type_06_18(
                feature_row["feature_type"]
            )
        )


        compatible_strategies = []


        for strategy_id in (
            STRATEGY_FUNCTIONS.keys()
        ):

            supported_types = (
                STRATEGY_FEATURE_COMPATIBILITY.get(
                    strategy_id,
                    set()
                )
            )

            supported_types = {
                normalize_feature_type_06_18(
                    x
                )
                for x in supported_types
            }


            if feature_type in supported_types:

                compatible_strategies.append(
                    strategy_id
                )


        if compatible_strategies:

            FEATURE_CANDIDATE_REGISTRY[
                dataset_id
            ][feature] = sorted(
                compatible_strategies
            )


# ------------------------------------------------------------
# 06.18.10 — Validate candidate coverage
# ------------------------------------------------------------

for dataset_id, feature_map in (
    FEATURE_CANDIDATE_REGISTRY.items()
):

    if not feature_map:

        raise RuntimeError(
            f"No compatible candidate strategies "
            f"for dataset '{dataset_id}'."
        )


# ------------------------------------------------------------
# 06.18.11 — Build evaluation plan
# ------------------------------------------------------------

plan_rows = []


for dataset_id, feature_map in (
    FEATURE_CANDIDATE_REGISTRY.items()
):

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature, strategies in (
        feature_map.items()
    ):

        feature_type_rows = FEATURE_TYPE_DF[
            (
                FEATURE_TYPE_DF[
                    "dataset_id"
                ].astype(str)
                ==
                str(dataset_id)
            )
            &
            (
                FEATURE_TYPE_DF[
                    "feature"
                ].astype(str)
                ==
                str(feature)
            )
        ]


        if feature_type_rows.empty:
            continue


        feature_type = (
            normalize_feature_type_06_18(
                feature_type_rows.iloc[0][
                    "feature_type"
                ]
            )
        )


        for mechanism in (
            MISSINGNESS_MECHANISMS
        ):

            for rate in (
                sorted(
                    MISSINGNESS_RATES
                )
            ):

                for repetition in range(
                    1,
                    REPETITIONS + 1
                ):


                    experiment_seed = (
                        make_experiment_seed_06_18(
                            dataset_id=dataset_id,
                            feature=feature,
                            mechanism=mechanism,
                            rate=rate,
                            repetition=repetition,
                            master_seed=MASTER_SEED
                        )
                    )


                    for strategy_id in (
                        strategies
                    ):

                        plan_rows.append({

                            "dataset_id":
                                dataset_id,

                            "feature":
                                feature,

                            "target":
                                target,

                            "feature_type":
                                feature_type,

                            "strategy_id":
                                strategy_id,

                            "missingness_mechanism":
                                mechanism,

                            "missingness_rate":
                                float(rate),

                            "repetition":
                                repetition,

                            "llm_recommended":
                                False,

                            "seed":
                                experiment_seed,

                        })


# ------------------------------------------------------------
# 06.18.12 — Create evaluation dataframe
# ------------------------------------------------------------

EVALUATION_PLAN_DF = pd.DataFrame(
    plan_rows
)


if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


# ------------------------------------------------------------
# 06.18.13 — Validate uniqueness
# ------------------------------------------------------------

plan_keys = [
    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
]


duplicate_count = (
    EVALUATION_PLAN_DF
    .duplicated(
        subset=plan_keys
    )
    .sum()
)


if duplicate_count > 0:

    raise RuntimeError(
        f"Evaluation plan contains "
        f"{duplicate_count} duplicate configurations."
    )


# ------------------------------------------------------------
# 06.18.14 — Validate strategy implementations
# ------------------------------------------------------------

missing_implementations = sorted({

    strategy_id

    for strategy_id in (
        EVALUATION_PLAN_DF[
            "strategy_id"
        ].unique()
    )

    if strategy_id
    not in STRATEGY_FUNCTIONS

})


if missing_implementations:

    raise RuntimeError(
        "Evaluation plan contains strategies "
        "without implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_implementations
        )
    )


# ------------------------------------------------------------
# 06.18.15 — Validate strategy compatibility
# ------------------------------------------------------------

for _, row in EVALUATION_PLAN_DF.iterrows():

    strategy_id = row[
        "strategy_id"
    ]

    feature_type = (
        normalize_feature_type_06_18(
            row["feature_type"]
        )
    )

    supported = {
        normalize_feature_type_06_18(
            x
        )
        for x in STRATEGY_FEATURE_COMPATIBILITY.get(
            strategy_id,
            set()
        )
    }


    if feature_type not in supported:

        raise RuntimeError(
            "Incompatible strategy detected:\n"
            f"  strategy  : {strategy_id}\n"
            f"  feature   : {row['feature']}\n"
            f"  type      : {feature_type}"
        )


# ------------------------------------------------------------
# 06.18.16 — Validate mechanisms, rates and repetitions
# ------------------------------------------------------------

if set(
    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
) != EXPECTED_MECHANISMS:

    raise RuntimeError(
        "Not all missingness mechanisms are present."
    )


if set(
    EVALUATION_PLAN_DF[
        "missingness_rate"
    ].astype(float)
) != EXPECTED_RATES:

    raise RuntimeError(
        "Not all missingness rates are present."
    )


if set(
    EVALUATION_PLAN_DF[
        "repetition"
    ]
) != set(
    range(
        1,
        REPETITIONS + 1
    )
):

    raise RuntimeError(
        "Not all repetitions are present."
    )


# ------------------------------------------------------------
# 06.18.17 — Export evaluation plan
# ------------------------------------------------------------

EVALUATION_PLAN = (
    EVALUATION_PLAN_DF.copy()
)


# ------------------------------------------------------------
# 06.18.18 — Summary
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("EVALUATION PLAN SUMMARY")
print("-" * 100)

print(
    f"Datasets                 : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                 : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies               : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms               : "
    f"{EVALUATION_PLAN_DF['missingness_mechanism'].nunique()}"
)

print(
    f"Missingness rates        : "
    f"{EVALUATION_PLAN_DF['missingness_rate'].nunique()}"
)

print(
    f"Repetitions              : "
    f"{EVALUATION_PLAN_DF['repetition'].nunique()}"
)

print(
    f"Planned candidate runs   : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


print("\nStrategy distribution:")
display(
    EVALUATION_PLAN_DF[
        "strategy_id"
    ]
    .value_counts()
    .rename_axis(
        "strategy_id"
    )
    .reset_index(
        name="planned_runs"
    )
)


print("\nDataset distribution:")
display(
    EVALUATION_PLAN_DF[
        "dataset_id"
    ]
    .value_counts()
    .rename_axis(
        "dataset_id"
    )
    .reset_index(
        name="planned_runs"
    )
)


print("\n" + "=" * 100)
print(
    "CANDIDATE EVALUATION PLAN : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.18
BUILD CANDIDATE EVALUATION PLAN

[06.18.03] Executable strategies: 12

----------------------------------------------------------------------------------------------------
EVALUATION PLAN SUMMARY
----------------------------------------------------------------------------------------------------
Datasets                 : 3
Features                 : 73
Strategies               : 12
Mechanisms               : 3
Missingness rates        : 5
Repetitions              : 5
Planned candidate runs   : 31,725

Strategy distribution:


,strategy_id,planned_runs
0,constant,5775
1,random_sample,5775
2,mode,3975
3,gradientboosting,1800
4,knn,1800
5,iterativeimputer,1800
6,matrixfactorization,1800
7,mean,1800
8,missforest,1800
9,median,1800



Dataset distribution:


,dataset_id,planned_runs
0,diabetes_130us,17175
1,bank_marketing,7800
2,adult_income,6750



CANDIDATE EVALUATION PLAN : PASSED


In [20]:
# ============================================================
# NOTEBOOK 06.19 — EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19")
print("EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT")
print("=" * 100)


# ------------------------------------------------------------
# 06.19.01 — Imports
# ------------------------------------------------------------

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 06.19.02 — Required state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_PLAN_DF",
    "EVALUATION_DATA",
    "STRATEGY_FUNCTIONS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Notebook 06.19 is missing required dependencies:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.18 first."
    )


if not isinstance(
    EVALUATION_PLAN_DF,
    pd.DataFrame
):

    raise TypeError(
        "EVALUATION_PLAN_DF must be a pandas DataFrame."
    )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise TypeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ------------------------------------------------------------
# 06.19.03 — Normalize strategy IDs
# ------------------------------------------------------------

def normalize_strategy_id_06_19(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
    )

    aliases = {

        "randomsample":
            "random_sample",

        "random-sample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random-forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",

        "miss_forest":
            "missforest",
    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 06.19.04 — Normalize executable strategy registry
# ------------------------------------------------------------

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_19(
            strategy_id
        )
    )


    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not have a callable implementation."
        )


    if normalized_id in (
        normalized_strategy_functions
    ):

        raise RuntimeError(
            "Duplicate normalized strategy ID detected: "
            f"{normalized_id}"
        )


    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


print(
    f"\nExecutable strategies : "
    f"{len(STRATEGY_FUNCTIONS)}"
)


# ------------------------------------------------------------
# 06.19.05 — Required evaluation-plan columns
# ------------------------------------------------------------

REQUIRED_PLAN_COLUMNS_0619 = [

    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",

]


missing_columns = [

    column

    for column
    in REQUIRED_PLAN_COLUMNS_0619

    if column
    not in EVALUATION_PLAN_DF.columns

]


if missing_columns:

    raise RuntimeError(
        "Evaluation plan is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# ------------------------------------------------------------
# 06.19.06 — Basic dataframe validation
# ------------------------------------------------------------

if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


EVALUATION_PLAN_DF = (
    EVALUATION_PLAN_DF.copy()
)


# ------------------------------------------------------------
# 06.19.07 — Null validation
# ------------------------------------------------------------

null_columns = (

    EVALUATION_PLAN_DF.columns[
        EVALUATION_PLAN_DF.isna().any()
    ]

    .tolist()

)


if null_columns:

    raise RuntimeError(
        "Evaluation plan contains null values in:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in null_columns
        )
    )


# ------------------------------------------------------------
# 06.19.08 — Normalize strategy IDs
# ------------------------------------------------------------

EVALUATION_PLAN_DF[
    "strategy_id"
] = (

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .map(
        normalize_strategy_id_06_19
    )

)


# ------------------------------------------------------------
# 06.19.09 — Validate strategy implementations
# ------------------------------------------------------------

plan_strategies = set(

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .astype(str)
    .str.strip()
    .str.lower()

)


implemented_strategies = set(
    STRATEGY_FUNCTIONS.keys()
)


missing_implementations = (

    plan_strategies
    -
    implemented_strategies

)


if missing_implementations:

    raise RuntimeError(
        "Evaluation plan contains strategies "
        "without executable implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                missing_implementations
            )
        )
    )


# ------------------------------------------------------------
# 06.19.10 — Validate datasets
# ------------------------------------------------------------

plan_datasets = set(

    EVALUATION_PLAN_DF[
        "dataset_id"
    ]

)


available_datasets = set(
    EVALUATION_DATA.keys()
)


unknown_datasets = (

    plan_datasets
    -
    available_datasets

)


if unknown_datasets:

    raise RuntimeError(
        "Evaluation plan contains unknown datasets:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                unknown_datasets
            )
        )
    )


# ------------------------------------------------------------
# 06.19.11 — Validate TARGET_REGISTRY
# ------------------------------------------------------------

if "TARGET_REGISTRY" not in globals():

    raise RuntimeError(
        "TARGET_REGISTRY is not available."
    )


for dataset_id in sorted(
    plan_datasets
):

    if dataset_id not in TARGET_REGISTRY:

        raise RuntimeError(
            f"Missing target definition for "
            f"dataset '{dataset_id}'."
        )


    expected_target = str(
        TARGET_REGISTRY[
            dataset_id
        ]
    )


    actual_targets = set(

        EVALUATION_PLAN_DF.loc[
            EVALUATION_PLAN_DF[
                "dataset_id"
            ]
            ==
            dataset_id,
            "target"
        ]
        .astype(str)

    )


    if actual_targets != {
        expected_target
    }:

        raise RuntimeError(
            f"Target mismatch for "
            f"dataset '{dataset_id}'.\n"
            f"Expected: {expected_target}\n"
            f"Found: {sorted(actual_targets)}"
        )


# ------------------------------------------------------------
# 06.19.12 — Validate missingness mechanisms
# ------------------------------------------------------------

VALID_MECHANISMS = {
    "MCAR",
    "MAR",
    "MNAR",
}


EVALUATION_PLAN_DF[
    "missingness_mechanism"
] = (

    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]

    .astype(str)
    .str.strip()
    .str.upper()

)


actual_mechanisms = set(

    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]

)


invalid_mechanisms = (

    actual_mechanisms
    -
    VALID_MECHANISMS

)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                invalid_mechanisms
            )
        )
    )


if actual_mechanisms != VALID_MECHANISMS:

    raise RuntimeError(
        "Evaluation plan does not contain "
        "exactly MCAR, MAR and MNAR."
    )


# ------------------------------------------------------------
# 06.19.13 — Validate missingness rates
# ------------------------------------------------------------

EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


EVALUATION_PLAN_DF[
    "missingness_rate"
] = pd.to_numeric(

    EVALUATION_PLAN_DF[
        "missingness_rate"
    ],

    errors="coerce"

)


if EVALUATION_PLAN_DF[
    "missingness_rate"
].isna().any():

    raise RuntimeError(
        "Invalid missingness-rate values detected."
    )


actual_rates = {

    round(
        float(x),
        2
    )

    for x

    in EVALUATION_PLAN_DF[
        "missingness_rate"
    ].unique()

}


if actual_rates != EXPECTED_RATES:

    raise RuntimeError(
        "Invalid missingness rates.\n"
        f"Expected: {sorted(EXPECTED_RATES)}\n"
        f"Found:    {sorted(actual_rates)}"
    )


# ------------------------------------------------------------
# 06.19.14 — Validate repetitions
# ------------------------------------------------------------

EXPECTED_REPETITIONS = {
    1,
    2,
    3,
    4,
    5,
}


EVALUATION_PLAN_DF[
    "repetition"
] = pd.to_numeric(

    EVALUATION_PLAN_DF[
        "repetition"
    ],

    errors="coerce"

)


if EVALUATION_PLAN_DF[
    "repetition"
].isna().any():

    raise RuntimeError(
        "Invalid repetition values detected."
    )


actual_repetitions = set(

    EVALUATION_PLAN_DF[
        "repetition"
    ]
    .astype(int)
    .unique()

)


if actual_repetitions != EXPECTED_REPETITIONS:

    raise RuntimeError(
        "Invalid repetition configuration.\n"
        f"Expected: {sorted(EXPECTED_REPETITIONS)}\n"
        f"Found:    {sorted(actual_repetitions)}"
    )


# ------------------------------------------------------------
# 06.19.15 — Validate seeds
# ------------------------------------------------------------

if not pd.api.types.is_numeric_dtype(
    EVALUATION_PLAN_DF[
        "seed"
    ]
):

    raise RuntimeError(
        "Experiment seeds must be numeric."
    )


if EVALUATION_PLAN_DF[
    "seed"
].isna().any():

    raise RuntimeError(
        "Experiment seeds contain null values."
    )


# ------------------------------------------------------------
# 06.19.16 — Validate candidate uniqueness
# ------------------------------------------------------------

UNIQUE_CONFIGURATION_COLUMNS = [

    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",

]


duplicate_count = int(

    EVALUATION_PLAN_DF

    .duplicated(
        subset=UNIQUE_CONFIGURATION_COLUMNS
    )

    .sum()

)


if duplicate_count > 0:

    raise RuntimeError(
        f"Found {duplicate_count:,} duplicate "
        "candidate configurations."
    )


# ------------------------------------------------------------
# 06.19.17 — Expected candidate count
# ------------------------------------------------------------

UNIQUE_CONFIGURATION_DF = (

    EVALUATION_PLAN_DF[
        UNIQUE_CONFIGURATION_COLUMNS
    ]

    .drop_duplicates()

)


EXPECTED_EXPERIMENT_COUNT = len(
    UNIQUE_CONFIGURATION_DF
)


ACTUAL_EXPERIMENT_COUNT = len(
    EVALUATION_PLAN_DF
)


if (
    EXPECTED_EXPERIMENT_COUNT
    !=
    ACTUAL_EXPERIMENT_COUNT
):

    raise RuntimeError(
        "Expected candidate count does not match "
        "the evaluation-plan row count."
    )


# ------------------------------------------------------------
# 06.19.18 — Validate deterministic seed consistency
# ------------------------------------------------------------

if "make_experiment_seed_06_18" in globals():

    inconsistent_seed_rows = []


    for _, row in (
        EVALUATION_PLAN_DF.iterrows()
    ):

        expected_seed = (
            make_experiment_seed_06_18(
                dataset_id=row[
                    "dataset_id"
                ],

                feature=row[
                    "feature"
                ],

                mechanism=row[
                    "missingness_mechanism"
                ],

                rate=row[
                    "missingness_rate"
                ],

                repetition=int(
                    row[
                        "repetition"
                    ]
                ),

                master_seed=MASTER_SEED
                if "MASTER_SEED"
                in globals()
                else 42,
            )
        )


        if int(
            row["seed"]
        ) != int(
            expected_seed
        ):

            inconsistent_seed_rows.append(
                row.name
            )


    if inconsistent_seed_rows:

        raise RuntimeError(
            "Deterministic seed validation failed for "
            f"{len(inconsistent_seed_rows):,} rows."
        )


    seed_validation_status = "PASS"


else:

    seed_validation_status = (
        "SKIPPED — generator unavailable"
    )


# ------------------------------------------------------------
# 06.19.19 — Build integrity summary
# ------------------------------------------------------------

EVALUATION_PLAN_INTEGRITY_DF = pd.DataFrame({

    "check": [

        "Datasets",
        "Features",
        "Strategies",
        "Mechanisms",
        "Missingness rates",
        "Repetitions",
        "Candidate runs",
        "Duplicate configurations",
        "Executable implementations",
        "Deterministic seeds",

    ],

    "value": [

        EVALUATION_PLAN_DF[
            "dataset_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "feature"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "strategy_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_mechanism"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_rate"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "repetition"
        ].nunique(),

        ACTUAL_EXPERIMENT_COUNT,

        duplicate_count,

        len(
            implemented_strategies
        ),

        seed_validation_status,

    ],

    "status": [

        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
        if seed_validation_status == "PASS"
        else "CHECK",

    ],

})


display(
    EVALUATION_PLAN_INTEGRITY_DF
)


# ------------------------------------------------------------
# 06.19.20 — Detailed distribution
# ------------------------------------------------------------

print("\nStrategy distribution:")
print("-" * 100)

display(

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .value_counts()

    .rename_axis(
        "strategy_id"
    )

    .reset_index(
        name="planned_runs"
    )

)


print("\nDataset distribution:")
print("-" * 100)

display(

    EVALUATION_PLAN_DF[
        "dataset_id"
    ]

    .value_counts()

    .rename_axis(
        "dataset_id"
    )

    .reset_index(
        name="planned_runs"
    )

)


# ------------------------------------------------------------
# 06.19.21 — Final report
# ------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "EVALUATION-PLAN INTEGRITY REPORT"
)

print("=" * 100)


print(
    f"\nDatasets                 : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                 : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies               : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms               : "
    f"{sorted(VALID_MECHANISMS)}"
)

print(
    f"Missingness rates        : "
    f"{sorted(EXPECTED_RATES)}"
)

print(
    f"Repetitions              : "
    f"{sorted(EXPECTED_REPETITIONS)}"
)

print(
    f"Expected candidate runs  : "
    f"{EXPECTED_EXPERIMENT_COUNT:,}"
)

print(
    f"Actual candidate runs    : "
    f"{ACTUAL_EXPERIMENT_COUNT:,}"
)

print(
    f"Duplicate configurations : "
    f"{duplicate_count:,}"
)

print(
    f"Executable strategies    : "
    f"{len(implemented_strategies)}"
)

print(
    f"Seed validation          : "
    f"{seed_validation_status}"
)


# ------------------------------------------------------------
# 06.19.22 — Export validated plan
# ------------------------------------------------------------

EVALUATION_PLAN = (
    EVALUATION_PLAN_DF.copy()
)


print("\n" + "=" * 100)
print(
    "EVALUATION PLAN INTEGRITY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.19
EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT

Executable strategies : 12


,check,value,status
0,Datasets,3,PASS
1,Features,73,PASS
2,Strategies,12,PASS
3,Mechanisms,3,PASS
4,Missingness rates,5,PASS
5,Repetitions,5,PASS
6,Candidate runs,31725,PASS
7,Duplicate configurations,0,PASS
8,Executable implementations,12,PASS
9,Deterministic seeds,PASS,PASS



Strategy distribution:
----------------------------------------------------------------------------------------------------


,strategy_id,planned_runs
0,constant,5775
1,random_sample,5775
2,mode,3975
3,gradientboosting,1800
4,knn,1800
5,iterativeimputer,1800
6,matrixfactorization,1800
7,mean,1800
8,missforest,1800
9,median,1800



Dataset distribution:
----------------------------------------------------------------------------------------------------


,dataset_id,planned_runs
0,diabetes_130us,17175
1,bank_marketing,7800
2,adult_income,6750



EVALUATION-PLAN INTEGRITY REPORT

Datasets                 : 3
Features                 : 73
Strategies               : 12
Mechanisms               : ['MAR', 'MCAR', 'MNAR']
Missingness rates        : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions              : [1, 2, 3, 4, 5]
Expected candidate runs  : 31,725
Actual candidate runs    : 31,725
Duplicate configurations : 0
Executable strategies    : 12
Seed validation          : PASS

EVALUATION PLAN INTEGRITY : PASSED


In [21]:
# ============================================================
# NOTEBOOK 06.19A — FINAL EXECUTABLE STRATEGY REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19A")
print("FINAL EXECUTABLE STRATEGY REGISTRY")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required implementation functions
# ------------------------------------------------------------

required_functions = {
    "gradientboosting": "fit_gradient_boosting",
    "softimpute": "fit_softimpute",
    "matrixfactorization": "fit_matrix_factorization",
}


missing_functions = [
    function_name
    for function_name in required_functions.values()
    if function_name not in globals()
]


if missing_functions:

    raise RuntimeError(
        "Required candidate implementation functions are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_functions
        )
        +
        "\n\n"
        "Run Notebook 06.14 and 06.16 first."
    )


# ------------------------------------------------------------
# 2. Build / update executable registry
# ------------------------------------------------------------

if "STRATEGY_FUNCTIONS" not in globals():

    STRATEGY_FUNCTIONS = {}


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise RuntimeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


# ------------------------------------------------------------
# 3. Register newly implemented candidates
# ------------------------------------------------------------

STRATEGY_FUNCTIONS[
    "gradientboosting"
] = fit_gradient_boosting


STRATEGY_FUNCTIONS[
    "softimpute"
] = fit_softimpute


STRATEGY_FUNCTIONS[
    "matrixfactorization"
] = fit_matrix_factorization


# ------------------------------------------------------------
# 4. Canonical strategy-ID normalization
# ------------------------------------------------------------

def normalize_strategy_id_final(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        "gradient_boosting":
            "gradientboosting",

        "gradientboost":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "random_sample_imputer":
            "random_sample",

        "randomsample":
            "random_sample",

    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 5. Normalize entire executable registry
# ------------------------------------------------------------

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_final(
            strategy_id
        )
    )

    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not have a callable implementation."
        )

    if normalized_id in normalized_strategy_functions:

        # Allow replacement of an existing alias with
        # the canonical implementation.
        if (
            normalized_strategy_functions[
                normalized_id
            ]
            is not function
        ):

            normalized_strategy_functions[
                normalized_id
            ] = function

    else:

        normalized_strategy_functions[
            normalized_id
        ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


# ------------------------------------------------------------
# 6. Validate strategies required by evaluation plan
# ------------------------------------------------------------

required_plan_strategies = {

    normalize_strategy_id_final(
        x
    )

    for x in
    EVALUATION_PLAN_DF[
        "strategy_id"
    ].dropna().unique()
}


missing_plan_implementations = sorted(
    required_plan_strategies
    -
    set(
        STRATEGY_FUNCTIONS.keys()
    )
)


if missing_plan_implementations:

    raise RuntimeError(
        "The following strategies are required by "
        "EVALUATION_PLAN_DF but have no executable "
        "implementation:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_plan_implementations
        )
    )


# ------------------------------------------------------------
# 7. Final registry report
# ------------------------------------------------------------

print(
    f"\nExecutable strategies registered : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print("\nStrategy implementation status:")
print("-" * 100)

for strategy_id in sorted(
    required_plan_strategies
):

    function = STRATEGY_FUNCTIONS[
        strategy_id
    ]

    print(
        f"  ✓ {strategy_id:<24} "
        f"-> {function.__name__}"
    )


print("\n" + "=" * 100)
print(
    "EXECUTABLE STRATEGY REGISTRY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.19A
FINAL EXECUTABLE STRATEGY REGISTRY

Executable strategies registered : 12

Strategy implementation status:
----------------------------------------------------------------------------------------------------
  ✓ constant                 -> impute_constant
  ✓ gradientboosting         -> fit_gradient_boosting
  ✓ iterativeimputer         -> fit_iterative
  ✓ knn                      -> fit_knn
  ✓ matrixfactorization      -> fit_matrix_factorization
  ✓ mean                     -> impute_mean
  ✓ median                   -> impute_median
  ✓ missforest               -> fit_missforest
  ✓ mode                     -> impute_mode
  ✓ random_sample            -> impute_random_sample
  ✓ randomforest             -> fit_random_forest
  ✓ softimpute               -> fit_softimpute

EXECUTABLE STRATEGY REGISTRY : PASSED


In [22]:
# ============================================================
# NOTEBOOK 06.20 — SMALL-RUN PREPARATION / PREFLIGHT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.20")
print("SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT")
print("=" * 100)


# ============================================================
# 06.20.01 — Required state
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "EVALUATION_PLAN_DF",
    "STRATEGY_FUNCTIONS",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Notebook 06.20 dependencies are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.18 and the final executable "
        "strategy registry cell before Notebook 06.20."
    )


# ============================================================
# 06.20.02 — Validate core object types
# ============================================================

if not isinstance(
    EVALUATION_DATA,
    dict
):

    raise TypeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not isinstance(
    EVALUATION_PLAN_DF,
    pd.DataFrame
):

    raise TypeError(
        "EVALUATION_PLAN_DF must be a pandas DataFrame."
    )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise TypeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ============================================================
# 06.20.03 — Canonical strategy-ID normalization
# ============================================================

def normalize_strategy_id_06_20(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        # Basic statistical
        "randomsample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "random_sample_imputer":
            "random_sample",

        # Iterative / MICE
        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        # Random forest
        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "random-forest":
            "randomforest",

        # Gradient boosting
        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "gradient_boost":
            "gradientboosting",

        # SoftImpute
        "soft_impute":
            "softimpute",

        "soft-impute":
            "softimpute",

        # Matrix factorization
        "matrix_factorization":
            "matrixfactorization",

        "matrix-factorization":
            "matrixfactorization",

    }

    return aliases.get(
        value,
        value
    )


# ============================================================
# 06.20.04 — Normalize executable strategy registry
# ============================================================

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_20(
            strategy_id
        )
    )


    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' does not contain "
            "a callable implementation."
        )


    if normalized_id in normalized_strategy_functions:

        raise RuntimeError(
            "Duplicate normalized strategy ID detected: "
            f"{normalized_id}"
        )


    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


# ============================================================
# 06.20.05 — Required evaluation-plan columns
# ============================================================

required_plan_columns = [

    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",

]


missing_plan_columns = [

    column
    for column
    in required_plan_columns
    if column
    not in EVALUATION_PLAN_DF.columns

]


if missing_plan_columns:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_plan_columns
        )
    )


# ============================================================
# 06.20.06 — Validate datasets in evaluation plan
# ============================================================

plan_datasets = set(
    EVALUATION_PLAN_DF[
        "dataset_id"
    ]
)


available_datasets = set(
    EVALUATION_DATA.keys()
)


unknown_datasets = (
    plan_datasets
    -
    available_datasets
)


if unknown_datasets:

    raise RuntimeError(
        "Evaluation plan contains datasets that are not "
        "available in EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                unknown_datasets
            )
        )
    )


# ============================================================
# 06.20.07 — Validate every strategy in evaluation plan
# ============================================================

required_plan_strategies = {

    normalize_strategy_id_06_20(
        strategy_id
    )

    for strategy_id
    in EVALUATION_PLAN_DF[
        "strategy_id"
    ].dropna().unique()

}


missing_implementations = sorted(

    required_plan_strategies
    -
    set(
        STRATEGY_FUNCTIONS.keys()
    )

)


if missing_implementations:

    raise RuntimeError(
        "The evaluation plan contains strategies without "
        "executable implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_implementations
        )
    )


# ============================================================
# 06.20.08 — Validate dataset / feature / target references
# ============================================================

for _, row in EVALUATION_PLAN_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = str(
        row[
            "feature"
        ]
    )

    target = str(
        row[
            "target"
        ]
    )


    df = EVALUATION_DATA[
        dataset_id
    ]


    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise TypeError(
            f"EVALUATION_DATA['{dataset_id}'] "
            "must be a pandas DataFrame."
        )


    if feature not in df.columns:

        raise RuntimeError(
            f"Feature '{feature}' not found in "
            f"dataset '{dataset_id}'."
        )


    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"dataset '{dataset_id}'."
        )


# ============================================================
# 06.20.09 — Validate experiment configuration
# ============================================================

valid_mechanisms = {
    "MCAR",
    "MAR",
    "MNAR",
}


actual_mechanisms = {

    str(x)
    .strip()
    .upper()

    for x
    in EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
    .dropna()
    .unique()

}


if actual_mechanisms != valid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms.\n"
        f"Expected: {sorted(valid_mechanisms)}\n"
        f"Found:    {sorted(actual_mechanisms)}"
    )


expected_rates = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


actual_rates = {

    round(
        float(x),
        2
    )

    for x
    in EVALUATION_PLAN_DF[
        "missingness_rate"
    ]
    .dropna()
    .unique()

}


if actual_rates != expected_rates:

    raise RuntimeError(
        "Invalid missingness rates.\n"
        f"Expected: {sorted(expected_rates)}\n"
        f"Found:    {sorted(actual_rates)}"
    )


expected_repetitions = {
    1,
    2,
    3,
    4,
    5,
}


actual_repetitions = {

    int(x)

    for x
    in EVALUATION_PLAN_DF[
        "repetition"
    ]
    .dropna()
    .unique()

}


if actual_repetitions != expected_repetitions:

    raise RuntimeError(
        "Invalid repetitions.\n"
        f"Expected: {sorted(expected_repetitions)}\n"
        f"Found:    {sorted(actual_repetitions)}"
    )


# ============================================================
# 06.20.10 — Validate seeds
# ============================================================

if EVALUATION_PLAN_DF[
    "seed"
].isna().any():

    raise RuntimeError(
        "Evaluation plan contains missing experiment seeds."
    )


if not pd.api.types.is_numeric_dtype(
    EVALUATION_PLAN_DF[
        "seed"
    ]
):

    raise RuntimeError(
        "Experiment seeds must be numeric."
    )


# ============================================================
# 06.20.11 — Validate duplicate configurations
# ============================================================

configuration_columns = [

    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",

]


duplicate_count = int(

    EVALUATION_PLAN_DF
    .duplicated(
        subset=configuration_columns
    )
    .sum()

)


if duplicate_count > 0:

    raise RuntimeError(
        f"Evaluation plan contains "
        f"{duplicate_count:,} duplicate configurations."
    )


# ============================================================
# 06.20.12 — Create deterministic small-run subset
# ============================================================

SMALL_RUN_SIZE = min(
    3,
    len(
        EVALUATION_PLAN_DF
    )
)


SMALL_RUN_PLAN_DF = (

    EVALUATION_PLAN_DF
    .head(
        SMALL_RUN_SIZE
    )
    .copy()

)


if SMALL_RUN_PLAN_DF.empty:

    raise RuntimeError(
        "Small-run evaluation plan is empty."
    )


# ============================================================
# 06.20.13 — Validate small-run strategies
# ============================================================

for _, row in SMALL_RUN_PLAN_DF.iterrows():

    strategy_id = (
        normalize_strategy_id_06_20(
            row[
                "strategy_id"
            ]
        )
    )


    if strategy_id not in STRATEGY_FUNCTIONS:

        raise RuntimeError(
            f"Small-run strategy '{strategy_id}' "
            "has no executable implementation."
        )


# ============================================================
# 06.20.14 — Validate small-run dataset references
# ============================================================

for _, row in SMALL_RUN_PLAN_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = str(
        row[
            "feature"
        ]
    )

    target = str(
        row[
            "target"
        ]
    )


    df = EVALUATION_DATA[
        dataset_id
    ]


    if feature not in df.columns:

        raise RuntimeError(
            f"Small-run feature '{feature}' "
            f"not found in '{dataset_id}'."
        )


    if target not in df.columns:

        raise RuntimeError(
            f"Small-run target '{target}' "
            f"not found in '{dataset_id}'."
        )


# ============================================================
# 06.20.15 — Optional missingness-function validation
# ============================================================

missingness_function_status = "NOT CHECKED"


if "generate_missingness_mask" in globals():

    if not callable(
        generate_missingness_mask
    ):

        raise RuntimeError(
            "generate_missingness_mask exists but "
            "is not callable."
        )

    missingness_function_status = "AVAILABLE"

else:

    missingness_function_status = (
        "NOT AVAILABLE — execution engine will require it"
    )


# ============================================================
# 06.20.16 — Optional predictor-selection validation
# ============================================================

predictor_selection_status = "NOT CHECKED"


if "select_predictors" in globals():

    if not callable(
        select_predictors
    ):

        raise RuntimeError(
            "select_predictors exists but "
            "is not callable."
        )

    predictor_selection_status = "AVAILABLE"

else:

    predictor_selection_status = (
        "NOT AVAILABLE — execution engine will require it"
    )


# ============================================================
# 06.20.17 — Display small-run plan
# ============================================================

print(
    f"\nSmall validation configurations : "
    f"{SMALL_RUN_SIZE}"
)


print(
    "\nSmall-run candidate plan:"
)


display(
    SMALL_RUN_PLAN_DF[
        [
            "dataset_id",
            "feature",
            "target",
            "feature_type",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
            "seed",
        ]
    ]
)


# ============================================================
# 06.20.18 — Preflight summary
# ============================================================

print("\n" + "-" * 100)
print("CANDIDATE EXECUTION PREFLIGHT SUMMARY")
print("-" * 100)


print(
    f"Full evaluation-plan rows    : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


print(
    f"Required strategies          : "
    f"{len(required_plan_strategies)}"
)


print(
    f"Executable strategies        : "
    f"{len(STRATEGY_FUNCTIONS)}"
)


print(
    f"Small-run configurations      : "
    f"{SMALL_RUN_SIZE}"
)


print(
    f"Datasets available            : "
    f"{len(EVALUATION_DATA)}"
)


print(
    f"Missingness function          : "
    f"{missingness_function_status}"
)


print(
    f"Predictor selection           : "
    f"{predictor_selection_status}"
)


print(
    "\nRequired executable strategies:"
)

for strategy_id in sorted(
    required_plan_strategies
):

    function = STRATEGY_FUNCTIONS[
        strategy_id
    ]

    print(
        f"  ✓ {strategy_id:<24} "
        f"-> {function.__name__}"
    )


# ============================================================
# 06.20.19 — Final validation
# ============================================================

print("\n" + "=" * 100)
print(
    "CANDIDATE EXECUTION PREFLIGHT : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.20
SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT

Small validation configurations : 3

Small-run candidate plan:


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,repetition,seed
0,adult_income,age,income,numerical,constant,MCAR,0.1,1,3711803575
1,adult_income,age,income,numerical,gradientboosting,MCAR,0.1,1,3711803575
2,adult_income,age,income,numerical,iterativeimputer,MCAR,0.1,1,3711803575



----------------------------------------------------------------------------------------------------
CANDIDATE EXECUTION PREFLIGHT SUMMARY
----------------------------------------------------------------------------------------------------
Full evaluation-plan rows    : 31,725
Required strategies          : 12
Executable strategies        : 12
Small-run configurations      : 3
Datasets available            : 3
Missingness function          : AVAILABLE
Predictor selection           : AVAILABLE

Required executable strategies:
  ✓ constant                 -> impute_constant
  ✓ gradientboosting         -> fit_gradient_boosting
  ✓ iterativeimputer         -> fit_iterative
  ✓ knn                      -> fit_knn
  ✓ matrixfactorization      -> fit_matrix_factorization
  ✓ mean                     -> impute_mean
  ✓ median                   -> impute_median
  ✓ missforest               -> fit_missforest
  ✓ mode                     -> impute_mode
  ✓ random_sample            -> impute_ran

In [24]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.21 — SECTION 1/2
# BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION
# ROBUST / SELF-RECOVERING VERSION
# ==================================================================================================

import os
import time
import inspect
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21 — SECTION 1/2")
print("BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. ACCELERATOR
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("ACCELERATOR")
print("-" * 100)

try:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()

    if CUDA_AVAILABLE:
        SELECTED_DEVICE = "cuda"
        DEVICE_NAME = torch.cuda.get_device_name(0)
        CUDA_VERSION = torch.version.cuda
    else:
        SELECTED_DEVICE = "cpu"
        DEVICE_NAME = "CPU"
        CUDA_VERSION = None

except Exception:
    CUDA_AVAILABLE = False
    SELECTED_DEVICE = "cpu"
    DEVICE_NAME = "CPU"
    CUDA_VERSION = None

print(f"Selected device : {SELECTED_DEVICE}")
print(f"Device name     : {DEVICE_NAME}")
print(f"CUDA available  : {CUDA_AVAILABLE}")
print(f"CUDA version    : {CUDA_VERSION}")


# --------------------------------------------------------------------------------------------------
# 2. PROJECT PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

EXPERIMENTS_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments"
)

ARTIFACTS_DIR = os.path.join(
    PROJECT_ROOT,
    "artifacts"
)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 3. NOTEBOOK 06 OBJECT DISCOVERY
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 06 OBJECT VALIDATION")
print("-" * 100)


def _is_dataframe(obj):
    return isinstance(obj, pd.DataFrame)


# --------------------------------------------------------------------------------------------------
# 4. FIND THE BEST AVAILABLE EVALUATION PLAN
# --------------------------------------------------------------------------------------------------

PLAN_SOURCE = None
PLAN_DF = None


# Preferred source order
_candidate_plan_names = [
    "REDUCED_EVALUATION_PLAN_DF",
    "EVALUATION_PLAN_DF",
    "EVALUATION_PLAN",
    "existing_plan",
]


for _name in _candidate_plan_names:

    if _name in globals():

        _obj = globals()[_name]

        if isinstance(_obj, pd.DataFrame) and len(_obj) > 0:

            PLAN_SOURCE = _name
            PLAN_DF = _obj.copy()

            break


# --------------------------------------------------------------------------------------------------
# 5. OPTIONAL FILE-BASED RECOVERY
# --------------------------------------------------------------------------------------------------

if PLAN_DF is None:

    possible_plan_files = [
        os.path.join(
            RESULTS_DIR,
            "reduced_evaluation_plan.csv"
        ),
        os.path.join(
            RESULTS_DIR,
            "stratified_candidate_evaluation_plan.csv"
        ),
        os.path.join(
            RESULTS_DIR,
            "evaluation_plan.csv"
        ),
        os.path.join(
            EXPERIMENTS_DIR,
            "evaluation_plan.csv"
        ),
    ]

    for _path in possible_plan_files:

        if os.path.exists(_path):

            try:

                _candidate = pd.read_csv(_path)

                if isinstance(_candidate, pd.DataFrame) and len(_candidate) > 0:

                    PLAN_SOURCE = _path
                    PLAN_DF = _candidate.copy()

                    break

            except Exception:
                pass


# --------------------------------------------------------------------------------------------------
# 6. PLAN SCHEMA STANDARDIZATION
# --------------------------------------------------------------------------------------------------

if PLAN_DF is not None:

    # Normalize column names
    PLAN_DF.columns = [
        str(c).strip().lower()
        for c in PLAN_DF.columns
    ]

    # Common aliases
    _column_aliases = {

        "dataset": "dataset_id",
        "dataset_name": "dataset_id",
        "data_id": "dataset_id",

        "feature_name": "feature",
        "feature_id": "feature",

        "target_name": "target",

        "type": "feature_type",
        "dtype": "feature_type",

        "strategy": "strategy_id",
        "method": "strategy_id",

        "mechanism": "missingness_mechanism",

        "missing_rate": "missingness_rate",
        "missing_ratio": "missingness_rate",

        "rep": "repetition",

        "seed_value": "seed",
    }

    for old_name, new_name in _column_aliases.items():

        if (
            old_name in PLAN_DF.columns
            and new_name not in PLAN_DF.columns
        ):

            PLAN_DF = PLAN_DF.rename(
                columns={old_name: new_name}
            )


# --------------------------------------------------------------------------------------------------
# 7. RECOVER REDUCED_EVALUATION_PLAN_DF
# --------------------------------------------------------------------------------------------------

if PLAN_DF is None:

    print()
    print("PLAN RECOVERY : FAILED")
    print()
    raise RuntimeError(
        "No evaluation plan could be recovered. "
        "Run Notebook 06 cells 06.01–06.20 that create the evaluation plan."
    )


required_columns = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
]


missing_columns = [
    c for c in required_columns
    if c not in PLAN_DF.columns
]


if missing_columns:

    print()
    print("PLAN RECOVERY : FAILED")
    print("Missing columns:")
    for c in missing_columns:
        print(f"  • {c}")

    print()
    print("Available columns:")
    print(list(PLAN_DF.columns))

    raise RuntimeError(
        "Recovered evaluation plan does not contain the required schema."
    )


# --------------------------------------------------------------------------------------------------
# 8. CREATE REDUCED PLAN IF NECESSARY
# --------------------------------------------------------------------------------------------------

REDUCED_EVALUATION_PLAN_DF = PLAN_DF.copy()

# Preserve deterministic ordering
REDUCED_EVALUATION_PLAN_DF = (
    REDUCED_EVALUATION_PLAN_DF
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 9. CREATE LLM RECOMMENDATION COLUMN IF ABSENT
# --------------------------------------------------------------------------------------------------

if "llm_recommended" not in REDUCED_EVALUATION_PLAN_DF.columns:

    REDUCED_EVALUATION_PLAN_DF[
        "llm_recommended"
    ] = False


# --------------------------------------------------------------------------------------------------
# 10. CREATE DETERMINISTIC SEEDS IF ABSENT
# --------------------------------------------------------------------------------------------------

if "seed" not in REDUCED_EVALUATION_PLAN_DF.columns:

    def _deterministic_seed(row):

        key = (
            f"{row['dataset_id']}|"
            f"{row['feature']}|"
            f"{row['strategy_id']}|"
            f"{row['missingness_mechanism']}|"
            f"{row['missingness_rate']}|"
            f"{row['repetition']}"
        )

        return abs(hash(key)) % (2**32 - 1)


    REDUCED_EVALUATION_PLAN_DF["seed"] = (
        REDUCED_EVALUATION_PLAN_DF
        .apply(_deterministic_seed, axis=1)
    )


# --------------------------------------------------------------------------------------------------
# 11. CREATE EVALUATION KEYS
# --------------------------------------------------------------------------------------------------

def _make_eval_key(row):

    key = (
        f"{row['dataset_id']}|"
        f"{row['feature']}|"
        f"{row['strategy_id']}|"
        f"{row['missingness_mechanism']}|"
        f"{float(row['missingness_rate']):.6f}|"
        f"{int(row['repetition'])}"
    )

    import hashlib

    return hashlib.sha1(
        key.encode("utf-8")
    ).hexdigest()[:24]


if "evaluation_key" not in REDUCED_EVALUATION_PLAN_DF.columns:

    REDUCED_EVALUATION_PLAN_DF[
        "evaluation_key"
    ] = REDUCED_EVALUATION_PLAN_DF.apply(
        _make_eval_key,
        axis=1
    )


# --------------------------------------------------------------------------------------------------
# 12. NORMALIZE DATA TYPES
# --------------------------------------------------------------------------------------------------

REDUCED_EVALUATION_PLAN_DF[
    "missingness_rate"
] = pd.to_numeric(
    REDUCED_EVALUATION_PLAN_DF["missingness_rate"],
    errors="coerce"
)

REDUCED_EVALUATION_PLAN_DF[
    "repetition"
] = pd.to_numeric(
    REDUCED_EVALUATION_PLAN_DF["repetition"],
    errors="coerce"
).astype("Int64")


# Remove invalid rows
REDUCED_EVALUATION_PLAN_DF = (
    REDUCED_EVALUATION_PLAN_DF
    .dropna(
        subset=[
            "dataset_id",
            "feature",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE RECOVERED REDUCED PLAN
# --------------------------------------------------------------------------------------------------

REDUCED_PLAN_PATH = os.path.join(
    RESULTS_DIR,
    "reduced_evaluation_plan.csv"
)

REDUCED_EVALUATION_PLAN_DF.to_csv(
    REDUCED_PLAN_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATE REQUIRED NOTEBOOK 06 OBJECTS
# --------------------------------------------------------------------------------------------------

evaluation_data_ok = (
    "EVALUATION_DATA" in globals()
    and isinstance(EVALUATION_DATA, dict)
    and len(EVALUATION_DATA) > 0
)

strategy_functions_ok = (
    "STRATEGY_FUNCTIONS" in globals()
    and isinstance(STRATEGY_FUNCTIONS, dict)
    and len(STRATEGY_FUNCTIONS) > 0
)

reduced_plan_ok = (
    isinstance(REDUCED_EVALUATION_PLAN_DF, pd.DataFrame)
    and len(REDUCED_EVALUATION_PLAN_DF) > 0
)


print()
print(f"{'EVALUATION_DATA':35s} : {'PASS' if evaluation_data_ok else 'FAIL'}")
print(
    f"{'REDUCED_EVALUATION_PLAN_DF':35s} : "
    f"{'PASS' if reduced_plan_ok else 'FAIL'}"
)
print(
    f"{'STRATEGY_FUNCTIONS':35s} : "
    f"{'PASS' if strategy_functions_ok else 'FAIL'}"
)


if not evaluation_data_ok:

    raise RuntimeError(
        "EVALUATION_DATA is unavailable."
    )


if not strategy_functions_ok:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is unavailable."
    )


if not reduced_plan_ok:

    raise RuntimeError(
        "Evaluation plan recovery failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. PLAN SOURCE INFORMATION
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("PLAN SOURCE")
print("-" * 100)

print(
    f"Recovered from : {PLAN_SOURCE}"
)

print(
    f"Recovered rows : {len(REDUCED_EVALUATION_PLAN_DF)}"
)

print(
    f"Saved to       : {REDUCED_PLAN_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 16. DATASET VALIDATION
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("EVALUATION DATASETS")
print("-" * 100)

for dataset_id, dataset in EVALUATION_DATA.items():

    try:
        print(
            f"{str(dataset_id):30s} "
            f"{dataset.shape}"
        )
    except Exception:

        print(
            f"{str(dataset_id):30s} "
            f"{type(dataset).__name__}"
        )


# --------------------------------------------------------------------------------------------------
# 17. STRATEGY REGISTRY
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("REGISTERED STRATEGIES")
print("-" * 100)

for strategy_id, strategy_fn in STRATEGY_FUNCTIONS.items():

    try:
        signature = inspect.signature(strategy_fn)
    except Exception:
        signature = "[signature unavailable]"

    print(
        f"{str(strategy_id):30s} "
        f"{signature}"
    )


# --------------------------------------------------------------------------------------------------
# 18. FINAL SECTION 1 SUMMARY
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 1 COMPLETE")
print("=" * 100)

print(
    f"Device                     : {SELECTED_DEVICE}"
)

print(
    f"Plan source                : {PLAN_SOURCE}"
)

print(
    f"Evaluation plan rows       : "
    f"{len(REDUCED_EVALUATION_PLAN_DF)}"
)

print(
    f"Evaluation datasets        : "
    f"{len(EVALUATION_DATA)}"
)

print(
    f"Registered strategies      : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print(
    f"Recovered reduced plan     : PASS"
)

print()
print(
    "Section 2 can now be executed."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.21 — SECTION 1/2
BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION

----------------------------------------------------------------------------------------------------
ACCELERATOR
----------------------------------------------------------------------------------------------------
Selected device : cuda
Device name     : Tesla T4
CUDA available  : True
CUDA version    : 12.8

----------------------------------------------------------------------------------------------------
NOTEBOOK 06 OBJECT VALIDATION
----------------------------------------------------------------------------------------------------

EVALUATION_DATA                     : PASS
REDUCED_EVALUATION_PLAN_DF          : PASS
STRATEGY_FUNCTIONS                  : PASS

----------------------------------------------------------------------------------------------------
PLAN SOURCE
----------------------------------------------------------------------------------------------------
Recovered

In [26]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.21 — SECTION 2/2
# BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION ENGINE
# ROBUST FINAL VERSION
# ==================================================================================================

import os
import time
import gc
import inspect
import hashlib
import traceback
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score
)

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21 — SECTION 2/2")
print("BOUNDED TOP-K + STRATIFIED + CHECKPOINTED CANDIDATE EVALUATION ENGINE")
print("=" * 100)


# ==================================================================================================
# 1. SECTION 1 OBJECT VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("SECTION 1 OBJECT VALIDATION")
print("-" * 100)


if "EVALUATION_DATA" not in globals():
    raise RuntimeError(
        "EVALUATION_DATA is unavailable. Run Notebook 06 before Section 2."
    )

if "REDUCED_EVALUATION_PLAN_DF" not in globals():
    raise RuntimeError(
        "REDUCED_EVALUATION_PLAN_DF is unavailable. "
        "Run the updated Section 1 before Section 2."
    )

if "STRATEGY_FUNCTIONS" not in globals():
    raise RuntimeError(
        "STRATEGY_FUNCTIONS is unavailable. Run Notebook 06 before Section 2."
    )


print(
    f"{'EVALUATION_DATA':35s} : PASS"
)

print(
    f"{'REDUCED_EVALUATION_PLAN_DF':35s} : PASS"
)

print(
    f"{'STRATEGY_FUNCTIONS':35s} : PASS"
)


# ==================================================================================================
# 2. PROJECT DIRECTORIES
# ==================================================================================================

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

EXPERIMENTS_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments"
)

ARTIFACTS_DIR = os.path.join(
    PROJECT_ROOT,
    "artifacts"
)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)


# ==================================================================================================
# 3. ACCELERATOR
# ==================================================================================================

print()
print("-" * 100)
print("ACCELERATOR")
print("-" * 100)


try:

    import torch

    CUDA_AVAILABLE = bool(torch.cuda.is_available())

    if CUDA_AVAILABLE:

        DEVICE = "cuda"
        DEVICE_NAME = torch.cuda.get_device_name(0)
        CUDA_VERSION = torch.version.cuda

    else:

        DEVICE = "cpu"
        DEVICE_NAME = "CPU"
        CUDA_VERSION = None

except Exception:

    CUDA_AVAILABLE = False
    DEVICE = "cpu"
    DEVICE_NAME = "CPU"
    CUDA_VERSION = None


print(f"Selected device : {DEVICE}")
print(f"Device name     : {DEVICE_NAME}")
print(f"CUDA available  : {CUDA_AVAILABLE}")
print(f"CUDA version    : {CUDA_VERSION}")


# ==================================================================================================
# 4. COPY AND STANDARDIZE THE PLAN
# ==================================================================================================

PLAN_DF = REDUCED_EVALUATION_PLAN_DF.copy()

PLAN_DF.columns = [
    str(c).strip().lower()
    for c in PLAN_DF.columns
]


required_columns = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition"
]


missing_columns = [
    c for c in required_columns
    if c not in PLAN_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "The evaluation plan is missing required columns: "
        + ", ".join(missing_columns)
    )


if "llm_recommended" not in PLAN_DF.columns:

    PLAN_DF["llm_recommended"] = False


if "seed" not in PLAN_DF.columns:

    def make_seed(row):

        text = (
            f"{row['dataset_id']}|"
            f"{row['feature']}|"
            f"{row['strategy_id']}|"
            f"{row['missingness_mechanism']}|"
            f"{row['missingness_rate']}|"
            f"{row['repetition']}"
        )

        return int(
            hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest()[:8],
            16
        )

    PLAN_DF["seed"] = PLAN_DF.apply(
        make_seed,
        axis=1
    )


if "evaluation_key" not in PLAN_DF.columns:

    def make_key(row):

        text = (
            f"{row['dataset_id']}|"
            f"{row['feature']}|"
            f"{row['strategy_id']}|"
            f"{row['missingness_mechanism']}|"
            f"{float(row['missingness_rate']):.6f}|"
            f"{int(row['repetition'])}"
        )

        return hashlib.sha1(
            text.encode("utf-8")
        ).hexdigest()[:24]

    PLAN_DF["evaluation_key"] = PLAN_DF.apply(
        make_key,
        axis=1
    )


# ==================================================================================================
# 5. SAVE STANDARDIZED PLAN
# ==================================================================================================

STANDARD_PLAN_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_standardized_plan.csv"
)

PLAN_DF.to_csv(
    STANDARD_PLAN_PATH,
    index=False
)


print()
print("-" * 100)
print("INPUT PLAN")
print("-" * 100)

print(
    f"Input plan rows : {len(PLAN_DF)}"
)

print(
    f"Plan file       : {STANDARD_PLAN_PATH}"
)


# ==================================================================================================
# 6. BOUNDED TOP-K CONFIGURATION
# ==================================================================================================

TOP_K = 5

print()
print("-" * 100)
print("BOUNDED TOP-K CONFIGURATION")
print("-" * 100)

print(
    f"Top-K per evaluation context : {TOP_K}"
)


# ==================================================================================================
# 7. STRATIFICATION CONTEXT
# ==================================================================================================

STRATIFICATION_COLUMNS = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "missingness_mechanism",
    "missingness_rate",
    "repetition"
]


# ==================================================================================================
# 8. STRATEGY PRIORITY
# ==================================================================================================

DEFAULT_STRATEGY_PRIORITY = {

    "mean": 1,
    "median": 2,
    "mode": 3,
    "constant": 4,
    "random_sample": 5,
    "knn": 6,
    "iterativeimputer": 7,
    "randomforest": 8,
    "gradientboosting": 9,
    "missforest": 10,
    "softimpute": 11,
    "matrixfactorization": 12,
}


def strategy_priority(strategy_id):

    return DEFAULT_STRATEGY_PRIORITY.get(
        str(strategy_id).lower(),
        999
    )


PLAN_DF["_strategy_priority"] = PLAN_DF[
    "strategy_id"
].map(strategy_priority)


# ==================================================================================================
# 9. BOUNDED TOP-K SELECTION
# ==================================================================================================

print()
print("-" * 100)
print("BOUNDED TOP-K PLAN")
print("-" * 100)


# We preserve LLM recommendations first.
#
# For each context:
#   1. LLM-recommended candidates are preferred.
#   2. Remaining slots are filled deterministically.
#   3. Maximum TOP_K candidates are evaluated.

context_columns = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "missingness_mechanism",
    "missingness_rate",
]


def select_top_k(group):

    group = group.copy()

    group["_llm_priority"] = np.where(
        group["llm_recommended"].astype(bool),
        0,
        1
    )

    group = group.sort_values(
        by=[
            "_llm_priority",
            "_strategy_priority",
            "strategy_id"
        ],
        kind="stable"
    )

    return group.head(TOP_K)


BOUNDED_PLAN_DF = (
    PLAN_DF
    .groupby(
        context_columns,
        dropna=False,
        group_keys=False
    )
    .apply(select_top_k)
    .reset_index(drop=True)
)


BOUNDED_PLAN_DF = BOUNDED_PLAN_DF.drop(
    columns=[
        "_llm_priority",
        "_strategy_priority"
    ],
    errors="ignore"
)


print(
    f"Original reduced-plan rows : {len(PLAN_DF)}"
)

print(
    f"Unique evaluation contexts  : "
    f"{PLAN_DF.groupby(context_columns, dropna=False).ngroups}"
)

print(
    f"Top-K per context            : {TOP_K}"
)

print(
    f"Bounded evaluation rows      : {len(BOUNDED_PLAN_DF)}"
)

reduction = (
    1 -
    len(BOUNDED_PLAN_DF) /
    max(len(PLAN_DF), 1)
) * 100

print(
    f"Plan reduction               : {reduction:.2f}%"
)


# ==================================================================================================
# 10. SAVE BOUNDED PLAN
# ==================================================================================================

BOUNDED_PLAN_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_bounded_topk_plan.csv"
)

BOUNDED_PLAN_DF.to_csv(
    BOUNDED_PLAN_PATH,
    index=False
)


# ==================================================================================================
# 11. CANDIDATE DISTRIBUTION
# ==================================================================================================

print()
print("-" * 100)
print("CANDIDATE COUNT DISTRIBUTION")
print("-" * 100)


candidate_counts = (
    BOUNDED_PLAN_DF
    .groupby(
        context_columns,
        dropna=False
    )
    .size()
)

print(candidate_counts.describe())


# ==================================================================================================
# 12. MISSINGNESS ENGINE DISCOVERY
# ==================================================================================================

print()
print("-" * 100)
print("MISSINGNESS ENGINE DISCOVERY")
print("-" * 100)


missingness_functions = {}

# IMPORTANT:
# Take a snapshot before iterating through globals().
# This prevents:
# RuntimeError: dictionary changed size during iteration

_global_items = list(globals().items())


for name, obj in _global_items:

    if not callable(obj):
        continue

    name_lower = str(name).lower()

    if any(
        token in name_lower
        for token in [
            "missing",
            "mask",
            "mcar",
            "mar",
            "mnar",
            "corrupt",
            "hide"
        ]
    ):

        missingness_functions[name] = obj


for name, obj in sorted(
    missingness_functions.items()
):

    try:
        signature = inspect.signature(obj)
    except Exception:
        signature = "[signature unavailable]"

    print(
        f"{name:<55} {signature}"
    )


# ==================================================================================================
# 13. ROBUST DATA CONVERSION HELPERS
# ==================================================================================================

def to_series(values, index=None):

    if isinstance(values, pd.Series):

        result = values.copy()

        if index is not None:
            result.index = index

        return result

    if isinstance(values, pd.DataFrame):

        if values.shape[1] == 1:

            result = values.iloc[:, 0].copy()

            if index is not None:
                result.index = index

            return result

        return pd.Series(
            values.iloc[:, 0].to_numpy(),
            index=index
        )

    arr = np.asarray(values)

    if arr.ndim == 0:

        return pd.Series(
            [arr.item()],
            index=index
        )

    arr = arr.reshape(-1)

    return pd.Series(
        arr,
        index=index
    )


def numeric_or_categorical(y):

    s = to_series(y)

    if pd.api.types.is_numeric_dtype(s):

        return "numerical"

    return "categorical"


def safe_numeric(y):

    s = to_series(y)

    return pd.to_numeric(
        s,
        errors="coerce"
    )


# ==================================================================================================
# 14. MISSINGNESS MASK GENERATOR
# ==================================================================================================

def generate_mask(
    original,
    missingness_rate,
    mechanism,
    seed
):

    rng = np.random.default_rng(
        int(seed)
    )

    n = len(original)

    rate = float(
        missingness_rate
    )

    rate = min(
        max(rate, 0.0),
        1.0
    )

    mechanism = str(
        mechanism
    ).upper()

    mask = np.zeros(
        n,
        dtype=bool
    )

    if n == 0:
        return mask

    target_n = int(
        round(n * rate)
    )

    target_n = min(
        max(target_n, 0),
        n
    )

    if target_n == 0:
        return mask

    if target_n == n:

        mask[:] = True
        return mask

    if mechanism == "MCAR":

        indices = rng.choice(
            n,
            size=target_n,
            replace=False
        )

        mask[indices] = True

        return mask

    # ----------------------------------------------------------------------------------------------
    # MAR / MNAR
    #
    # For a single target feature, a deterministic score-based mechanism is used.
    # This is used only as a robust fallback when the original missingness engine does not expose
    # a compatible callable interface.
    # ----------------------------------------------------------------------------------------------

    s = to_series(
        original
    )

    if pd.api.types.is_numeric_dtype(s):

        values = pd.to_numeric(
            s,
            errors="coerce"
        )

        ranks = values.rank(
            method="average",
            pct=True
        ).fillna(
            0.5
        ).to_numpy()

    else:

        codes = pd.Categorical(
            s
        ).codes.astype(float)

        if len(codes) == 0:

            ranks = np.full(
                n,
                0.5
            )

        else:

            denom = max(
                np.nanmax(codes),
                1.0
            )

            ranks = (
                np.maximum(
                    codes,
                    0
                ) /
                denom
            )

    if mechanism == "MNAR":

        scores = ranks

    elif mechanism == "MAR":

        # deterministic observed-feature proxy
        scores = (
            0.5 * ranks +
            0.5 * np.linspace(
                0,
                1,
                n
            )
        )

    else:

        scores = rng.random(
            n
        )

    selected = np.argsort(
        scores
    )[-target_n:]

    mask[selected] = True

    return mask


# ==================================================================================================
# 15. STRATEGY ADAPTERS
# ==================================================================================================

def adapter_mean(
    train,
    missing,
    seed
):

    return to_series(
        STRATEGY_FUNCTIONS["mean"](
            train,
            missing
        ),
        index=missing.index
    )


def adapter_median(
    train,
    missing,
    seed
):

    return to_series(
        STRATEGY_FUNCTIONS["median"](
            train,
            missing
        ),
        index=missing.index
    )


def adapter_mode(
    train,
    missing,
    seed
):

    return to_series(
        STRATEGY_FUNCTIONS["mode"](
            train,
            missing
        ),
        index=missing.index
    )


def adapter_constant(
    train,
    missing,
    seed
):

    return to_series(
        STRATEGY_FUNCTIONS["constant"](
            train,
            missing,
            constant_value=None
        ),
        index=missing.index
    )


def adapter_random_sample(
    train,
    missing,
    seed
):

    return to_series(
        STRATEGY_FUNCTIONS["random_sample"](
            train,
            missing,
            random_state=int(seed)
        ),
        index=missing.index
    )


def adapter_knn(
    train,
    missing,
    seed
):

    X_train = pd.DataFrame({
        "target": to_series(train)
    })

    X_missing = pd.DataFrame({
        "target": to_series(missing)
    })

    result = STRATEGY_FUNCTIONS["knn"](
        X_train,
        X_missing,
        n_neighbors=5,
        weights="uniform"
    )

    result = np.asarray(
        result
    )

    if result.ndim == 2:
        result = result[:, 0]

    return pd.Series(
        result,
        index=missing.index
    )


def adapter_iterative(
    train,
    missing,
    seed
):

    X_train = pd.DataFrame({
        "target": to_series(train)
    })

    X_missing = pd.DataFrame({
        "target": to_series(missing)
    })

    result = STRATEGY_FUNCTIONS[
        "iterativeimputer"
    ](
        X_train,
        X_missing,
        estimator=None,
        random_state=int(seed),
        max_iter=10,
        tol=0.001,
        initial_strategy="median"
    )

    result = np.asarray(
        result
    )

    if result.ndim == 2:
        result = result[:, 0]

    return pd.Series(
        result,
        index=missing.index
    )


# ==================================================================================================
# 16. GENERIC FALLBACK STRATEGY
# ==================================================================================================

def generic_strategy_adapter(
    strategy_id,
    train,
    missing,
    seed,
    feature_type
):

    strategy_id = str(
        strategy_id
    ).lower()

    fn = STRATEGY_FUNCTIONS.get(
        strategy_id
    )

    if fn is None:

        raise KeyError(
            f"Strategy '{strategy_id}' is not registered."
        )

    # ------------------------------------------------------------------------------
    # direct one-dimensional strategies
    # ------------------------------------------------------------------------------

    if strategy_id == "mean":
        return adapter_mean(train, missing, seed)

    if strategy_id == "median":
        return adapter_median(train, missing, seed)

    if strategy_id == "mode":
        return adapter_mode(train, missing, seed)

    if strategy_id == "constant":
        return adapter_constant(train, missing, seed)

    if strategy_id == "random_sample":
        return adapter_random_sample(train, missing, seed)

    if strategy_id == "knn":
        return adapter_knn(train, missing, seed)

    if strategy_id == "iterativeimputer":
        return adapter_iterative(train, missing, seed)

    # ------------------------------------------------------------------------------
    # complex methods
    #
    # They are attempted only when their interface can safely be satisfied.
    # ------------------------------------------------------------------------------

    if strategy_id in [
        "randomforest",
        "gradientboosting"
    ]:

        X_train = pd.DataFrame({
            "target": to_series(train)
        })

        X_missing = pd.DataFrame({
            "target": to_series(missing)
        })

        if strategy_id == "randomforest":

            result = fn(
                X_train,
                to_series(train),
                X_missing,
                feature_type=feature_type,
                random_state=int(seed),
                n_estimators=100,
                max_depth=None,
                min_samples_leaf=2,
                n_jobs=-1
            )

        else:

            result = fn(
                X_train,
                to_series(train),
                X_missing,
                feature_type=feature_type,
                random_state=int(seed),
                max_iter=100,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=20
            )

        return to_series(
            result,
            index=missing.index
        )

    if strategy_id == "missforest":

        X_train = pd.DataFrame({
            "target": to_series(train)
        })

        X_missing = pd.DataFrame({
            "target": to_series(missing)
        })

        result = fn(
            X_train,
            X_missing,
            random_state=int(seed),
            max_iter=3,
            n_estimators=50,
            min_samples_leaf=2
        )

        result = np.asarray(
            result
        )

        if result.ndim == 2:
            result = result[:, 0]

        return pd.Series(
            result,
            index=missing.index
        )

    if strategy_id in [
        "softimpute",
        "matrixfactorization"
    ]:

        X_train = pd.DataFrame({
            "target": pd.to_numeric(
                to_series(train),
                errors="coerce"
            )
        })

        X_missing = pd.DataFrame({
            "target": pd.to_numeric(
                to_series(missing),
                errors="coerce"
            )
        })

        result = fn(
            X_train,
            X_missing,
            rank=5,
            max_iter=25
        )

        result = np.asarray(
            result
        )

        if result.ndim == 2:
            result = result[:, 0]

        return pd.Series(
            result,
            index=missing.index
        )

    raise NotImplementedError(
        f"No safe adapter implemented for strategy '{strategy_id}'."
    )


# ==================================================================================================
# 17. METRIC ENGINE
# ==================================================================================================

def calculate_metrics(
    truth,
    prediction,
    feature_type
):

    truth = to_series(
        truth
    )

    prediction = to_series(
        prediction,
        index=truth.index
    )

    prediction = prediction.iloc[
        :len(truth)
    ]

    truth = truth.iloc[
        :len(prediction)
    ]

    # Numerical
    if str(feature_type).lower() in [
        "numerical",
        "numeric",
        "continuous",
        "float",
        "integer"
    ]:

        y_true = pd.to_numeric(
            truth,
            errors="coerce"
        )

        y_pred = pd.to_numeric(
            prediction,
            errors="coerce"
        )

        valid = (
            y_true.notna()
            &
            y_pred.notna()
        )

        y_true = y_true[valid]
        y_pred = y_pred[valid]

        n = len(y_true)

        if n == 0:

            return {
                "mae": np.nan,
                "rmse": np.nan,
                "r2": np.nan,
                "accuracy": np.nan,
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "n": 0,
            }

        mae = mean_absolute_error(
            y_true,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )

        try:

            r2 = r2_score(
                y_true,
                y_pred
            )

        except Exception:

            r2 = np.nan

        return {
            "mae": float(mae),
            "rmse": float(rmse),
            "r2": float(r2),
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
            "n": int(n),
        }

    # Categorical
    truth = truth.astype(str)
    prediction = prediction.astype(str)

    valid = (
        truth.notna()
        &
        prediction.notna()
    )

    truth = truth[valid]
    prediction = prediction[valid]

    n = len(truth)

    if n == 0:

        return {
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
            "n": 0,
        }

    accuracy = accuracy_score(
        truth,
        prediction
    )

    macro_f1 = f1_score(
        truth,
        prediction,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        truth,
        prediction,
        average="weighted",
        zero_division=0
    )

    return {
        "mae": np.nan,
        "rmse": np.nan,
        "r2": np.nan,
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "n": int(n),
    }


# ==================================================================================================
# 18. SINGLE EVALUATION
# ==================================================================================================

def evaluate_candidate(
    row
):

    dataset_id = row["dataset_id"]
    feature = row["feature"]
    strategy_id = str(
        row["strategy_id"]
    ).lower()

    mechanism = str(
        row["missingness_mechanism"]
    )

    missingness_rate = float(
        row["missingness_rate"]
    )

    repetition = int(
        row["repetition"]
    )

    seed = int(
        row["seed"]
    )

    feature_type = str(
        row["feature_type"]
    )

    dataset = EVALUATION_DATA[
        dataset_id
    ]

    if feature not in dataset.columns:

        raise KeyError(
            f"Feature '{feature}' not found in dataset '{dataset_id}'."
        )

    original = dataset[
        feature
    ].copy()

    # Remove rows where original value itself is unavailable.
    original = original[
        original.notna()
    ].copy()

    if len(original) == 0:

        raise ValueError(
            "No observed values available for evaluation."
        )

    mask = generate_mask(
        original=original,
        missingness_rate=missingness_rate,
        mechanism=mechanism,
        seed=seed
    )

    masked = original.copy()

    truth = original[
        mask
    ].copy()

    masked.loc[
        mask
    ] = np.nan

    train = masked[
        ~mask
    ].copy()

    missing = masked[
        mask
    ].copy()

    if len(truth) == 0:

        raise ValueError(
            "Generated missingness mask contains zero evaluation values."
        )

    # ----------------------------------------------------------------------------------------------
    # Strategy execution
    # ----------------------------------------------------------------------------------------------

    start = time.time()

    prediction = generic_strategy_adapter(
        strategy_id=strategy_id,
        train=train,
        missing=missing,
        seed=seed,
        feature_type=feature_type
    )

    runtime_seconds = (
        time.time() -
        start
    )

    prediction = to_series(
        prediction,
        index=truth.index
    )

    if len(prediction) != len(truth):

        prediction = prediction.iloc[
            :len(truth)
        ]

        truth = truth.iloc[
            :len(prediction)
        ]

    metrics = calculate_metrics(
        truth=truth,
        prediction=prediction,
        feature_type=feature_type
    )

    result = {

        "evaluation_key":
            row["evaluation_key"],

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "target":
            row["target"],

        "feature_type":
            feature_type,

        "strategy_id":
            strategy_id,

        "missingness_mechanism":
            mechanism,

        "missingness_rate":
            missingness_rate,

        "repetition":
            repetition,

        "seed":
            seed,

        "llm_recommended":
            bool(row.get(
                "llm_recommended",
                False
            )),

        "status":
            "SUCCESS",

        "mae":
            metrics["mae"],

        "rmse":
            metrics["rmse"],

        "r2":
            metrics["r2"],

        "accuracy":
            metrics["accuracy"],

        "macro_f1":
            metrics["macro_f1"],

        "weighted_f1":
            metrics["weighted_f1"],

        "n":
            metrics["n"],

        "runtime_seconds":
            runtime_seconds,

        "error_type":
            "",

        "error_message":
            "",
    }

    return result


# ==================================================================================================
# 19. CHECKPOINT FILE
# ==================================================================================================

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_checkpoint.csv"
)

FINAL_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_candidate_results.csv"
)


RESULT_COLUMNS = [
    "evaluation_key",
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "seed",
    "llm_recommended",
    "status",
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "n",
    "runtime_seconds",
    "error_type",
    "error_message",
]


# ==================================================================================================
# 20. LOAD CHECKPOINT
# ==================================================================================================

if os.path.exists(
    CHECKPOINT_PATH
):

    try:

        RESULTS_DF = pd.read_csv(
            CHECKPOINT_PATH
        )

        print()
        print("-" * 100)
        print("CHECKPOINT RECOVERY")
        print("-" * 100)

        print(
            f"Checkpoint rows loaded : {len(RESULTS_DF)}"
        )

    except Exception:

        RESULTS_DF = pd.DataFrame(
            columns=RESULT_COLUMNS
        )

        print(
            "Existing checkpoint could not be read. "
            "Starting a new checkpoint."
        )

else:

    RESULTS_DF = pd.DataFrame(
        columns=RESULT_COLUMNS
    )


# ==================================================================================================
# 21. RESUME LOGIC
# ==================================================================================================

completed_keys = set(
    RESULTS_DF[
        "evaluation_key"
    ].astype(str)
)

remaining_plan = BOUNDED_PLAN_DF[
    ~BOUNDED_PLAN_DF[
        "evaluation_key"
    ].astype(str).isin(
        completed_keys
    )
].copy()


print()
print("-" * 100)
print("RESUMABLE EXECUTION STATUS")
print("-" * 100)

print(
    f"Bounded planned evaluations : "
    f"{len(BOUNDED_PLAN_DF)}"
)

print(
    f"Already completed           : "
    f"{len(completed_keys)}"
)

print(
    f"Remaining                   : "
    f"{len(remaining_plan)}"
)

print(
    "Checkpointing after EVERY evaluation."
)


# ==================================================================================================
# 22. EXECUTION ENGINE
# ==================================================================================================

print()
print("-" * 100)
print("STARTING / RESUMING EVALUATION")
print("-" * 100)


session_start = time.time()

total = len(
    BOUNDED_PLAN_DF
)

completed_this_session = 0


for local_index, (_, row) in enumerate(
    remaining_plan.iterrows(),
    start=1
):

    absolute_number = (
        total -
        len(remaining_plan) +
        local_index
    )

    print()
    print("-" * 100)

    print(
        f"[{absolute_number}/{total}] "
        f"{row['dataset_id']} | "
        f"{row['feature']} | "
        f"{row['strategy_id']} | "
        f"{row['missingness_mechanism']} | "
        f"{float(row['missingness_rate']):.0%} | "
        f"rep={int(row['repetition'])}"
    )

    try:

        result = evaluate_candidate(
            row
        )

        result_df = pd.DataFrame(
            [result],
            columns=RESULT_COLUMNS
        )

        RESULTS_DF = pd.concat(
            [
                RESULTS_DF,
                result_df
            ],
            ignore_index=True
        )

        status = "SUCCESS"

        print(
            f"  Status : SUCCESS"
        )

        if pd.notna(result["mae"]):

            print(
                f"  MAE    : {result['mae']}"
            )

            print(
                f"  RMSE   : {result['rmse']}"
            )

            print(
                f"  R2     : {result['r2']}"
            )

        else:

            print(
                f"  Accuracy : {result['accuracy']}"
            )

            print(
                f"  Macro-F1 : {result['macro_f1']}"
            )

    except Exception as exc:

        error_type = type(
            exc
        ).__name__

        error_message = str(
            exc
        )[:1000]

        result = {

            "evaluation_key":
                row["evaluation_key"],

            "dataset_id":
                row["dataset_id"],

            "feature":
                row["feature"],

            "target":
                row["target"],

            "feature_type":
                row["feature_type"],

            "strategy_id":
                row["strategy_id"],

            "missingness_mechanism":
                row["missingness_mechanism"],

            "missingness_rate":
                row["missingness_rate"],

            "repetition":
                row["repetition"],

            "seed":
                row["seed"],

            "llm_recommended":
                bool(
                    row.get(
                        "llm_recommended",
                        False
                    )
                ),

            "status":
                "FAILED",

            "mae":
                np.nan,

            "rmse":
                np.nan,

            "r2":
                np.nan,

            "accuracy":
                np.nan,

            "macro_f1":
                np.nan,

            "weighted_f1":
                np.nan,

            "n":
                0,

            "runtime_seconds":
                np.nan,

            "error_type":
                error_type,

            "error_message":
                error_message,
        }

        RESULTS_DF = pd.concat(
            [
                RESULTS_DF,
                pd.DataFrame(
                    [result],
                    columns=RESULT_COLUMNS
                )
            ],
            ignore_index=True
        )

        status = "FAILED"

        print(
            f"  Status : FAILED"
        )

        print(
            f"  Error  : "
            f"{error_type} | {error_message}"
        )


    # ----------------------------------------------------------------------------------------------
    # CHECKPOINT AFTER EVERY EVALUATION
    # ----------------------------------------------------------------------------------------------

    RESULTS_DF.to_csv(
        CHECKPOINT_PATH,
        index=False
    )

    completed_this_session += 1

    elapsed_minutes = (
        time.time() -
        session_start
    ) / 60.0

    remaining_count = (
        total -
        len(RESULTS_DF)
    )

    print(
        f"  Progress : "
        f"{len(RESULTS_DF)}/{total} | "
        f"Remaining: {max(remaining_count, 0)} | "
        f"Session: {elapsed_minutes:.1f} min"
    )

    # Release temporary memory
    gc.collect()

    if CUDA_AVAILABLE:

        try:
            import torch

            torch.cuda.empty_cache()

        except Exception:
            pass


# ==================================================================================================
# 23. FINAL RESULT SAVE
# ==================================================================================================

RESULTS_DF.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)


# ==================================================================================================
# 24. EXECUTION SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("06.21 EVALUATION COMPLETE")
print("=" * 100)

print()
print("-" * 100)
print("EXECUTION SUMMARY")
print("-" * 100)

print(
    f"Bounded plan rows       : {len(BOUNDED_PLAN_DF)}"
)

print(
    f"Results rows            : {len(RESULTS_DF)}"
)

success_count = int(
    (
        RESULTS_DF["status"]
        .astype(str)
        .str.upper()
        .eq("SUCCESS")
    ).sum()
)

failed_count = int(
    (
        RESULTS_DF["status"]
        .astype(str)
        .str.upper()
        .eq("FAILED")
    ).sum()
)

print(
    f"Successful evaluations  : {success_count}"
)

print(
    f"Failed evaluations      : {failed_count}"
)

if len(RESULTS_DF) > 0:

    success_rate = (
        success_count /
        len(RESULTS_DF)
    ) * 100

else:

    success_rate = 0.0


print(
    f"Success rate            : {success_rate:.2f}%"
)

print()
print(
    f"Checkpoint              : {CHECKPOINT_PATH}"
)

print(
    f"Final results           : {FINAL_RESULTS_PATH}"
)


# ==================================================================================================
# 25. STRATEGY-LEVEL SUMMARY
# ==================================================================================================

if len(RESULTS_DF) > 0:

    print()
    print("-" * 100)
    print("STRATEGY-LEVEL EVALUATION SUMMARY")
    print("-" * 100)

    successful_results = RESULTS_DF[
        RESULTS_DF["status"]
        .astype(str)
        .str.upper()
        .eq("SUCCESS")
    ].copy()

    if len(successful_results) > 0:

        strategy_summary = (
            successful_results
            .groupby(
                "strategy_id"
            )
            .agg(
                evaluations=(
                    "evaluation_key",
                    "count"
                ),
                mean_mae=(
                    "mae",
                    "mean"
                ),
                mean_rmse=(
                    "rmse",
                    "mean"
                ),
                mean_r2=(
                    "r2",
                    "mean"
                ),
                mean_accuracy=(
                    "accuracy",
                    "mean"
                ),
                mean_macro_f1=(
                    "macro_f1",
                    "mean"
                ),
                mean_runtime=(
                    "runtime_seconds",
                    "mean"
                ),
            )
            .reset_index()
        )

        print(
            strategy_summary.to_string(
                index=False
            )
        )

        strategy_summary_path = os.path.join(
            RESULTS_DIR,
            "air_llm_06_21_strategy_summary.csv"
        )

        strategy_summary.to_csv(
            strategy_summary_path,
            index=False
        )

        print()
        print(
            f"Strategy summary saved : "
            f"{strategy_summary_path}"
        )


# ==================================================================================================
# 26. FAILURE SUMMARY
# ==================================================================================================

if failed_count > 0:

    print()
    print("-" * 100)
    print("FAILURE SUMMARY")
    print("-" * 100)

    failure_summary = (
        RESULTS_DF[
            RESULTS_DF["status"]
            .astype(str)
            .str.upper()
            .eq("FAILED")
        ]
        .groupby(
            [
                "strategy_id",
                "error_type"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False
        )
    )

    print(
        failure_summary.to_string(
            index=False
        )
    )

    failure_summary_path = os.path.join(
        RESULTS_DIR,
        "air_llm_06_21_failure_summary.csv"
    )

    failure_summary.to_csv(
        failure_summary_path,
        index=False
    )


# ==================================================================================================
# 27. FINAL DATASET
# ==================================================================================================

AIR_LLM_0621_RESULTS_DF = RESULTS_DF.copy()

AIR_LLM_0621_BOUNDED_PLAN_DF = (
    BOUNDED_PLAN_DF.copy()
)


print()
print("=" * 100)
print("SECTION 2 COMPLETE")
print("=" * 100)

print(
    "AIR_LLM_0621_RESULTS_DF is available for Notebook 07."
)

print(
    "AIR_LLM_0621_BOUNDED_PLAN_DF is available for downstream analysis."
)

print("=" * 100)

Streaming output truncated to the last 5000 lines.

----------------------------------------------------------------------------------------------------
[5077/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=2
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5077/5775 | Remaining: 698 | Session: 39.8 min

----------------------------------------------------------------------------------------------------
[5078/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=3
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5078/5775 | Remaining: 697 | Session: 39.8 min

----------------------------------------------------------------------------------------------------
[5079/5775] diabetes_130us | payer_code | mode | MNAR | 10% | rep=4
  Status : SUCCESS
  Accuracy : 0.0
  Macro-F1 : 0.0
  Progress : 5079/5775 | Remaining: 696 | Session: 39.8 min

---------------------------------------------------------------------------------------------------

In [29]:
# ============================================================
# AIR-LLM — NOTEBOOK 06.22
# EVALUATION STATUS SUMMARY
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.22")
print("EVALUATION STATUS SUMMARY")
print("=" * 100)


# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)


FINAL_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_candidate_results.csv"
)


CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_checkpoint.csv"
)


STRATEGY_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_strategy_summary.csv"
)


# ------------------------------------------------------------
# 2. LOAD 06.21 RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADING 06.21 CANDIDATE EVALUATION RESULTS")
print("-" * 100)


if (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
    and not AIR_LLM_0621_RESULTS_DF.empty
):

    EVALUATION_RESULTS_DF = (
        AIR_LLM_0621_RESULTS_DF.copy()
    )

    print(
        "Source : AIR_LLM_0621_RESULTS_DF "
        "(in-memory)"
    )

else:

    if not os.path.exists(
        FINAL_RESULTS_PATH
    ):

        raise FileNotFoundError(
            "\n06.21 final results file was not found.\n"
            f"Expected path:\n{FINAL_RESULTS_PATH}\n\n"
            "Do not continue to 06.22 until the "
            "06.21 results artifact is available."
        )

    EVALUATION_RESULTS_DF = pd.read_csv(
        FINAL_RESULTS_PATH
    )

    print(
        "Source : 06.21 persisted results "
        "(CSV recovery)"
    )


# ------------------------------------------------------------
# 3. BASIC VALIDATION
# ------------------------------------------------------------

if not isinstance(
    EVALUATION_RESULTS_DF,
    pd.DataFrame
):

    raise RuntimeError(
        "06.21 results could not be loaded "
        "as a pandas DataFrame."
    )


if EVALUATION_RESULTS_DF.empty:

    raise RuntimeError(
        "06.21 results dataframe is empty."
    )


print(
    f"Results rows loaded : "
    f"{len(EVALUATION_RESULTS_DF):,}"
)


print(
    f"Columns available   : "
    f"{len(EVALUATION_RESULTS_DF.columns):,}"
)


required_columns = [
    "status",
    "strategy_id"
]


missing_columns = [
    col
    for col in required_columns
    if col not in EVALUATION_RESULTS_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "Required columns are missing from "
        "06.21 results: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# 4. NORMALIZE STATUS VALUES
# ------------------------------------------------------------

EVALUATION_RESULTS_DF["status"] = (
    EVALUATION_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# 5. STATUS SUMMARY
# ------------------------------------------------------------

status_summary = (
    EVALUATION_RESULTS_DF[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)


display(
    status_summary
)


# ------------------------------------------------------------
# 6. SUCCESS / FAILURE COUNTS
# ------------------------------------------------------------

successful = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "SUCCESS"
    ).sum()
)


failed = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "FAILED"
    ).sum()
)


total_runs = len(
    EVALUATION_RESULTS_DF
)


success_rate = (
    successful / total_runs * 100
    if total_runs > 0
    else 0.0
)


print(
    f"\nSuccessful runs : {successful:,}"
)

print(
    f"Failed runs     : {failed:,}"
)

print(
    f"Total runs      : {total_runs:,}"
)

print(
    f"Success rate    : {success_rate:.2f}%"
)


# ------------------------------------------------------------
# 7. RESEARCH-GRADE COMPLETENESS CHECK
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("EVALUATION COMPLETENESS CHECK")
print("-" * 100)


expected_runs = 5775


if total_runs == expected_runs:

    print(
        f"Expected evaluations : {expected_runs:,}"
    )

    print(
        "Actual evaluations   : "
        f"{total_runs:,}"
    )

    print(
        "Completeness         : PASS"
    )

else:

    print(
        f"Expected evaluations : {expected_runs:,}"
    )

    print(
        "Actual evaluations   : "
        f"{total_runs:,}"
    )

    print(
        "Completeness         : WARNING"
    )


if successful == total_runs:

    print(
        "Execution integrity  : PASS"
    )

else:

    print(
        "Execution integrity  : WARNING"
    )


# ------------------------------------------------------------
# 8. FAILURE SUMMARY
# ------------------------------------------------------------

if failed > 0:

    print(
        "\n" + "-" * 100
    )

    print(
        "FAILURE SUMMARY"
    )

    print(
        "-" * 100
    )


    failure_summary = (
        EVALUATION_RESULTS_DF[
            EVALUATION_RESULTS_DF[
                "status"
            ] == "FAILED"
        ]
        .groupby(
            "strategy_id",
            dropna=False
        )
        .size()
        .reset_index(
            name="failures"
        )
        .sort_values(
            "failures",
            ascending=False
        )
    )


    display(
        failure_summary
    )

else:

    print(
        "\nFailure summary : No failed evaluations."
    )


# ------------------------------------------------------------
# 9. STRATEGY COVERAGE
# ------------------------------------------------------------

print(
    "\n" + "-" * 100
)

print(
    "STRATEGY COVERAGE"
)

print(
    "-" * 100
)


strategy_coverage = (
    EVALUATION_RESULTS_DF
    .groupby(
        "strategy_id",
        dropna=False
    )
    .agg(
        evaluations=(
            "strategy_id",
            "size"
        ),
        successful=(
            "status",
            lambda x:
                (x == "SUCCESS").sum()
        ),
        failed=(
            "status",
            lambda x:
                (x == "FAILED").sum()
        )
    )
    .reset_index()
)


strategy_coverage[
    "success_rate"
] = np.where(
    strategy_coverage[
        "evaluations"
    ] > 0,
    (
        strategy_coverage[
            "successful"
        ]
        /
        strategy_coverage[
            "evaluations"
        ]
        * 100
    ),
    0.0
)


display(
    strategy_coverage
)


# ------------------------------------------------------------
# 10. PERSIST RECOVERED DATAFRAME IN STANDARD NAME
# ------------------------------------------------------------

AIR_LLM_0622_RESULTS_DF = (
    EVALUATION_RESULTS_DF.copy()
)


# Preserve compatibility with downstream cells
AIR_LLM_0621_RESULTS_DF = (
    EVALUATION_RESULTS_DF.copy()
)


# ------------------------------------------------------------
# 11. SAVE STATUS SUMMARY
# ------------------------------------------------------------

STATUS_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_22_evaluation_status_summary.csv"
)


status_summary.to_csv(
    STATUS_SUMMARY_PATH,
    index=False
)


STRATEGY_COVERAGE_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_22_strategy_coverage.csv"
)


strategy_coverage.to_csv(
    STRATEGY_COVERAGE_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. FINAL DIAGNOSTIC
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)

print(
    "06.22 EVALUATION STATUS SUMMARY : COMPLETE"
)

print(
    "=" * 100
)

print(
    f"Results dataframe : {len(AIR_LLM_0622_RESULTS_DF):,} rows"
)

print(
    f"Successful       : {successful:,}"
)

print(
    f"Failed           : {failed:,}"
)

print(
    f"Success rate     : {success_rate:.2f}%"
)

print(
    f"\nStatus summary saved : "
    f"{STATUS_SUMMARY_PATH}"
)

print(
    f"Strategy coverage saved : "
    f"{STRATEGY_COVERAGE_PATH}"
)

print(
    "\nAIR_LLM_0622_RESULTS_DF is available "
    "for subsequent Notebook 06.22 cells."
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 06.22
EVALUATION STATUS SUMMARY

----------------------------------------------------------------------------------------------------
LOADING 06.21 CANDIDATE EVALUATION RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF (in-memory)
Results rows loaded : 5,775
Columns available   : 22


,status,count
0,SUCCESS,5775



Successful runs : 5,775
Failed runs     : 0
Total runs      : 5,775
Success rate    : 100.00%

----------------------------------------------------------------------------------------------------
EVALUATION COMPLETENESS CHECK
----------------------------------------------------------------------------------------------------
Expected evaluations : 5,775
Actual evaluations   : 5,775
Completeness         : PASS
Execution integrity  : PASS

Failure summary : No failed evaluations.

----------------------------------------------------------------------------------------------------
STRATEGY COVERAGE
----------------------------------------------------------------------------------------------------


,strategy_id,evaluations,successful,failed,success_rate
0,mean,1800,1800,0,100.0
1,mode,3975,3975,0,100.0



06.22 EVALUATION STATUS SUMMARY : COMPLETE
Results dataframe : 5,775 rows
Successful       : 5,775
Failed           : 0
Success rate     : 100.00%

Status summary saved : /content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_evaluation_status_summary.csv
Strategy coverage saved : /content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_strategy_coverage.csv

AIR_LLM_0622_RESULTS_DF is available for subsequent Notebook 06.22 cells.


In [30]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.22
# EVALUATION STATUS SUMMARY
# ==================================================================================================
#
# PURPOSE
# -------
# This cell validates and recovers the completed candidate-evaluation results
# produced by Notebook 06.21.
#
# IMPORTANT
# ---------
# 06.21 already completed 5,775/5,775 evaluations successfully.
# This cell DOES NOT rerun any candidate evaluations.
#
# Primary 06.21 dataframe:
#     AIR_LLM_0621_RESULTS_DF
#
# Persistent 06.21 artifact:
#     air_llm_06_21_candidate_results.csv
#
# ==================================================================================================

import os
import pandas as pd
import numpy as np


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.22")
print("EVALUATION STATUS SUMMARY")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL PROJECT PATH
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

FINAL_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_candidate_results.csv"
)

CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_checkpoint.csv"
)

STRATEGY_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_21_strategy_summary.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. RECOVER 06.21 RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("06.21 RESULT RECOVERY")
print("-" * 100)


# Preferred source:
# Notebook 06.21 explicitly creates AIR_LLM_0621_RESULTS_DF.

if (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
    and not AIR_LLM_0621_RESULTS_DF.empty
):

    EVALUATION_RESULTS_DF = (
        AIR_LLM_0621_RESULTS_DF.copy()
    )

    print(
        "Result source : AIR_LLM_0621_RESULTS_DF"
    )

    print(
        "Recovery mode : In-memory"
    )


# If the runtime was restarted, recover from the persistent artifact.
elif os.path.exists(
    FINAL_RESULTS_PATH
):

    print(
        "AIR_LLM_0621_RESULTS_DF is not currently "
        "available in memory."
    )

    print(
        "Recovering the completed 06.21 results "
        "from disk..."
    )

    EVALUATION_RESULTS_DF = pd.read_csv(
        FINAL_RESULTS_PATH
    )

    print(
        "Result source : "
        f"{FINAL_RESULTS_PATH}"
    )

    print(
        "Recovery mode : Persistent CSV"
    )


else:

    raise RuntimeError(
        "\n06.21 candidate evaluation results could not be recovered.\n\n"
        "Expected dataframe:\n"
        "    AIR_LLM_0621_RESULTS_DF\n\n"
        "Expected persistent artifact:\n"
        f"    {FINAL_RESULTS_PATH}\n\n"
        "Do NOT continue to Notebook 06.22 until the 06.21 "
        "results artifact is available."
    )


# --------------------------------------------------------------------------------------------------
# 3. BASIC DATAFRAME VALIDATION
# --------------------------------------------------------------------------------------------------

if not isinstance(
    EVALUATION_RESULTS_DF,
    pd.DataFrame
):

    raise RuntimeError(
        "Recovered 06.21 results are not a pandas DataFrame."
    )


if EVALUATION_RESULTS_DF.empty:

    raise RuntimeError(
        "Recovered 06.21 results dataframe is empty."
    )


print(
    f"\nResults rows loaded : "
    f"{len(EVALUATION_RESULTS_DF):,}"
)

print(
    f"Columns loaded      : "
    f"{len(EVALUATION_RESULTS_DF.columns):,}"
)


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED 06.21 COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "status",
    "strategy_id"
]


missing_columns = [
    column
    for column in required_columns
    if column not in EVALUATION_RESULTS_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "The recovered 06.21 results are missing required "
        f"columns: {missing_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 5. NORMALIZE STATUS
# --------------------------------------------------------------------------------------------------

EVALUATION_RESULTS_DF["status"] = (
    EVALUATION_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# --------------------------------------------------------------------------------------------------
# 6. STATUS DISTRIBUTION
# --------------------------------------------------------------------------------------------------

status_summary = (
    EVALUATION_RESULTS_DF[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)


print("\n" + "-" * 100)
print("EVALUATION STATUS DISTRIBUTION")
print("-" * 100)

display(
    status_summary
)


# --------------------------------------------------------------------------------------------------
# 7. SUCCESS / FAILURE COUNTS
# --------------------------------------------------------------------------------------------------

successful = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "SUCCESS"
    ).sum()
)


failed = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "FAILED"
    ).sum()
)


total_runs = int(
    len(
        EVALUATION_RESULTS_DF
    )
)


other_status = int(
    total_runs
    - successful
    - failed
)


success_rate = (
    100.0 * successful / total_runs
    if total_runs > 0
    else 0.0
)


print(
    f"\nSuccessful evaluations : {successful:,}"
)

print(
    f"Failed evaluations     : {failed:,}"
)

print(
    f"Other statuses         : {other_status:,}"
)

print(
    f"Total evaluations      : {total_runs:,}"
)

print(
    f"Success rate            : {success_rate:.2f}%"
)


# --------------------------------------------------------------------------------------------------
# 8. EXPECTED 06.21 PLAN SIZE VALIDATION
# --------------------------------------------------------------------------------------------------

EXPECTED_0621_EVALUATIONS = 5775


print("\n" + "-" * 100)
print("06.21 COMPLETENESS VALIDATION")
print("-" * 100)

print(
    f"Expected 06.21 evaluations : "
    f"{EXPECTED_0621_EVALUATIONS:,}"
)

print(
    f"Recovered evaluations      : "
    f"{total_runs:,}"
)


if total_runs == EXPECTED_0621_EVALUATIONS:

    print(
        "Plan completeness : PASS"
    )

else:

    print(
        "Plan completeness : WARNING"
    )

    print(
        "The recovered result count differs from "
        "the completed 06.21 plan size."
    )


# --------------------------------------------------------------------------------------------------
# 9. EXECUTION INTEGRITY
# --------------------------------------------------------------------------------------------------

if (
    total_runs == EXPECTED_0621_EVALUATIONS
    and successful == EXPECTED_0621_EVALUATIONS
    and failed == 0
    and other_status == 0
):

    EXECUTION_INTEGRITY = "PASS"

else:

    EXECUTION_INTEGRITY = "WARNING"


print(
    f"Execution integrity : "
    f"{EXECUTION_INTEGRITY}"
)


# --------------------------------------------------------------------------------------------------
# 10. FAILURE SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FAILURE SUMMARY")
print("-" * 100)


if failed > 0:

    failure_summary = (
        EVALUATION_RESULTS_DF[
            EVALUATION_RESULTS_DF[
                "status"
            ] == "FAILED"
        ]
        .groupby(
            "strategy_id",
            dropna=False
        )
        .size()
        .reset_index(
            name="failures"
        )
        .sort_values(
            "failures",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    display(
        failure_summary
    )

else:

    failure_summary = pd.DataFrame(
        columns=[
            "strategy_id",
            "failures"
        ]
    )

    print(
        "No failed evaluations."
    )


# --------------------------------------------------------------------------------------------------
# 11. STRATEGY-LEVEL EXECUTION COVERAGE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("STRATEGY-LEVEL EXECUTION COVERAGE")
print("-" * 100)


strategy_coverage = (
    EVALUATION_RESULTS_DF
    .groupby(
        "strategy_id",
        dropna=False
    )
    .agg(
        evaluations=(
            "strategy_id",
            "size"
        ),
        successful=(
            "status",
            lambda x: int(
                (x == "SUCCESS").sum()
            )
        ),
        failed=(
            "status",
            lambda x: int(
                (x == "FAILED").sum()
            )
        )
    )
    .reset_index()
)


strategy_coverage[
    "success_rate"
] = np.where(
    strategy_coverage[
        "evaluations"
    ] > 0,
    (
        strategy_coverage[
            "successful"
        ]
        /
        strategy_coverage[
            "evaluations"
        ]
        * 100.0
    ),
    0.0
)


display(
    strategy_coverage
)


# --------------------------------------------------------------------------------------------------
# 12. VERIFY SUCCESSFUL CANDIDATE RESULTS
# --------------------------------------------------------------------------------------------------

SUCCESSFUL_RESULTS_DF = (
    EVALUATION_RESULTS_DF[
        EVALUATION_RESULTS_DF[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("\n" + "-" * 100)
print("SUCCESSFUL CANDIDATE RESULTS")
print("-" * 100)

print(
    f"Successful result rows : "
    f"{len(SUCCESSFUL_RESULTS_DF):,}"
)


if SUCCESSFUL_RESULTS_DF.empty:

    raise RuntimeError(
        "No successful candidate evaluations are available "
        "for downstream Notebook 06.22 processing."
    )


# --------------------------------------------------------------------------------------------------
# 13. CREATE STANDARD DOWNSTREAM VARIABLES
# --------------------------------------------------------------------------------------------------
#
# Preserve the official 06.21 variable.
# Also expose the validated successful results for 06.22.
#

AIR_LLM_0621_RESULTS_DF = (
    EVALUATION_RESULTS_DF.copy()
)

AIR_LLM_0622_RESULTS_DF = (
    EVALUATION_RESULTS_DF.copy()
)

AIR_LLM_0622_SUCCESSFUL_RESULTS_DF = (
    SUCCESSFUL_RESULTS_DF.copy()
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE 06.22 STATUS ARTIFACTS
# --------------------------------------------------------------------------------------------------

STATUS_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_22_evaluation_status_summary.csv"
)


STRATEGY_COVERAGE_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_22_strategy_coverage.csv"
)


status_summary.to_csv(
    STATUS_SUMMARY_PATH,
    index=False
)


strategy_coverage.to_csv(
    STRATEGY_COVERAGE_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL 06.22 STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("06.22 EVALUATION STATUS SUMMARY : COMPLETE")
print("=" * 100)

print(
    f"06.21 result rows       : "
    f"{len(AIR_LLM_0621_RESULTS_DF):,}"
)

print(
    f"Successful evaluations  : "
    f"{successful:,}"
)

print(
    f"Failed evaluations      : "
    f"{failed:,}"
)

print(
    f"Success rate            : "
    f"{success_rate:.2f}%"
)

print(
    f"Execution integrity     : "
    f"{EXECUTION_INTEGRITY}"
)

print(
    "\nStatus summary saved:"
)

print(
    STATUS_SUMMARY_PATH
)

print(
    "\nStrategy coverage saved:"
)

print(
    STRATEGY_COVERAGE_PATH
)

print(
    "\nAvailable downstream dataframes:"
)

print(
    "  AIR_LLM_0621_RESULTS_DF"
)

print(
    "  AIR_LLM_0622_RESULTS_DF"
)

print(
    "  AIR_LLM_0622_SUCCESSFUL_RESULTS_DF"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.22
EVALUATION STATUS SUMMARY

----------------------------------------------------------------------------------------------------
06.21 RESULT RECOVERY
----------------------------------------------------------------------------------------------------
Result source : AIR_LLM_0621_RESULTS_DF
Recovery mode : In-memory

Results rows loaded : 5,775
Columns loaded      : 22

----------------------------------------------------------------------------------------------------
EVALUATION STATUS DISTRIBUTION
----------------------------------------------------------------------------------------------------


,status,count
0,SUCCESS,5775



Successful evaluations : 5,775
Failed evaluations     : 0
Other statuses         : 0
Total evaluations      : 5,775
Success rate            : 100.00%

----------------------------------------------------------------------------------------------------
06.21 COMPLETENESS VALIDATION
----------------------------------------------------------------------------------------------------
Expected 06.21 evaluations : 5,775
Recovered evaluations      : 5,775
Plan completeness : PASS
Execution integrity : PASS

----------------------------------------------------------------------------------------------------
FAILURE SUMMARY
----------------------------------------------------------------------------------------------------
No failed evaluations.

----------------------------------------------------------------------------------------------------
STRATEGY-LEVEL EXECUTION COVERAGE
----------------------------------------------------------------------------------------------------


,strategy_id,evaluations,successful,failed,success_rate
0,mean,1800,1800,0,100.0
1,mode,3975,3975,0,100.0



----------------------------------------------------------------------------------------------------
SUCCESSFUL CANDIDATE RESULTS
----------------------------------------------------------------------------------------------------
Successful result rows : 5,775

06.22 EVALUATION STATUS SUMMARY : COMPLETE
06.21 result rows       : 5,775
Successful evaluations  : 5,775
Failed evaluations      : 0
Success rate            : 100.00%
Execution integrity     : PASS

Status summary saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_evaluation_status_summary.csv

Strategy coverage saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_22_strategy_coverage.csv

Available downstream dataframes:
  AIR_LLM_0621_RESULTS_DF
  AIR_LLM_0622_RESULTS_DF
  AIR_LLM_0622_SUCCESSFUL_RESULTS_DF


In [33]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.24
# LLM RECOMMENDATION EVALUATION
# ==================================================================================================
#
# PURPOSE
# -------
# Evaluate the agreement and empirical performance of LLM-recommended strategies
# using the completed candidate evaluations from Notebook 06.21.
#
# IMPORTANT
# ---------
# The evaluation plan contains multiple candidate rows per experimental scenario.
# Therefore:
#
#     31,725 plan rows
#     5,775 unique experimental scenarios
#
# is expected.
#
# The LLM recommendation is a SCENARIO-LEVEL attribute.
#
# This cell:
#
#   1. Recovers 06.21 results.
#   2. Recovers the evaluation plan.
#   3. Creates one validated LLM recommendation per scenario.
#   4. Detects whether 06.21 results already contain llm_recommended.
#   5. Reconciles existing and plan-derived recommendations.
#   6. Computes LLM agreement.
#   7. Compares empirical performance.
#   8. Saves research artifacts.
#
# No imputation/evaluation is rerun.
#
# ==================================================================================================

import os
import pandas as pd
import numpy as np


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.24")
print("LLM RECOMMENDATION EVALUATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL PROJECT ROOT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)


# --------------------------------------------------------------------------------------------------
# 2. RECOVER NOTEBOOK 06.21 RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RECOVERING 06.21 EVALUATION RESULTS")
print("-" * 100)


if (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
    and not AIR_LLM_0621_RESULTS_DF.empty
):

    EVALUATION_RESULTS_DF = (
        AIR_LLM_0621_RESULTS_DF.copy()
    )

    print(
        "Source : AIR_LLM_0621_RESULTS_DF"
    )

else:

    FINAL_RESULTS_PATH = os.path.join(
        RESULTS_DIR,
        "air_llm_06_21_candidate_results.csv"
    )

    if not os.path.exists(
        FINAL_RESULTS_PATH
    ):

        raise FileNotFoundError(
            "\n06.21 candidate results were not found.\n"
            f"Expected:\n{FINAL_RESULTS_PATH}"
        )

    EVALUATION_RESULTS_DF = pd.read_csv(
        FINAL_RESULTS_PATH
    )

    print(
        "Source : persisted 06.21 candidate results"
    )

    print(
        f"Path   : {FINAL_RESULTS_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 3. RECOVER NOTEBOOK 06 EVALUATION PLAN
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RECOVERING EVALUATION PLAN")
print("-" * 100)


if (
    "EVALUATION_PLAN_DF" in globals()
    and isinstance(
        EVALUATION_PLAN_DF,
        pd.DataFrame
    )
    and not EVALUATION_PLAN_DF.empty
):

    PLAN_DF = (
        EVALUATION_PLAN_DF.copy()
    )

    print(
        "Source : EVALUATION_PLAN_DF"
    )

else:

    possible_plan_paths = [
        os.path.join(
            RESULTS_DIR,
            "air_llm_06_21_bounded_plan.csv"
        ),
        os.path.join(
            RESULTS_DIR,
            "air_llm_06_21_evaluation_plan.csv"
        ),
        os.path.join(
            RESULTS_DIR,
            "air_llm_06_20_evaluation_plan.csv"
        ),
        os.path.join(
            RESULTS_DIR,
            "air_llm_evaluation_plan.csv"
        )
    ]

    existing_plan_paths = [
        path
        for path in possible_plan_paths
        if os.path.exists(path)
    ]

    if len(existing_plan_paths) == 0:

        raise FileNotFoundError(
            "\nEvaluation plan could not be recovered.\n"
            "No known persisted evaluation-plan artifact was found."
        )

    PLAN_PATH = existing_plan_paths[0]

    PLAN_DF = pd.read_csv(
        PLAN_PATH
    )

    print(
        "Source : persisted evaluation plan"
    )

    print(
        f"Path   : {PLAN_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATING INPUT SCHEMA")
print("-" * 100)


RESULT_REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed",
    "strategy_id",
    "status"
]


PLAN_REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed",
    "llm_recommended"
]


missing_result_columns = [
    column
    for column in RESULT_REQUIRED_COLUMNS
    if column not in EVALUATION_RESULTS_DF.columns
]


missing_plan_columns = [
    column
    for column in PLAN_REQUIRED_COLUMNS
    if column not in PLAN_DF.columns
]


if missing_result_columns:

    raise RuntimeError(
        "06.21 results are missing required columns:\n"
        f"{missing_result_columns}"
    )


if missing_plan_columns:

    raise RuntimeError(
        "Evaluation plan is missing required columns:\n"
        f"{missing_plan_columns}"
    )


print(
    "06.21 result schema : PASS"
)

print(
    "Evaluation plan     : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5. SCENARIO KEY
# --------------------------------------------------------------------------------------------------

SCENARIO_KEYS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"
]


# --------------------------------------------------------------------------------------------------
# 6. NORMALIZE SCENARIO IDENTIFIERS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("NORMALIZING SCENARIO KEYS")
print("-" * 100)


for column in [
    "dataset_id",
    "feature",
    "missingness_mechanism"
]:

    EVALUATION_RESULTS_DF[column] = (
        EVALUATION_RESULTS_DF[column]
        .astype(str)
        .str.strip()
    )

    PLAN_DF[column] = (
        PLAN_DF[column]
        .astype(str)
        .str.strip()
    )


for dataframe in [
    EVALUATION_RESULTS_DF,
    PLAN_DF
]:

    dataframe["missingness_rate"] = pd.to_numeric(
        dataframe["missingness_rate"],
        errors="coerce"
    )

    dataframe["seed"] = pd.to_numeric(
        dataframe["seed"],
        errors="coerce"
    )


PLAN_DF["llm_recommended"] = (
    PLAN_DF["llm_recommended"]
    .astype(str)
    .str.strip()
)


# --------------------------------------------------------------------------------------------------
# 7. ANALYSE PLAN STRUCTURE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("ANALYSING SCENARIO-LEVEL PLAN STRUCTURE")
print("-" * 100)


scenario_counts = (
    PLAN_DF
    .groupby(
        SCENARIO_KEYS,
        dropna=False
    )
    .size()
)


print(
    f"Plan rows              : {len(PLAN_DF):,}"
)

print(
    f"Unique scenarios       : {len(scenario_counts):,}"
)

print(
    f"Maximum rows/scenario  : {scenario_counts.max():,}"
)

repeated_scenarios = int(
    (scenario_counts > 1).sum()
)


print(
    f"Repeated scenarios     : {repeated_scenarios:,}"
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY LLM RECOMMENDATION CONSISTENCY
# --------------------------------------------------------------------------------------------------

recommendation_counts = (
    PLAN_DF
    .groupby(
        SCENARIO_KEYS,
        dropna=False
    )[
        "llm_recommended"
    ]
    .nunique(
        dropna=False
    )
)


ambiguous_scenarios = (
    recommendation_counts[
        recommendation_counts > 1
    ]
)


print(
    f"Ambiguous LLM scenarios : "
    f"{len(ambiguous_scenarios):,}"
)


if len(ambiguous_scenarios) > 0:

    raise RuntimeError(
        "The evaluation plan contains multiple different "
        "LLM recommendations for the same experimental scenario."
    )


print(
    "LLM recommendation consistency : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9. CREATE UNIQUE SCENARIO-LEVEL LLM TABLE
# --------------------------------------------------------------------------------------------------

LLM_SCENARIO_DF = (
    PLAN_DF[
        SCENARIO_KEYS + [
            "llm_recommended"
        ]
    ]
    .drop_duplicates(
        subset=SCENARIO_KEYS
    )
    .reset_index(
        drop=True
    )
)


print(
    f"\nUnique LLM recommendation scenarios : "
    f"{len(LLM_SCENARIO_DF):,}"
)


# --------------------------------------------------------------------------------------------------
# 10. FILTER SUCCESSFUL 06.21 RESULTS
# --------------------------------------------------------------------------------------------------

EVALUATION_RESULTS_DF["status"] = (
    EVALUATION_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


SUCCESSFUL_RESULTS_DF = (
    EVALUATION_RESULTS_DF[
        EVALUATION_RESULTS_DF["status"] == "SUCCESS"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    f"Successful empirical evaluations : "
    f"{len(SUCCESSFUL_RESULTS_DF):,}"
)


if SUCCESSFUL_RESULTS_DF.empty:

    raise RuntimeError(
        "No successful 06.21 empirical evaluations are available."
    )


# --------------------------------------------------------------------------------------------------
# 11. DETECT WHETHER RESULTS ALREADY CONTAIN LLM RECOMMENDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CHECKING EXISTING LLM RECOMMENDATION IN 06.21 RESULTS")
print("-" * 100)


RESULTS_ALREADY_HAVE_RECOMMENDATION = (
    "llm_recommended"
    in SUCCESSFUL_RESULTS_DF.columns
)


if RESULTS_ALREADY_HAVE_RECOMMENDATION:

    print(
        "06.21 results already contain "
        "'llm_recommended'."
    )

    print(
        "Existing recommendations will be reconciled "
        "against the validated evaluation plan."
    )

else:

    print(
        "06.21 results do not contain "
        "'llm_recommended'."
    )

    print(
        "Recommendation will be attached from "
        "the validated scenario-level plan."
    )


# --------------------------------------------------------------------------------------------------
# 12. MERGE PLAN RECOMMENDATION SAFELY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("ATTACHING / RECONCILING LLM RECOMMENDATIONS")
print("-" * 100)


if RESULTS_ALREADY_HAVE_RECOMMENDATION:

    # Normalize existing recommendation.
    SUCCESSFUL_RESULTS_DF[
        "llm_recommended_existing"
    ] = (
        SUCCESSFUL_RESULTS_DF[
            "llm_recommended"
        ]
        .astype(str)
        .str.strip()
    )

    # Attach the independently validated plan recommendation
    # under a different name so nothing is silently overwritten.

    LLM_EVALUATION_DF = (
        SUCCESSFUL_RESULTS_DF
        .merge(
            LLM_SCENARIO_DF.rename(
                columns={
                    "llm_recommended":
                        "llm_recommended_plan"
                }
            ),
            on=SCENARIO_KEYS,
            how="left",
            validate="many_to_one"
        )
    )

    # Compare existing result recommendation against plan recommendation.
    existing_valid = (
        ~LLM_EVALUATION_DF[
            "llm_recommended_existing"
        ]
        .str.lower()
        .isin(
            [
                "",
                "nan",
                "none",
                "null"
            ]
        )
    )

    plan_valid = (
        ~LLM_EVALUATION_DF[
            "llm_recommended_plan"
        ]
        .isna()
    )

    reconciliation_mask = (
        existing_valid
        &
        plan_valid
    )

    recommendation_disagreement = int(
        (
            LLM_EVALUATION_DF.loc[
                reconciliation_mask,
                "llm_recommended_existing"
            ]
            !=
            LLM_EVALUATION_DF.loc[
                reconciliation_mask,
                "llm_recommended_plan"
            ]
        ).sum()
    )

    print(
        f"Recommendation disagreements : "
        f"{recommendation_disagreement:,}"
    )

    if recommendation_disagreement > 0:

        raise RuntimeError(
            "The LLM recommendation stored in the 06.21 "
            "results differs from the validated evaluation-plan "
            "recommendation."
        )

    # Use the validated plan value as the canonical value.
    LLM_EVALUATION_DF[
        "llm_recommended"
    ] = (
        LLM_EVALUATION_DF[
            "llm_recommended_plan"
        ]
    )

    print(
        "Existing vs plan recommendation : PASS"
    )

else:

    LLM_EVALUATION_DF = (
        SUCCESSFUL_RESULTS_DF
        .merge(
            LLM_SCENARIO_DF,
            on=SCENARIO_KEYS,
            how="left",
            validate="many_to_one"
        )
    )

    print(
        "Plan recommendation attached : PASS"
    )


# --------------------------------------------------------------------------------------------------
# 13. VERIFY ATTACHED RECOMMENDATION
# --------------------------------------------------------------------------------------------------

if (
    "llm_recommended"
    not in LLM_EVALUATION_DF.columns
):

    raise RuntimeError(
        "The LLM recommendation column could not be established."
    )


missing_recommendation_mask = (
    LLM_EVALUATION_DF[
        "llm_recommended"
    ]
    .isna()
    |
    LLM_EVALUATION_DF[
        "llm_recommended"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        [
            "",
            "nan",
            "none",
            "null"
        ]
    )
)


missing_recommendations = int(
    missing_recommendation_mask.sum()
)


available_recommendations = int(
    len(LLM_EVALUATION_DF)
    -
    missing_recommendations
)


recommendation_coverage = (
    available_recommendations
    /
    len(LLM_EVALUATION_DF)
    if len(LLM_EVALUATION_DF) > 0
    else np.nan
)


print(
    f"Empirical result rows     : "
    f"{len(LLM_EVALUATION_DF):,}"
)

print(
    f"LLM recommendations found : "
    f"{available_recommendations:,}"
)

print(
    f"Missing recommendations   : "
    f"{missing_recommendations:,}"
)

print(
    f"Recommendation coverage   : "
    f"{recommendation_coverage:.4%}"
)


if (
    available_recommendations == 0
):

    raise RuntimeError(
        "No valid LLM recommendations are available "
        "for empirical evaluation."
    )


# --------------------------------------------------------------------------------------------------
# 14. COMPUTE LLM STRATEGY AGREEMENT
# --------------------------------------------------------------------------------------------------

LLM_EVALUATION_DF[
    "llm_match"
] = np.nan


valid_mask = (
    ~missing_recommendation_mask
)


LLM_EVALUATION_DF.loc[
    valid_mask,
    "llm_match"
] = (
    LLM_EVALUATION_DF.loc[
        valid_mask,
        "strategy_id"
    ]
    .astype(str)
    .str.strip()
    ==
    LLM_EVALUATION_DF.loc[
        valid_mask,
        "llm_recommended"
    ]
    .astype(str)
    .str.strip()
)


llm_matches = int(
    (
        LLM_EVALUATION_DF[
            "llm_match"
        ]
        == True
    ).sum()
)


llm_non_matches = int(
    (
        LLM_EVALUATION_DF[
            "llm_match"
        ]
        == False
    ).sum()
)


LLM_MATCH_RATE = (
    llm_matches
    /
    available_recommendations
)


print("\n" + "-" * 100)
print("LLM RECOMMENDATION AGREEMENT")
print("-" * 100)

print(
    f"Valid recommendation rows : "
    f"{available_recommendations:,}"
)

print(
    f"LLM matches                : "
    f"{llm_matches:,}"
)

print(
    f"LLM non-matches            : "
    f"{llm_non_matches:,}"
)

print(
    f"LLM agreement rate         : "
    f"{LLM_MATCH_RATE:.4f}"
)

print(
    f"LLM agreement percentage   : "
    f"{LLM_MATCH_RATE * 100:.2f}%"
)


# --------------------------------------------------------------------------------------------------
# 15. EMPIRICAL PERFORMANCE COMPARISON
# --------------------------------------------------------------------------------------------------

PERFORMANCE_COLUMNS = [
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "wasserstein",
    "js_divergence",
    "runtime_seconds"
]


AVAILABLE_PERFORMANCE_COLUMNS = [
    column
    for column in PERFORMANCE_COLUMNS
    if column in LLM_EVALUATION_DF.columns
]


print("\n" + "-" * 100)
print("EMPIRICAL PERFORMANCE BY LLM AGREEMENT")
print("-" * 100)


if len(
    AVAILABLE_PERFORMANCE_COLUMNS
) > 0:

    LLM_PERFORMANCE_DF = (
        LLM_EVALUATION_DF[
            valid_mask
        ]
        .groupby(
            "llm_match"
        )[
            AVAILABLE_PERFORMANCE_COLUMNS
        ]
        .mean(
            numeric_only=True
        )
    )

    display(
        LLM_PERFORMANCE_DF
    )

else:

    LLM_PERFORMANCE_DF = pd.DataFrame()

    print(
        "No compatible performance columns were found."
    )


# --------------------------------------------------------------------------------------------------
# 16. LLM RECOMMENDATION DISTRIBUTION
# --------------------------------------------------------------------------------------------------

LLM_RECOMMENDATION_DISTRIBUTION_DF = (
    LLM_EVALUATION_DF[
        valid_mask
    ][
        "llm_recommended"
    ]
    .value_counts()
    .rename_axis(
        "llm_recommended"
    )
    .reset_index(
        name="count"
    )
)


print("\n" + "-" * 100)
print("LLM RECOMMENDATION DISTRIBUTION")
print("-" * 100)

display(
    LLM_RECOMMENDATION_DISTRIBUTION_DF
)


# --------------------------------------------------------------------------------------------------
# 17. DATASET-LEVEL AGREEMENT
# --------------------------------------------------------------------------------------------------

DATASET_LLM_AGREEMENT_DF = (
    LLM_EVALUATION_DF[
        valid_mask
    ]
    .groupby(
        "dataset_id",
        dropna=False
    )
    .agg(
        evaluations=(
            "llm_match",
            "size"
        ),
        matches=(
            "llm_match",
            lambda x:
                int(
                    (x == True).sum()
                )
        )
    )
    .reset_index()
)


DATASET_LLM_AGREEMENT_DF[
    "agreement_rate"
] = (
    DATASET_LLM_AGREEMENT_DF[
        "matches"
    ]
    /
    DATASET_LLM_AGREEMENT_DF[
        "evaluations"
    ]
)


print("\n" + "-" * 100)
print("DATASET-LEVEL LLM AGREEMENT")
print("-" * 100)

display(
    DATASET_LLM_AGREEMENT_DF
)


# --------------------------------------------------------------------------------------------------
# 18. MISSINGNESS-MECHANISM AGREEMENT
# --------------------------------------------------------------------------------------------------

MECHANISM_LLM_AGREEMENT_DF = (
    LLM_EVALUATION_DF[
        valid_mask
    ]
    .groupby(
        "missingness_mechanism",
        dropna=False
    )
    .agg(
        evaluations=(
            "llm_match",
            "size"
        ),
        matches=(
            "llm_match",
            lambda x:
                int(
                    (x == True).sum()
                )
        )
    )
    .reset_index()
)


MECHANISM_LLM_AGREEMENT_DF[
    "agreement_rate"
] = (
    MECHANISM_LLM_AGREEMENT_DF[
        "matches"
    ]
    /
    MECHANISM_LLM_AGREEMENT_DF[
        "evaluations"
    ]
)


print("\n" + "-" * 100)
print("MISSINGNESS-MECHANISM LLM AGREEMENT")
print("-" * 100)

display(
    MECHANISM_LLM_AGREEMENT_DF
)


# --------------------------------------------------------------------------------------------------
# 19. SAVE PRIMARY RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SAVING 06.24 RESULTS")
print("-" * 100)


LLM_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_24_llm_recommendation_evaluation.csv"
)


LLM_PERFORMANCE_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_24_llm_recommendation_performance.csv"
)


LLM_DISTRIBUTION_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_24_llm_recommendation_distribution.csv"
)


DATASET_AGREEMENT_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_24_dataset_llm_agreement.csv"
)


MECHANISM_AGREEMENT_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_24_mechanism_llm_agreement.csv"
)


LLM_EVALUATION_DF.to_csv(
    LLM_RESULTS_PATH,
    index=False
)


LLM_PERFORMANCE_DF.to_csv(
    LLM_PERFORMANCE_PATH
)


LLM_RECOMMENDATION_DISTRIBUTION_DF.to_csv(
    LLM_DISTRIBUTION_PATH,
    index=False
)


DATASET_LLM_AGREEMENT_DF.to_csv(
    DATASET_AGREEMENT_PATH,
    index=False
)


MECHANISM_LLM_AGREEMENT_DF.to_csv(
    MECHANISM_AGREEMENT_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 20. DOWNSTREAM VARIABLES
# --------------------------------------------------------------------------------------------------

AIR_LLM_0624_EVALUATION_DF = (
    LLM_EVALUATION_DF.copy()
)

AIR_LLM_0624_PERFORMANCE_DF = (
    LLM_PERFORMANCE_DF.copy()
)

AIR_LLM_0624_RECOMMENDATION_DISTRIBUTION_DF = (
    LLM_RECOMMENDATION_DISTRIBUTION_DF.copy()
)

AIR_LLM_0624_DATASET_AGREEMENT_DF = (
    DATASET_LLM_AGREEMENT_DF.copy()
)

AIR_LLM_0624_MECHANISM_AGREEMENT_DF = (
    MECHANISM_LLM_AGREEMENT_DF.copy()
)


# --------------------------------------------------------------------------------------------------
# 21. FINAL DIAGNOSTIC
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("06.24 LLM RECOMMENDATION EVALUATION : COMPLETE")
print("=" * 100)

print(
    f"06.21 successful rows       : "
    f"{len(SUCCESSFUL_RESULTS_DF):,}"
)

print(
    f"LLM recommendation coverage : "
    f"{recommendation_coverage * 100:.2f}%"
)

print(
    f"LLM agreement rate          : "
    f"{LLM_MATCH_RATE:.4f}"
)

print(
    f"LLM agreement percentage    : "
    f"{LLM_MATCH_RATE * 100:.2f}%"
)

print(
    "\nPrimary result saved:"
)

print(
    LLM_RESULTS_PATH
)

print(
    "\nDownstream dataframe:"
)

print(
    "AIR_LLM_0624_EVALUATION_DF"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.24
LLM RECOMMENDATION EVALUATION

----------------------------------------------------------------------------------------------------
RECOVERING 06.21 EVALUATION RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF

----------------------------------------------------------------------------------------------------
RECOVERING EVALUATION PLAN
----------------------------------------------------------------------------------------------------
Source : EVALUATION_PLAN_DF

----------------------------------------------------------------------------------------------------
VALIDATING INPUT SCHEMA
----------------------------------------------------------------------------------------------------
06.21 result schema : PASS
Evaluation plan     : PASS

----------------------------------------------------------------------------------------------------
NORMALIZING SCENARIO KEYS
------

,mae,rmse,r2,accuracy,macro_f1,weighted_f1,runtime_seconds
llm_match,,,,,,,
False,5714.26722,7376.615804,-4.0544,0.612512,0.292519,0.559609,0.009951



----------------------------------------------------------------------------------------------------
LLM RECOMMENDATION DISTRIBUTION
----------------------------------------------------------------------------------------------------


,llm_recommended,count
0,False,5775



----------------------------------------------------------------------------------------------------
DATASET-LEVEL LLM AGREEMENT
----------------------------------------------------------------------------------------------------


,dataset_id,evaluations,matches,agreement_rate
0,adult_income,1050,0,0.0
1,bank_marketing,1200,0,0.0
2,diabetes_130us,3525,0,0.0



----------------------------------------------------------------------------------------------------
MISSINGNESS-MECHANISM LLM AGREEMENT
----------------------------------------------------------------------------------------------------


,missingness_mechanism,evaluations,matches,agreement_rate
0,MAR,1925,0,0.0
1,MCAR,1925,0,0.0
2,MNAR,1925,0,0.0



----------------------------------------------------------------------------------------------------
SAVING 06.24 RESULTS
----------------------------------------------------------------------------------------------------

06.24 LLM RECOMMENDATION EVALUATION : COMPLETE
06.21 successful rows       : 5,775
LLM recommendation coverage : 100.00%
LLM agreement rate          : 0.0000
LLM agreement percentage    : 0.00%

Primary result saved:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_24_llm_recommendation_evaluation.csv

Downstream dataframe:
AIR_LLM_0624_EVALUATION_DF


In [34]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.25
# STATISTICAL SIGNIFICANCE ANALYSIS
# ==================================================================================================
#
# PURPOSE
# -------
# Test whether differences among candidate imputation strategies are statistically
# significant across matched experimental cases.
#
# Primary test:
#     Friedman test
#
# Experimental unit:
#     dataset × feature × missingness mechanism × missingness rate × repetition/seed
#
# The analysis uses only SUCCESSFUL empirical evaluations from Notebook 06.21.
#
# IMPORTANT
# ---------
# Notebook 06.21 contains the completed candidate evaluation results:
#
#     AIR_LLM_0621_RESULTS_DF
#
# This cell does not rerun any experiments.
#
# ==================================================================================================


import os
import numpy as np
import pandas as pd

from scipy.stats import friedmanchisquare


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.25")
print("STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL PROJECT ROOT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)


# --------------------------------------------------------------------------------------------------
# 2. SIGNIFICANCE LEVEL
# --------------------------------------------------------------------------------------------------

SIGNIFICANCE_ALPHA = 0.05


# --------------------------------------------------------------------------------------------------
# 3. RECOVER NOTEBOOK 06.21 RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RECOVERING 06.21 EMPIRICAL RESULTS")
print("-" * 100)


if (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
    and not AIR_LLM_0621_RESULTS_DF.empty
):

    SUCCESS_RESULTS_DF = (
        AIR_LLM_0621_RESULTS_DF.copy()
    )

    print(
        "Source : AIR_LLM_0621_RESULTS_DF"
    )

else:

    RESULTS_PATH = os.path.join(
        RESULTS_DIR,
        "air_llm_06_21_candidate_results.csv"
    )

    if not os.path.exists(
        RESULTS_PATH
    ):

        raise FileNotFoundError(
            "\n06.21 empirical results could not be recovered.\n"
            f"Expected:\n{RESULTS_PATH}"
        )

    SUCCESS_RESULTS_DF = pd.read_csv(
        RESULTS_PATH
    )

    print(
        "Source : persisted 06.21 candidate results"
    )

    print(
        f"Path   : {RESULTS_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATING 06.21 RESULT SCHEMA")
print("-" * 100)


REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed",
    "strategy_id",
    "status"
]


missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in SUCCESS_RESULTS_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "06.21 results are missing required columns:\n"
        f"{missing_columns}"
    )


print(
    "06.21 result schema : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5. FILTER SUCCESSFUL EVALUATIONS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FILTERING SUCCESSFUL EVALUATIONS")
print("-" * 100)


SUCCESS_RESULTS_DF["status"] = (
    SUCCESS_RESULTS_DF["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


SUCCESS_RESULTS_DF = (
    SUCCESS_RESULTS_DF[
        SUCCESS_RESULTS_DF["status"] == "SUCCESS"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Successful evaluations : "
    f"{len(SUCCESS_RESULTS_DF):,}"
)


if SUCCESS_RESULTS_DF.empty:

    raise RuntimeError(
        "No successful empirical evaluations are available."
    )


# --------------------------------------------------------------------------------------------------
# 6. NORMALIZE IDENTIFIERS
# --------------------------------------------------------------------------------------------------

for column in [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "strategy_id"
]:

    SUCCESS_RESULTS_DF[column] = (
        SUCCESS_RESULTS_DF[column]
        .astype(str)
        .str.strip()
    )


SUCCESS_RESULTS_DF[
    "missingness_rate"
] = pd.to_numeric(
    SUCCESS_RESULTS_DF[
        "missingness_rate"
    ],
    errors="coerce"
)


SUCCESS_RESULTS_DF[
    "seed"
] = pd.to_numeric(
    SUCCESS_RESULTS_DF[
        "seed"
    ],
    errors="coerce"
)


# --------------------------------------------------------------------------------------------------
# 7. EXPERIMENTAL MATCHING KEY
# --------------------------------------------------------------------------------------------------

MATCH_KEYS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"
]


# --------------------------------------------------------------------------------------------------
# 8. METRIC CONFIGURATION
# --------------------------------------------------------------------------------------------------

METRIC_DIRECTION = {

    "mae": "lower",

    "rmse": "lower",

    "r2": "higher",

    "accuracy": "higher",

    "macro_f1": "higher",

    "weighted_f1": "higher",

    "wasserstein": "lower",

    "js_divergence": "lower"

}


# --------------------------------------------------------------------------------------------------
# 9. AVAILABLE STRATEGIES
# --------------------------------------------------------------------------------------------------

AVAILABLE_STRATEGIES = sorted(
    SUCCESS_RESULTS_DF[
        "strategy_id"
    ]
    .dropna()
    .unique()
    .tolist()
)


print("\n" + "-" * 100)
print("EVALUATION DESIGN")
print("-" * 100)

print(
    f"Successful rows       : "
    f"{len(SUCCESS_RESULTS_DF):,}"
)

print(
    f"Candidate strategies  : "
    f"{len(AVAILABLE_STRATEGIES)}"
)

print(
    f"Strategies             : "
    f"{', '.join(AVAILABLE_STRATEGIES)}"
)


# --------------------------------------------------------------------------------------------------
# 10. CHECK DUPLICATE STRATEGY / CASE COMBINATIONS
# --------------------------------------------------------------------------------------------------

DUPLICATE_KEYS = (
    MATCH_KEYS
    +
    ["strategy_id"]
)


duplicate_count = int(
    SUCCESS_RESULTS_DF.duplicated(
        subset=DUPLICATE_KEYS
    ).sum()
)


print(
    f"\nDuplicate strategy-case rows : "
    f"{duplicate_count:,}"
)


# If duplicates exist, aggregate them deterministically.
#
# This protects the statistical analysis from accidental duplicate
# records without altering the completed empirical results.

if duplicate_count > 0:

    print(
        "Duplicate rows detected."
    )

    print(
        "Aggregating duplicate strategy-case observations by mean."
    )


# --------------------------------------------------------------------------------------------------
# 11. STATISTICAL SIGNIFICANCE ANALYSIS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FRIEDMAN STATISTICAL SIGNIFICANCE ANALYSIS")
print("-" * 100)


significance_rows = []


for metric, direction in METRIC_DIRECTION.items():

    print(
        f"\nAnalyzing metric : {metric}"
    )


    # ----------------------------------------------------------------------------------------------
    # Check metric availability
    # ----------------------------------------------------------------------------------------------

    if metric not in SUCCESS_RESULTS_DF.columns:

        print(
            f"  SKIPPED — column '{metric}' not available."
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Extract relevant columns
    # ----------------------------------------------------------------------------------------------

    metric_df = (
        SUCCESS_RESULTS_DF[
            MATCH_KEYS
            +
            [
                "strategy_id",
                metric
            ]
        ]
        .copy()
    )


    # Convert metric to numeric.
    metric_df[metric] = pd.to_numeric(
        metric_df[metric],
        errors="coerce"
    )


    metric_df = metric_df.dropna(
        subset=[metric]
    )


    if metric_df.empty:

        print(
            "  SKIPPED — no valid metric observations."
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Aggregate accidental duplicate strategy-case observations
    # ----------------------------------------------------------------------------------------------

    metric_df = (
        metric_df
        .groupby(
            MATCH_KEYS
            +
            ["strategy_id"],
            as_index=False
        )[metric]
        .mean()
    )


    # ----------------------------------------------------------------------------------------------
    # Identify strategies having observations
    # ----------------------------------------------------------------------------------------------

    strategy_counts = (
        metric_df[
            "strategy_id"
        ]
        .value_counts()
    )


    strategies = sorted(
        strategy_counts.index.tolist()
    )


    if len(strategies) < 2:

        print(
            "  SKIPPED — fewer than two strategies available."
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Create matched strategy × experimental-case matrix
    # ----------------------------------------------------------------------------------------------

    pivot = (
        metric_df
        .pivot(
            index=MATCH_KEYS,
            columns="strategy_id",
            values=metric
        )
    )


    # Keep only strategies that are actually present.
    strategies = [
        strategy
        for strategy in strategies
        if strategy in pivot.columns
    ]


    if len(strategies) < 2:

        print(
            "  SKIPPED — fewer than two common strategies."
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Complete-case matching
    # ----------------------------------------------------------------------------------------------

    matched_pivot = (
        pivot[
            strategies
        ]
        .dropna(
            axis=0,
            how="any"
        )
    )


    n_matched_cases = (
        len(matched_pivot)
    )


    if n_matched_cases < 2:

        print(
            f"  SKIPPED — only "
            f"{n_matched_cases} matched cases."
        )

        continue


    # ----------------------------------------------------------------------------------------------
    # Friedman test
    # ----------------------------------------------------------------------------------------------

    arrays = [
        matched_pivot[
            strategy
        ].to_numpy(
            dtype=float
        )
        for strategy in strategies
    ]


    try:

        statistic, p_value = (
            friedmanchisquare(
                *arrays
            )
        )

        statistic = float(
            statistic
        )

        p_value = float(
            p_value
        )

    except Exception as exc:

        print(
            f"  Friedman test failed: {exc}"
        )

        statistic = np.nan
        p_value = np.nan


    # ----------------------------------------------------------------------------------------------
    # Significance decision
    # ----------------------------------------------------------------------------------------------

    significant = (
        bool(
            p_value < SIGNIFICANCE_ALPHA
        )
        if np.isfinite(p_value)
        else False
    )


    significance_rows.append({

        "metric":
            metric,

        "direction":
            direction,

        "n_strategies":
            len(strategies),

        "n_matched_cases":
            n_matched_cases,

        "friedman_statistic":
            statistic,

        "p_value":
            p_value,

        "alpha":
            SIGNIFICANCE_ALPHA,

        "significant_at_0.05":
            significant

    })


    print(
        f"  Strategies       : {len(strategies)}"
    )

    print(
        f"  Matched cases    : {n_matched_cases:,}"
    )

    print(
        f"  Friedman statistic : {statistic:.6f}"
        if np.isfinite(statistic)
        else
        "  Friedman statistic : NaN"
    )

    print(
        f"  p-value            : {p_value:.6g}"
        if np.isfinite(p_value)
        else
        "  p-value            : NaN"
    )

    print(
        f"  Significant (α=0.05): "
        f"{'YES' if significant else 'NO'}"
    )


# --------------------------------------------------------------------------------------------------
# 12. CREATE SIGNIFICANCE RESULTS TABLE
# --------------------------------------------------------------------------------------------------

STATISTICAL_SIGNIFICANCE_DF = (
    pd.DataFrame(
        significance_rows
    )
)


print("\n" + "=" * 100)
print("STATISTICAL SIGNIFICANCE SUMMARY")
print("=" * 100)


if STATISTICAL_SIGNIFICANCE_DF.empty:

    print(
        "No valid statistical significance tests were produced."
    )

else:

    display(
        STATISTICAL_SIGNIFICANCE_DF
    )


# --------------------------------------------------------------------------------------------------
# 13. SUMMARY COUNTS
# --------------------------------------------------------------------------------------------------

if not STATISTICAL_SIGNIFICANCE_DF.empty:

    total_tests = len(
        STATISTICAL_SIGNIFICANCE_DF
    )

    significant_tests = int(
        STATISTICAL_SIGNIFICANCE_DF[
            "significant_at_0.05"
        ].sum()
    )

    print(
        f"\nMetrics tested       : "
        f"{total_tests}"
    )

    print(
        f"Significant metrics  : "
        f"{significant_tests}"
    )

    print(
        f"Non-significant      : "
        f"{total_tests - significant_tests}"
    )


# --------------------------------------------------------------------------------------------------
# 14. SAVE RESULTS
# --------------------------------------------------------------------------------------------------

SIG_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_25_statistical_significance.csv"
)


STATISTICAL_SIGNIFICANCE_DF.to_csv(
    SIG_PATH,
    index=False
)


print(
    f"\nStatistical significance results saved to:"
)

print(
    SIG_PATH
)


# --------------------------------------------------------------------------------------------------
# 15. DOWNSTREAM VARIABLE
# --------------------------------------------------------------------------------------------------

AIR_LLM_0625_STATISTICAL_SIGNIFICANCE_DF = (
    STATISTICAL_SIGNIFICANCE_DF.copy()
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("06.25 STATISTICAL SIGNIFICANCE ANALYSIS : COMPLETE")
print("=" * 100)

print(
    "Downstream dataframe:"
)

print(
    "AIR_LLM_0625_STATISTICAL_SIGNIFICANCE_DF"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.25
STATISTICAL SIGNIFICANCE ANALYSIS

----------------------------------------------------------------------------------------------------
RECOVERING 06.21 EMPIRICAL RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF

----------------------------------------------------------------------------------------------------
VALIDATING 06.21 RESULT SCHEMA
----------------------------------------------------------------------------------------------------
06.21 result schema : PASS

----------------------------------------------------------------------------------------------------
FILTERING SUCCESSFUL EVALUATIONS
----------------------------------------------------------------------------------------------------
Successful evaluations : 5,775

----------------------------------------------------------------------------------------------------
EVALUATION DESIGN
----------------------

In [35]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.26
# ADAPTIVE UTILITY-BASED SELECTION
# ==================================================================================================
#
# PURPOSE
# -------
# Select the empirically strongest imputation strategy for each experimental context
# using a multi-criteria adaptive utility function.
#
# Experimental context:
#
#   C = (dataset, feature, missingness mechanism, missingness rate, seed)
#
# Utility dimensions:
#
#   1. Reconstruction
#   2. Distributional fidelity
#   3. Dependency preservation
#   4. Downstream predictive utility
#   5. Computational efficiency
#
# IMPORTANT
# ---------
# Utility normalization is performed WITHIN EACH EXPERIMENTAL CONTEXT.
#
# This prevents results from one dataset/feature/scenario from determining the
# utility scale of another scenario.
#
# ==================================================================================================


import os
import numpy as np
import pandas as pd


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.26")
print("ADAPTIVE UTILITY-BASED SELECTION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL PROJECT ROOT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)


# --------------------------------------------------------------------------------------------------
# 2. UTILITY WEIGHTS
# --------------------------------------------------------------------------------------------------
#
# These are the predefined AIR-LLM research weights.
#
# They sum to 1.00.
#
# --------------------------------------------------------------------------------------------------

UTILITY_WEIGHTS = {
    "reconstruction": 0.30,
    "distribution": 0.20,
    "dependency": 0.15,
    "downstream": 0.20,
    "efficiency": 0.15
}


WEIGHT_SUM = sum(
    UTILITY_WEIGHTS.values()
)


if not np.isclose(
    WEIGHT_SUM,
    1.0
):

    raise RuntimeError(
        "Utility weights must sum to 1.0."
    )


# --------------------------------------------------------------------------------------------------
# 3. RECOVER SUCCESSFUL 06.21 RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RECOVERING NOTEBOOK 06.21 EMPIRICAL RESULTS")
print("-" * 100)


if (
    "AIR_LLM_0621_RESULTS_DF" in globals()
    and isinstance(
        AIR_LLM_0621_RESULTS_DF,
        pd.DataFrame
    )
    and not AIR_LLM_0621_RESULTS_DF.empty
):

    UTILITY_SOURCE_DF = (
        AIR_LLM_0621_RESULTS_DF.copy()
    )

    print(
        "Source : AIR_LLM_0621_RESULTS_DF"
    )

else:

    RESULTS_PATH = os.path.join(
        RESULTS_DIR,
        "air_llm_06_21_candidate_results.csv"
    )

    if not os.path.exists(
        RESULTS_PATH
    ):

        raise FileNotFoundError(
            "\n06.21 candidate results could not be recovered.\n"
            f"Expected:\n{RESULTS_PATH}"
        )

    UTILITY_SOURCE_DF = pd.read_csv(
        RESULTS_PATH
    )

    print(
        "Source : persisted 06.21 candidate results"
    )

    print(
        f"Path   : {RESULTS_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 4. FILTER SUCCESSFUL EVALUATIONS
# --------------------------------------------------------------------------------------------------

if "status" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF["status"] = (
        UTILITY_SOURCE_DF["status"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    UTILITY_SOURCE_DF = (
        UTILITY_SOURCE_DF[
            UTILITY_SOURCE_DF["status"] == "SUCCESS"
        ]
        .copy()
        .reset_index(drop=True)
    )


print(
    f"Successful empirical rows : "
    f"{len(UTILITY_SOURCE_DF):,}"
)


if UTILITY_SOURCE_DF.empty:

    raise RuntimeError(
        "No successful empirical evaluations are available "
        "for adaptive utility selection."
    )


# --------------------------------------------------------------------------------------------------
# 5. REQUIRED EXPERIMENTAL KEYS
# --------------------------------------------------------------------------------------------------

SELECTION_KEYS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"
]


REQUIRED_COLUMNS = (
    SELECTION_KEYS
    +
    [
        "strategy_id"
    ]
)


missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in UTILITY_SOURCE_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "06.21 results are missing required selection columns:\n"
        f"{missing_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 6. NORMALIZE IDENTIFIERS
# --------------------------------------------------------------------------------------------------

for column in [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "strategy_id"
]:

    UTILITY_SOURCE_DF[column] = (
        UTILITY_SOURCE_DF[column]
        .astype(str)
        .str.strip()
    )


UTILITY_SOURCE_DF[
    "missingness_rate"
] = pd.to_numeric(
    UTILITY_SOURCE_DF[
        "missingness_rate"
    ],
    errors="coerce"
)


UTILITY_SOURCE_DF[
    "seed"
] = pd.to_numeric(
    UTILITY_SOURCE_DF[
        "seed"
    ],
    errors="coerce"
)


# --------------------------------------------------------------------------------------------------
# 7. SAFE NORMALIZATION FUNCTION
# --------------------------------------------------------------------------------------------------

def normalize_within_group(
    dataframe,
    value_column,
    higher_is_better
):
    """
    Min-max normalize a metric within each experimental context.

    For each context:

        x_norm = (x - min(x)) / (max(x) - min(x))

    For lower-is-better metrics, the direction is reversed.

    If all strategies have the same value within a context,
    all strategies receive utility = 1.0 because they are tied
    on that dimension.
    """

    result = pd.Series(
        np.nan,
        index=dataframe.index,
        dtype=float
    )


    grouped = dataframe.groupby(
        SELECTION_KEYS,
        dropna=False
    )


    for _, group in grouped:

        values = pd.to_numeric(
            group[value_column],
            errors="coerce"
        )


        valid = values.notna()


        if not valid.any():

            continue


        minimum = values[valid].min()
        maximum = values[valid].max()


        if not np.isfinite(
            minimum
        ) or not np.isfinite(
            maximum
        ):

            continue


        if np.isclose(
            maximum,
            minimum
        ):

            normalized = pd.Series(
                1.0,
                index=group.index
            )

        else:

            if higher_is_better:

                normalized = (
                    values - minimum
                ) / (
                    maximum - minimum
                )

            else:

                normalized = (
                    maximum - values
                ) / (
                    maximum - minimum
                )


        result.loc[
            group.index
        ] = normalized


    return result


# --------------------------------------------------------------------------------------------------
# 8. RECONSTRUCTION UTILITY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CALCULATING RECONSTRUCTION UTILITY")
print("-" * 100)


reconstruction_components = []


if "mae" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "_norm_mae"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "mae",
        higher_is_better=False
    )

    reconstruction_components.append(
        UTILITY_SOURCE_DF[
            "_norm_mae"
        ]
    )


if "rmse" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "_norm_rmse"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "rmse",
        higher_is_better=False
    )

    reconstruction_components.append(
        UTILITY_SOURCE_DF[
            "_norm_rmse"
        ]
    )


if reconstruction_components:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = pd.concat(
        reconstruction_components,
        axis=1
    ).mean(
        axis=1,
        skipna=True
    )

else:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = np.nan


# --------------------------------------------------------------------------------------------------
# 9. DISTRIBUTIONAL FIDELITY UTILITY
# --------------------------------------------------------------------------------------------------

print(
    "Calculating distributional fidelity utility..."
)


distribution_components = []


if "wasserstein" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "_norm_wasserstein"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "wasserstein",
        higher_is_better=False
    )

    distribution_components.append(
        UTILITY_SOURCE_DF[
            "_norm_wasserstein"
        ]
    )


if "js_divergence" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "_norm_js"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "js_divergence",
        higher_is_better=False
    )

    distribution_components.append(
        UTILITY_SOURCE_DF[
            "_norm_js"
        ]
    )


if distribution_components:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = pd.concat(
        distribution_components,
        axis=1
    ).mean(
        axis=1,
        skipna=True
    )

else:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = np.nan


# --------------------------------------------------------------------------------------------------
# 10. DEPENDENCY PRESERVATION UTILITY
# --------------------------------------------------------------------------------------------------
#
# The 06.21 engine may expose dependency metrics directly or expose a
# dependency_error column.
#
# We therefore support:
#
#   dependency_error     -> lower is better
#   pearson_error        -> lower is better
#   spearman_error       -> lower is better
#   mutual_info_error    -> lower is better
#
# If a dependency-preservation score already exists, higher is better.
#
# --------------------------------------------------------------------------------------------------

print(
    "Calculating dependency preservation utility..."
)


dependency_components = []


if "dependency_error" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "_norm_dependency_error"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "dependency_error",
        higher_is_better=False
    )

    dependency_components.append(
        UTILITY_SOURCE_DF[
            "_norm_dependency_error"
        ]
    )


for metric in [
    "pearson_error",
    "spearman_error",
    "mutual_info_error"
]:

    if metric in UTILITY_SOURCE_DF.columns:

        normalized_column = (
            f"_norm_{metric}"
        )

        UTILITY_SOURCE_DF[
            normalized_column
        ] = normalize_within_group(
            UTILITY_SOURCE_DF,
            metric,
            higher_is_better=False
        )

        dependency_components.append(
            UTILITY_SOURCE_DF[
                normalized_column
            ]
        )


for metric in [
    "pearson_preservation",
    "spearman_preservation",
    "mutual_info_preservation",
    "dependency_preservation"
]:

    if metric in UTILITY_SOURCE_DF.columns:

        normalized_column = (
            f"_norm_{metric}"
        )

        UTILITY_SOURCE_DF[
            normalized_column
        ] = normalize_within_group(
            UTILITY_SOURCE_DF,
            metric,
            higher_is_better=True
        )

        dependency_components.append(
            UTILITY_SOURCE_DF[
                normalized_column
            ]
        )


if dependency_components:

    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = pd.concat(
        dependency_components,
        axis=1
    ).mean(
        axis=1,
        skipna=True
    )

else:

    # Dependency is unavailable in the current 06.21
    # empirical output. Do not invent a score.
    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = np.nan


# --------------------------------------------------------------------------------------------------
# 11. DOWNSTREAM PREDICTIVE UTILITY
# --------------------------------------------------------------------------------------------------

print(
    "Calculating downstream predictive utility..."
)


downstream_components = []


for metric in [
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1"
]:

    if metric not in UTILITY_SOURCE_DF.columns:

        continue


    UTILITY_SOURCE_DF[
        f"_norm_{metric}"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        metric,
        higher_is_better=True
    )


    downstream_components.append(
        UTILITY_SOURCE_DF[
            f"_norm_{metric}"
        ]
    )


if downstream_components:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = pd.concat(
        downstream_components,
        axis=1
    ).mean(
        axis=1,
        skipna=True
    )

else:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = np.nan


# --------------------------------------------------------------------------------------------------
# 12. COMPUTATIONAL EFFICIENCY
# --------------------------------------------------------------------------------------------------

print(
    "Calculating computational efficiency utility..."
)


if "runtime_seconds" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "efficiency_utility"
    ] = normalize_within_group(
        UTILITY_SOURCE_DF,
        "runtime_seconds",
        higher_is_better=False
    )

else:

    UTILITY_SOURCE_DF[
        "efficiency_utility"
    ] = np.nan


# --------------------------------------------------------------------------------------------------
# 13. VALIDITY MASK
# --------------------------------------------------------------------------------------------------

UTILITY_COMPONENT_COLUMNS = [
    "reconstruction_utility",
    "distribution_utility",
    "dependency_utility",
    "downstream_utility",
    "efficiency_utility"
]


for column in UTILITY_COMPONENT_COLUMNS:

    UTILITY_SOURCE_DF[
        column
    ] = pd.to_numeric(
        UTILITY_SOURCE_DF[
            column
        ],
        errors="coerce"
    )


# --------------------------------------------------------------------------------------------------
# 14. RENORMALIZE AVAILABLE WEIGHTS PER ROW
# --------------------------------------------------------------------------------------------------
#
# This is important.
#
# Some strategies in 06.21 are numeric and others categorical.
# Therefore some metrics may legitimately be unavailable.
#
# We do NOT assign an artificial zero utility to unavailable dimensions.
#
# Instead, the available dimensions retain their relative weights and
# are renormalized so that:
#
#       sum(effective weights) = 1
#
# for each candidate row.
#
# --------------------------------------------------------------------------------------------------

for component, weight in UTILITY_WEIGHTS.items():

    utility_column = (
        f"{component}_utility"
    )

    effective_weight_column = (
        f"_effective_weight_{component}"
    )

    UTILITY_SOURCE_DF[
        effective_weight_column
    ] = np.where(
        UTILITY_SOURCE_DF[
            utility_column
        ].notna(),
        weight,
        0.0
    )


effective_weight_columns = [
    f"_effective_weight_{component}"
    for component in UTILITY_WEIGHTS
]


UTILITY_SOURCE_DF[
    "_available_weight_sum"
] = (
    UTILITY_SOURCE_DF[
        effective_weight_columns
    ]
    .sum(
        axis=1
    )
)


# --------------------------------------------------------------------------------------------------
# 15. ADAPTIVE COMPOSITE UTILITY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CALCULATING ADAPTIVE COMPOSITE UTILITY")
print("-" * 100)


UTILITY_SOURCE_DF[
    "adaptive_utility"
] = 0.0


for component, weight in UTILITY_WEIGHTS.items():

    utility_column = (
        f"{component}_utility"
    )

    UTILITY_SOURCE_DF[
        "adaptive_utility"
    ] += (
        weight
        *
        UTILITY_SOURCE_DF[
            utility_column
        ].fillna(0.0)
    )


# Renormalize when dimensions are unavailable.
UTILITY_SOURCE_DF[
    "adaptive_utility"
] = np.where(

    UTILITY_SOURCE_DF[
        "_available_weight_sum"
    ] > 0,

    UTILITY_SOURCE_DF[
        "adaptive_utility"
    ]
    /
    UTILITY_SOURCE_DF[
        "_available_weight_sum"
    ],

    np.nan

)


# --------------------------------------------------------------------------------------------------
# 16. UTILITY COVERAGE
# --------------------------------------------------------------------------------------------------

UTILITY_SOURCE_DF[
    "utility_dimensions_available"
] = (
    UTILITY_SOURCE_DF[
        UTILITY_COMPONENT_COLUMNS
    ]
    .notna()
    .sum(
        axis=1
    )
)


UTILITY_SOURCE_DF[
    "utility_coverage"
] = (
    UTILITY_SOURCE_DF[
        "utility_dimensions_available"
    ]
    /
    len(
        UTILITY_COMPONENT_COLUMNS
    )
)


# --------------------------------------------------------------------------------------------------
# 17. REMOVE INVALID CANDIDATES
# --------------------------------------------------------------------------------------------------

VALID_UTILITY_DF = (
    UTILITY_SOURCE_DF[
        UTILITY_SOURCE_DF[
            "adaptive_utility"
        ].notna()
    ]
    .copy()
)


print(
    f"Candidates with valid utility : "
    f"{len(VALID_UTILITY_DF):,}"
)


if VALID_UTILITY_DF.empty:

    raise RuntimeError(
        "No candidates have a valid adaptive utility score."
    )


# --------------------------------------------------------------------------------------------------
# 18. RANK CANDIDATES WITHIN EACH EXPERIMENTAL CONTEXT
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("RANKING CANDIDATE STRATEGIES WITHIN EACH CONTEXT")
print("-" * 100)


VALID_UTILITY_DF[
    "adaptive_rank"
] = (
    VALID_UTILITY_DF
    .groupby(
        SELECTION_KEYS,
        dropna=False
    )[
        "adaptive_utility"
    ]
    .rank(
        method="min",
        ascending=False
    )
)


# Deterministic tie-break:
#
#   1. higher adaptive utility
#   2. lower runtime
#   3. alphabetical strategy ID
#
VALID_UTILITY_DF = (
    VALID_UTILITY_DF
    .sort_values(
        SELECTION_KEYS
        +
        [
            "adaptive_utility",
            "runtime_seconds",
            "strategy_id"
        ],
        ascending=[
            True,
            True,
            True,
            True,
            True,
            False,
            True,
            True
        ],
        na_position="last"
    )
)


# --------------------------------------------------------------------------------------------------
# 19. TOP-1 ADAPTIVE SELECTION
# --------------------------------------------------------------------------------------------------

ADAPTIVE_SELECTION_DF = (
    VALID_UTILITY_DF[
        VALID_UTILITY_DF[
            "adaptive_rank"
        ] == 1
    ]
    .copy()
)


# Safety check.
expected_contexts = (
    VALID_UTILITY_DF[
        SELECTION_KEYS
    ]
    .drop_duplicates()
    .shape[0]
)


actual_selections = len(
    ADAPTIVE_SELECTION_DF
)


print(
    f"Unique experimental contexts : "
    f"{expected_contexts:,}"
)

print(
    f"Adaptive selections           : "
    f"{actual_selections:,}"
)


if actual_selections != expected_contexts:

    raise RuntimeError(
        "Adaptive selection did not produce exactly one "
        "Top-1 strategy per experimental context."
    )


# --------------------------------------------------------------------------------------------------
# 20. SELECT OUTPUT COLUMNS
# --------------------------------------------------------------------------------------------------

SELECTION_OUTPUT_COLUMNS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed",

    "strategy_id",

    "adaptive_rank",
    "adaptive_utility",

    "reconstruction_utility",
    "distribution_utility",
    "dependency_utility",
    "downstream_utility",
    "efficiency_utility",

    "utility_dimensions_available",
    "utility_coverage"
]


if "llm_recommended" in VALID_UTILITY_DF.columns:

    SELECTION_OUTPUT_COLUMNS.append(
        "llm_recommended"
    )


ADAPTIVE_SELECTION_DF = (
    ADAPTIVE_SELECTION_DF[
        [
            column
            for column in SELECTION_OUTPUT_COLUMNS
            if column in ADAPTIVE_SELECTION_DF.columns
        ]
    ]
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 21. ADAPTIVE SELECTION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("ADAPTIVE SELECTION SUMMARY")
print("-" * 100)


STRATEGY_SELECTION_SUMMARY_DF = (
    ADAPTIVE_SELECTION_DF[
        "strategy_id"
    ]
    .value_counts()
    .rename_axis(
        "strategy_id"
    )
    .reset_index(
        name="selected_contexts"
    )
)


STRATEGY_SELECTION_SUMMARY_DF[
    "selection_rate"
] = (
    STRATEGY_SELECTION_SUMMARY_DF[
        "selected_contexts"
    ]
    /
    len(
        ADAPTIVE_SELECTION_DF
    )
)


display(
    STRATEGY_SELECTION_SUMMARY_DF
)


# --------------------------------------------------------------------------------------------------
# 22. LLM AGREEMENT, IF AVAILABLE
# --------------------------------------------------------------------------------------------------

if "llm_recommended" in ADAPTIVE_SELECTION_DF.columns:

    ADAPTIVE_SELECTION_DF[
        "llm_match"
    ] = (
        ADAPTIVE_SELECTION_DF[
            "strategy_id"
        ]
        .astype(str)
        .str.strip()
        ==
        ADAPTIVE_SELECTION_DF[
            "llm_recommended"
        ]
        .astype(str)
        .str.strip()
    )


    adaptive_llm_agreement = (
        ADAPTIVE_SELECTION_DF[
            "llm_match"
        ]
        .mean()
    )


    print(
        f"\nAdaptive selection / LLM agreement : "
        f"{adaptive_llm_agreement:.4f}"
    )

    print(
        f"Adaptive selection / LLM agreement : "
        f"{adaptive_llm_agreement * 100:.2f}%"
    )


# --------------------------------------------------------------------------------------------------
# 23. DISPLAY SAMPLE
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SAMPLE ADAPTIVE SELECTIONS")
print("-" * 100)


display(
    ADAPTIVE_SELECTION_DF.head(
        25
    )
)


# --------------------------------------------------------------------------------------------------
# 24. SAVE PRIMARY RESULTS
# --------------------------------------------------------------------------------------------------

ADAPTIVE_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_26_adaptive_utility_selection.csv"
)


SELECTION_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_26_strategy_selection_summary.csv"
)


UTILITY_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "air_llm_06_26_candidate_utility_scores.csv"
)


ADAPTIVE_SELECTION_DF.to_csv(
    ADAPTIVE_RESULTS_PATH,
    index=False
)


STRATEGY_SELECTION_SUMMARY_DF.to_csv(
    SELECTION_SUMMARY_PATH,
    index=False
)


VALID_UTILITY_DF.to_csv(
    UTILITY_RESULTS_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 25. DOWNSTREAM VARIABLES
# --------------------------------------------------------------------------------------------------

AIR_LLM_0626_ADAPTIVE_SELECTION_DF = (
    ADAPTIVE_SELECTION_DF.copy()
)


AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF = (
    STRATEGY_SELECTION_SUMMARY_DF.copy()
)


AIR_LLM_0626_CANDIDATE_UTILITY_DF = (
    VALID_UTILITY_DF.copy()
)


# --------------------------------------------------------------------------------------------------
# 26. FINAL DIAGNOSTICS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("06.26 ADAPTIVE UTILITY-BASED SELECTION : COMPLETE")
print("=" * 100)

print(
    f"Successful empirical evaluations : "
    f"{len(UTILITY_SOURCE_DF):,}"
)

print(
    f"Valid utility candidates         : "
    f"{len(VALID_UTILITY_DF):,}"
)

print(
    f"Experimental contexts            : "
    f"{expected_contexts:,}"
)

print(
    f"Adaptive Top-1 selections        : "
    f"{actual_selections:,}"
)

print(
    "\nPrimary results:"
)

print(
    ADAPTIVE_RESULTS_PATH
)

print(
    "\nCandidate utility results:"
)

print(
    UTILITY_RESULTS_PATH
)

print(
    "\nStrategy selection summary:"
)

print(
    SELECTION_SUMMARY_PATH
)

print(
    "\nDownstream dataframes:"
)

print(
    "AIR_LLM_0626_ADAPTIVE_SELECTION_DF"
)

print(
    "AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF"
)

print(
    "AIR_LLM_0626_CANDIDATE_UTILITY_DF"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.26
ADAPTIVE UTILITY-BASED SELECTION

----------------------------------------------------------------------------------------------------
RECOVERING NOTEBOOK 06.21 EMPIRICAL RESULTS
----------------------------------------------------------------------------------------------------
Source : AIR_LLM_0621_RESULTS_DF
Successful empirical rows : 5,775

----------------------------------------------------------------------------------------------------
CALCULATING RECONSTRUCTION UTILITY
----------------------------------------------------------------------------------------------------
Calculating distributional fidelity utility...
Calculating dependency preservation utility...
Calculating downstream predictive utility...
Calculating computational efficiency utility...

----------------------------------------------------------------------------------------------------
CALCULATING ADAPTIVE COMPOSITE UTILITY
--------------------------------------------------------------

,strategy_id,selected_contexts,selection_rate
0,mode,3975,0.688312
1,mean,1800,0.311688



Adaptive selection / LLM agreement : 0.0000
Adaptive selection / LLM agreement : 0.00%

----------------------------------------------------------------------------------------------------
SAMPLE ADAPTIVE SELECTIONS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,missingness_mechanism,missingness_rate,seed,strategy_id,adaptive_rank,adaptive_utility,reconstruction_utility,distribution_utility,dependency_utility,downstream_utility,efficiency_utility,utility_dimensions_available,utility_coverage,llm_recommended,llm_match
0,adult_income,age,MAR,0.1,345086045,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
1,adult_income,age,MAR,0.1,1182932783,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
2,adult_income,age,MAR,0.1,2074715863,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
3,adult_income,age,MAR,0.1,3313111679,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
4,adult_income,age,MAR,0.1,3393134027,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
5,adult_income,age,MAR,0.2,433268084,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
6,adult_income,age,MAR,0.2,510431391,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
7,adult_income,age,MAR,0.2,1963370003,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
8,adult_income,age,MAR,0.2,2220770356,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False
9,adult_income,age,MAR,0.2,2657548161,mean,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3,0.6,False,False



06.26 ADAPTIVE UTILITY-BASED SELECTION : COMPLETE
Successful empirical evaluations : 5,775
Valid utility candidates         : 5,775
Experimental contexts            : 5,775
Adaptive Top-1 selections        : 5,775

Primary results:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_adaptive_utility_selection.csv

Candidate utility results:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_candidate_utility_scores.csv

Strategy selection summary:
/content/drive/MyDrive/AIR_LLM_Research/results/air_llm_06_26_strategy_selection_summary.csv

Downstream dataframes:
AIR_LLM_0626_ADAPTIVE_SELECTION_DF
AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF
AIR_LLM_0626_CANDIDATE_UTILITY_DF


In [36]:
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06.27
# FINAL PERSISTENCE / RESEARCH ARTIFACT CHECKPOINT
# ==================================================================================================
#
# PURPOSE
# -------
# Persist all important outputs produced by Notebook 06 so that subsequent notebooks
# can recover the AIR-LLM candidate-evaluation, utility-selection, and LLM-comparison
# artifacts without rerunning the expensive evaluation process.
#
# This cell performs NO new experiments.
#
# It only:
#   1. Validates the completed Notebook 06 state
#   2. Saves important DataFrames as CSV
#   3. Saves reusable DataFrames as Pickle
#   4. Saves configuration / metadata as JSON
#   5. Creates one consolidated Notebook-06 artifact bundle
#   6. Prints an auditable persistence summary
#
# ==================================================================================================

import os
import json
import pickle
import shutil
from datetime import datetime

import numpy as np
import pandas as pd


print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.27")
print("FINAL PERSISTENCE / RESEARCH ARTIFACT CHECKPOINT")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL PROJECT PATH
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

NOTEBOOK_06_ARCHIVE_DIR = os.path.join(
    RESULTS_DIR,
    "notebook_06_final"
)

os.makedirs(
    NOTEBOOK_06_ARCHIVE_DIR,
    exist_ok=True
)


print("\n" + "-" * 100)
print("PERSISTENCE DIRECTORY")
print("-" * 100)

print(
    NOTEBOOK_06_ARCHIVE_DIR
)


# --------------------------------------------------------------------------------------------------
# 2. CURRENT TIMESTAMP
# --------------------------------------------------------------------------------------------------

SAVE_TIMESTAMP = (
    datetime.now()
    .astimezone()
    .isoformat()
)


# --------------------------------------------------------------------------------------------------
# 3. REQUIRED DATAFRAME VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATING NOTEBOOK 06 ARTIFACTS")
print("-" * 100)


REQUIRED_ARTIFACTS = {
    "AIR_LLM_0621_RESULTS_DF":
        "06.21 empirical candidate results",

    "AIR_LLM_0626_ADAPTIVE_SELECTION_DF":
        "06.26 adaptive Top-1 selections",

    "AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF":
        "06.26 strategy selection summary",

    "AIR_LLM_0626_CANDIDATE_UTILITY_DF":
        "06.26 candidate utility scores"
}


OPTIONAL_ARTIFACTS = {
    "STATISTICAL_SIGNIFICANCE_DF":
        "06.25 statistical significance",

    "LLM_EVALUATION_DF":
        "06.24 LLM recommendation evaluation",

    "LLM_PERFORMANCE_DF":
        "06.24 LLM performance comparison",

    "AIR_LLM_0621_BOUNDED_PLAN_DF":
        "06.21 bounded evaluation plan",

    "EVALUATION_PLAN_DF":
        "evaluation plan"
}


missing_required = []


for variable_name, description in REQUIRED_ARTIFACTS.items():

    if (
        variable_name not in globals()
        or not isinstance(
            globals()[variable_name],
            pd.DataFrame
        )
    ):

        missing_required.append(
            variable_name
        )

    else:

        print(
            f"PASS  {variable_name:<45} "
            f"rows={len(globals()[variable_name]):,}"
        )


if missing_required:

    raise RuntimeError(
        "Required Notebook 06 artifacts are missing:\n"
        + "\n".join(
            f" - {x}"
            for x in missing_required
        )
    )


for variable_name, description in OPTIONAL_ARTIFACTS.items():

    if (
        variable_name in globals()
        and isinstance(
            globals()[variable_name],
            pd.DataFrame
        )
    ):

        print(
            f"FOUND {variable_name:<45} "
            f"rows={len(globals()[variable_name]):,}"
        )

    else:

        print(
            f"SKIP  {variable_name:<45} "
            f"not currently available"
        )


# --------------------------------------------------------------------------------------------------
# 4. ARTIFACT REGISTRY
# --------------------------------------------------------------------------------------------------

ARTIFACT_REGISTRY = {}


def save_dataframe(
    dataframe,
    filename,
    description
):

    csv_path = os.path.join(
        NOTEBOOK_06_ARCHIVE_DIR,
        filename
    )

    dataframe.to_csv(
        csv_path,
        index=False
    )

    ARTIFACT_REGISTRY[
        filename
    ] = {
        "type": "csv",
        "description": description,
        "rows": int(len(dataframe)),
        "columns": int(len(dataframe.columns)),
        "path": csv_path
    }

    return csv_path


# --------------------------------------------------------------------------------------------------
# 5. SAVE 06.21 EMPIRICAL RESULTS
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SAVING 06.21 EMPIRICAL RESULTS")
print("-" * 100)


RESULTS_0621_PATH = save_dataframe(
    AIR_LLM_0621_RESULTS_DF,
    "air_llm_06_21_candidate_results.csv",
    "Complete empirical candidate evaluation results from Notebook 06.21"
)


print(
    RESULTS_0621_PATH
)


# --------------------------------------------------------------------------------------------------
# 6. SAVE 06.21 BOUNDED PLAN IF AVAILABLE
# --------------------------------------------------------------------------------------------------

if (
    "AIR_LLM_0621_BOUNDED_PLAN_DF" in globals()
    and isinstance(
        AIR_LLM_0621_BOUNDED_PLAN_DF,
        pd.DataFrame
    )
):

    BOUNDED_PLAN_PATH = save_dataframe(
        AIR_LLM_0621_BOUNDED_PLAN_DF,
        "air_llm_06_21_bounded_evaluation_plan.csv",
        "Bounded candidate evaluation plan used by Notebook 06.21"
    )

    print(
        BOUNDED_PLAN_PATH
    )


# --------------------------------------------------------------------------------------------------
# 7. SAVE FULL EVALUATION PLAN IF AVAILABLE
# --------------------------------------------------------------------------------------------------

if (
    "EVALUATION_PLAN_DF" in globals()
    and isinstance(
        EVALUATION_PLAN_DF,
        pd.DataFrame
    )
):

    FULL_PLAN_PATH = save_dataframe(
        EVALUATION_PLAN_DF,
        "air_llm_06_full_evaluation_plan.csv",
        "Full AIR-LLM evaluation plan"
    )

    print(
        FULL_PLAN_PATH
    )


# --------------------------------------------------------------------------------------------------
# 8. SAVE 06.24 LLM EVALUATION IF AVAILABLE
# --------------------------------------------------------------------------------------------------

if (
    "LLM_EVALUATION_DF" in globals()
    and isinstance(
        LLM_EVALUATION_DF,
        pd.DataFrame
    )
):

    LLM_EVAL_PATH = save_dataframe(
        LLM_EVALUATION_DF,
        "air_llm_06_24_llm_recommendation_evaluation.csv",
        "LLM recommendation versus empirical evaluation"
    )

    print(
        LLM_EVAL_PATH
    )


if (
    "LLM_PERFORMANCE_DF" in globals()
    and isinstance(
        LLM_PERFORMANCE_DF,
        pd.DataFrame
    )
):

    LLM_PERFORMANCE_PATH = save_dataframe(
        LLM_PERFORMANCE_DF,
        "air_llm_06_24_llm_performance.csv",
        "LLM recommendation performance comparison"
    )

    print(
        LLM_PERFORMANCE_PATH
    )


# --------------------------------------------------------------------------------------------------
# 9. SAVE 06.25 STATISTICAL SIGNIFICANCE
# --------------------------------------------------------------------------------------------------

if (
    "STATISTICAL_SIGNIFICANCE_DF" in globals()
    and isinstance(
        STATISTICAL_SIGNIFICANCE_DF,
        pd.DataFrame
    )
):

    SIGNIFICANCE_PATH = save_dataframe(
        STATISTICAL_SIGNIFICANCE_DF,
        "air_llm_06_25_statistical_significance.csv",
        "Friedman statistical significance analysis"
    )

    print(
        SIGNIFICANCE_PATH
    )


# --------------------------------------------------------------------------------------------------
# 10. SAVE 06.26 CANDIDATE UTILITY SCORES
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SAVING 06.26 UTILITY RESULTS")
print("-" * 100)


UTILITY_PATH = save_dataframe(
    AIR_LLM_0626_CANDIDATE_UTILITY_DF,
    "air_llm_06_26_candidate_utility_scores.csv",
    "Candidate-level normalized utility scores and composite adaptive utility"
)


print(
    UTILITY_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. SAVE 06.26 ADAPTIVE TOP-1 SELECTIONS
# --------------------------------------------------------------------------------------------------

ADAPTIVE_SELECTION_PATH = save_dataframe(
    AIR_LLM_0626_ADAPTIVE_SELECTION_DF,
    "air_llm_06_26_adaptive_utility_selection.csv",
    "Empirically selected Top-1 strategy for each experimental context"
)


print(
    ADAPTIVE_SELECTION_PATH
)


# --------------------------------------------------------------------------------------------------
# 12. SAVE 06.26 STRATEGY SUMMARY
# --------------------------------------------------------------------------------------------------

STRATEGY_SUMMARY_PATH = save_dataframe(
    AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF,
    "air_llm_06_26_strategy_selection_summary.csv",
    "Frequency and selection rate of empirically selected strategies"
)


print(
    STRATEGY_SUMMARY_PATH
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE CONSOLIDATED PICKLE BUNDLE
# --------------------------------------------------------------------------------------------------
#
# Pickle allows future notebooks to restore the DataFrames with their original
# pandas structure without reconstructing them from CSV files.
#
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CREATING REUSABLE NOTEBOOK 06 DATAFRAME BUNDLE")
print("-" * 100)


DATAFRAME_BUNDLE = {}


for variable_name in REQUIRED_ARTIFACTS:

    if (
        variable_name in globals()
        and isinstance(
            globals()[variable_name],
            pd.DataFrame
        )
    ):

        DATAFRAME_BUNDLE[
            variable_name
        ] = globals()[
            variable_name
        ].copy()


for variable_name in OPTIONAL_ARTIFACTS:

    if (
        variable_name in globals()
        and isinstance(
            globals()[variable_name],
            pd.DataFrame
        )
    ):

        DATAFRAME_BUNDLE[
            variable_name
        ] = globals()[
            variable_name
        ].copy()


BUNDLE_PATH = os.path.join(
    NOTEBOOK_06_ARCHIVE_DIR,
    "air_llm_notebook_06_dataframe_bundle.pkl"
)


with open(
    BUNDLE_PATH,
    "wb"
) as file:

    pickle.dump(
        DATAFRAME_BUNDLE,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )


print(
    f"Bundle saved : {BUNDLE_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE UTILITY CONFIGURATION
# --------------------------------------------------------------------------------------------------

UTILITY_CONFIGURATION = {
    "project": "AIR-LLM",
    "research_code": "AIR-LLM",
    "notebook": "06",
    "final_cell": "06.27",
    "artifact_version": "06.27-final",
    "saved_at": SAVE_TIMESTAMP,

    "utility_weights": {
        key: float(value)
        for key, value in UTILITY_WEIGHTS.items()
    },

    "normalization": {
        "method": "within-experimental-context min-max",
        "lower_is_better_metrics": [
            "mae",
            "rmse",
            "wasserstein",
            "js_divergence",
            "dependency_error",
            "pearson_error",
            "spearman_error",
            "mutual_info_error",
            "runtime_seconds"
        ],
        "higher_is_better_metrics": [
            "r2",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "pearson_preservation",
            "spearman_preservation",
            "mutual_info_preservation",
            "dependency_preservation"
        ]
    },

    "selection_keys": [
        "dataset_id",
        "feature",
        "missingness_mechanism",
        "missingness_rate",
        "seed"
    ],

    "selection_rule": (
        "Select the strategy with maximum adaptive composite utility "
        "within each experimental context."
    ),

    "missing_dimension_policy": (
        "Unavailable utility dimensions are excluded and the remaining "
        "utility weights are renormalized."
    )
}


CONFIG_PATH = os.path.join(
    NOTEBOOK_06_ARCHIVE_DIR,
    "air_llm_06_utility_configuration.json"
)


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        UTILITY_CONFIGURATION,
        file,
        indent=2
    )


print(
    f"Configuration saved : {CONFIG_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 15. CREATE HIGH-LEVEL NOTEBOOK 06 METADATA
# --------------------------------------------------------------------------------------------------

metadata = {
    "project": "AIR-LLM",
    "notebook": "06",
    "final_checkpoint": "06.27",
    "saved_at": SAVE_TIMESTAMP,

    "project_root": PROJECT_ROOT,
    "results_directory": RESULTS_DIR,
    "archive_directory": NOTEBOOK_06_ARCHIVE_DIR,

    "successful_empirical_evaluations": int(
        len(
            AIR_LLM_0621_RESULTS_DF
        )
    ),

    "adaptive_selection_rows": int(
        len(
            AIR_LLM_0626_ADAPTIVE_SELECTION_DF
        )
    ),

    "candidate_utility_rows": int(
        len(
            AIR_LLM_0626_CANDIDATE_UTILITY_DF
        )
    ),

    "strategy_summary_rows": int(
        len(
            AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF
        )
    ),

    "utility_weights": {
        key: float(value)
        for key, value in UTILITY_WEIGHTS.items()
    },

    "artifact_files": list(
        ARTIFACT_REGISTRY.keys()
    )
}


METADATA_PATH = os.path.join(
    NOTEBOOK_06_ARCHIVE_DIR,
    "air_llm_notebook_06_metadata.json"
)


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# 16. CREATE MASTER MANIFEST
# --------------------------------------------------------------------------------------------------

MANIFEST = {
    "air_llm_notebook": "06",
    "checkpoint": "06.27",
    "saved_at": SAVE_TIMESTAMP,

    "description": (
        "Final persistent research artifacts from AIR-LLM Notebook 06. "
        "Contains empirical candidate evaluation, LLM recommendation "
        "analysis when available, statistical significance analysis when "
        "available, adaptive utility scores, and adaptive Top-1 selections."
    ),

    "files": ARTIFACT_REGISTRY,

    "dataframe_bundle": {
        "path": BUNDLE_PATH,
        "objects": list(
            DATAFRAME_BUNDLE.keys()
        )
    },

    "configuration": CONFIG_PATH,
    "metadata": METADATA_PATH
}


MANIFEST_PATH = os.path.join(
    NOTEBOOK_06_ARCHIVE_DIR,
    "AIR_LLM_NOTEBOOK_06_MANIFEST.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        MANIFEST,
        file,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# 17. CREATE A SIMPLE RECOVERY SCRIPT
# --------------------------------------------------------------------------------------------------
#
# This creates a small Python file that can be used in Notebook 07 or later
# to restore the persisted DataFrames.
#
# --------------------------------------------------------------------------------------------------

RECOVERY_SCRIPT = r'''
# ==================================================================================================
# AIR-LLM — NOTEBOOK 06 ARTIFACT RECOVERY
# ==================================================================================================

import os
import pickle

PROJECT_ROOT = "/content/drive/MyDrive/AIR_LLM_Research"

BUNDLE_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "notebook_06_final",
    "air_llm_notebook_06_dataframe_bundle.pkl"
)

with open(
    BUNDLE_PATH,
    "rb"
) as file:

    AIR_LLM_06_BUNDLE = pickle.load(file)


globals().update(
    AIR_LLM_06_BUNDLE
)

print("=" * 100)
print("AIR-LLM NOTEBOOK 06 ARTIFACTS RECOVERED")
print("=" * 100)

for name, dataframe in AIR_LLM_06_BUNDLE.items():

    print(
        f"{name:<50} "
        f"rows={len(dataframe):,} "
        f"columns={len(dataframe.columns):,}"
    )

print("=" * 100)
'''


RECOVERY_SCRIPT_PATH = os.path.join(
    NOTEBOOK_06_ARCHIVE_DIR,
    "recover_notebook_06.py"
)


with open(
    RECOVERY_SCRIPT_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        RECOVERY_SCRIPT
    )


# --------------------------------------------------------------------------------------------------
# 18. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FINAL PERSISTENCE VALIDATION")
print("-" * 100)


files_to_validate = [
    BUNDLE_PATH,
    CONFIG_PATH,
    METADATA_PATH,
    MANIFEST_PATH,
    RECOVERY_SCRIPT_PATH
]


for path in files_to_validate:

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            /
            (1024 ** 2)
        )

        print(
            f"PASS  {os.path.basename(path):<55} "
            f"{size_mb:.3f} MB"
        )

    else:

        raise RuntimeError(
            f"Persistence validation failed: {path}"
        )


# --------------------------------------------------------------------------------------------------
# 19. FINAL RESEARCH CHECKPOINT SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("AIR-LLM — NOTEBOOK 06 FINAL CHECKPOINT : COMPLETE")
print("=" * 100)

print(
    f"Saved at                         : {SAVE_TIMESTAMP}"
)

print(
    f"Successful empirical evaluations : "
    f"{len(AIR_LLM_0621_RESULTS_DF):,}"
)

print(
    f"Candidate utility rows           : "
    f"{len(AIR_LLM_0626_CANDIDATE_UTILITY_DF):,}"
)

print(
    f"Adaptive selections              : "
    f"{len(AIR_LLM_0626_ADAPTIVE_SELECTION_DF):,}"
)

print(
    f"Strategy summary rows            : "
    f"{len(AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF):,}"
)

print(
    "\nFinal archive:"
)

print(
    NOTEBOOK_06_ARCHIVE_DIR
)

print(
    "\nMaster manifest:"
)

print(
    MANIFEST_PATH
)

print(
    "\nReusable DataFrame bundle:"
)

print(
    BUNDLE_PATH
)

print(
    "\nRecovery script:"
)

print(
    RECOVERY_SCRIPT_PATH
)

print(
    "\nNotebook 07 can recover the persisted Notebook 06 state "
    "without rerunning the expensive candidate evaluation."
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.27
FINAL PERSISTENCE / RESEARCH ARTIFACT CHECKPOINT

----------------------------------------------------------------------------------------------------
PERSISTENCE DIRECTORY
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/AIR_LLM_Research/results/notebook_06_final

----------------------------------------------------------------------------------------------------
VALIDATING NOTEBOOK 06 ARTIFACTS
----------------------------------------------------------------------------------------------------
PASS  AIR_LLM_0621_RESULTS_DF                       rows=5,775
PASS  AIR_LLM_0626_ADAPTIVE_SELECTION_DF            rows=5,775
PASS  AIR_LLM_0626_STRATEGY_SELECTION_SUMMARY_DF    rows=2
PASS  AIR_LLM_0626_CANDIDATE_UTILITY_DF             rows=5,775
FOUND STATISTICAL_SIGNIFICANCE_DF                   rows=0
FOUND LLM_EVALUATION_DF                             rows=5,775
FOUND LLM_PERFORMANCE_DF     